# VietTSC — Master Notebook (baseline end-to-end) · J02 directional ROI · **Full mode (J04)**
> **Notebook Full-mode (J04):** bản sao của `VietTSC_master_J02_ROI.ipynb` (engine detect/ROI/calibration/flow/directional KHÔNG đổi), chỉ đổi cell cấu hình + cách chạy benchmark: mặc định theo source (light/medium/heavy × fixedtime/maxpressure/mplight, MPLight train từng mode → tự chọn checkpoint theo split **val** → eval trên split **test**), `EVAL_EPISODES` episode/seed, stress `test-heavy` opt-in, cài `libsumo` nếu thiếu. Xem cell "Hướng dẫn chọn mode chạy" ngay bên dưới.
**J02 (`DEMAND_MODE`, cell 00a):** `"j02_mixed"` (mặc định) = Cam1 full-frame 50/50 + Cam2/Cam3/Cam4 ROI FORWARD/REVERSE (`flow_estimation.directional` + `build_demand.py --settings --directions --tag j02`), mapping camera→edge GỐC (`camera_segments.json`), SCALE áp đúng 1 lần; `"baseline_5050"` = hành vi gốc. Cam4→Thành Thái là proxy mô hình; Cam1 50/50 là giả định kế thừa.

**Source-first, notebook-as-orchestrator.** Notebook chỉ nối các bước bằng CLI/hàm có sẵn của REPO: `flow_estimation.{detect,calibrate,infer,window}` → `build_demand.py` (routeSampler) → `run_tsc.py eval` / `run_tsc.make_env` + `run_tsc.run_episode`. Không sửa source, không viết lại thuật toán.

**Cách nối dữ liệu (không cần sửa source):**
- `REPO_DIR` (source gốc) chỉ **đọc**. Mọi lệnh chạy trong **staging copy** `STAGE` (code giữ nguyên byte, được kiểm bằng hash).
- `build_demand.py` đọc `STAGE/results/flow/demand_settings.json` (đường dẫn cố định `build_demand.py:35`) → notebook đặt đúng file vừa sinh ở đó, rồi gọi script gốc; `baseline_5050` ghi `STAGE/sumo/scenarios/`, `j02_mixed` (`--tag j02`) ghi `STAGE/sumo/scenarios_j02/`. Thư mục được chọn (`SCEN_ROOT`) được truyền **tường minh** cho mọi controller (`run_tsc.py --scenario-root`, cả eval/train/custom); controller kiểm cfg + SHA net/routes thực sự load với `scenario_expected_binding.json` (`--expect-binding`) **trước** episode. Thiếu scenario => lỗi, không fallback.
- Detection cache / calibration đang dùng được chốt một lần trong `ACTIVE`; mọi bước sau đọc từ `ACTIVE`, thiếu file thì dừng (không fallback ngầm).

**Chế độ chạy** (`RUN_MODE`, cell 00a): `baseline_cache` = dùng cache YOLO11x + calibration có sẵn trong repo, chạy tới hết SUMO + benchmark. `full_recompute` = chạy lại detect + calibrate (cần ảnh gốc, weights, Linux). Training là opt-in (`RUN_TRAIN`). `calibration_only` = chỉ chạy YOLO11x detect → calibrate → bảng/log ghi vào `OUT_DIR/calibration_runs/<RUN_ID>/`; mọi stage sau calibration (flow inference, routeSampler, SUMO, controllers, benchmark) bị **skip có chủ đích**.
**J02E — nguồn ảnh city & chế độ review (cell 00a):** `CITY_SOURCE="repo_cache"` (mặc định, hành vi cũ: `data/detections/city_cam_frames.jsonl` + cache repo) | `"images"` (`CITY_DATASET` = thư mục hoặc `.zip` ảnh snapshot; `flow_estimation.city_index` dựng index + kiểm tra; cần `SCENARIO_WINDOWS` tường minh; detection chạy trên ảnh, hoặc tái dùng cache có sẵn qua `VIETTSC_CITY_DETS_CACHE`). `RUN_MODE="image_review"` (MỚI) = chỉ detect city + đếm xe theo ROI từng frame + overlay (`flow_estimation.roi_frames`; **số đếm tức thời trong 1 ảnh, KHÔNG phải flow/veh-h**), không cần calibration; calibration/flow/SUMO/benchmark **bị bỏ qua có chủ đích** (kết thúc `IMAGE_REVIEW COMPLETE`). `baseline_cache` dùng calibration repo (yolo11x) hoặc `VIETTSC_CALIBRATION_ARTIFACT` (kiểm bằng `flow_estimation.calibration_artifact`; hybrid không bao giờ dùng β của yolo11x). Calibration set tuỳ chọn: `VIETTSC_CALIB_SET`; calibration trên ground truth tổng hợp = **CODE TEST, không có giá trị khoa học**.
Placeholder: 05 (M3 / J03) — chưa implement.

## Hướng dẫn chọn mode chạy
Mọi thứ chỉnh ở **cell 00a** (cell code đầu tiên).

**Mặc định hiện tại = Hybrid** (người dùng chọn 2026-11-22): `DETECTOR_BACKEND="hybrid_cleaned"` (YOLO11x car/truck/bus + YOLO12-X BMD45 xe máy + cleanup), `CITY_SOURCE="images"` (dataset `5cam-14p`), 1 cửa sổ `("medium", "5cam_14p", "13:06", "13:21")`, 3 controller, MPLight train 100 round, 1 episode eval. Trên Kaggle, cell 00a tự điền đường dẫn **weights + ảnh** (`VIETTSC_YOLO11X`, `VIETTSC_BMD45`, `VIETTSC_CITY_DATASET`) nếu chưa đặt và file tồn tại.

⚠ **Calibration KHÔNG tự điền** — phải đặt `os.environ["VIETTSC_CALIBRATION_ARTIFACT"]` TRƯỚC cell 00a (thiếu ⇒ cell 00a báo lỗi rõ ràng):
- calibration Hybrid **thật**: thư mục `calibration_runs/<RUN_ID>` của một lần chạy `RUN_MODE="calibration_only"` trước đó (hiện BLOCKED: β bus NaN trên dữ liệu J02E);
- fixture **giả lập** `synth_integration_fixture` (`SYNTHETIC_INTEGRATION_TEST`, chỉ để kiểm pipeline, KHÔNG phải benchmark thật): upload thư mục vào `/kaggle/working/synth_integration_fixture` rồi đặt `os.environ["VIETTSC_CALIBRATION_ARTIFACT"] = "/kaggle/working/synth_integration_fixture"`. Benchmark đủ 3 mode light/medium/heavy với calibration thật của repo: **công thức 9** (YOLO11x).

### Bảng tham số
| Tham số | Mặc định (source) | Ý nghĩa | Nguồn trong source |
|---|---|---|---|
| `RUN_MODE` | `baseline_cache` | `baseline_cache` dùng cache+calibration có sẵn của repo · `full_recompute` detect+calibrate lại (ảnh gốc, weights, Linux/GPU) · `calibration_only` chỉ calibration (bỏ SUMO) · `image_review` chỉ detect city + đếm ROI (bỏ calibration/SUMO) | README, `flow_estimation/*` |
| `DETECTOR_BACKEND` | `hybrid_cleaned` (source: `yolo11x`) | `hybrid_cleaned` (J01; cần 2 checkpoint + calibration hybrid) hoặc `yolo11x` (source) | `flow_estimation/hybrid_detector.py`, `flow_estimation/detect.py` |
| `CITY_SOURCE` | `images` (source: `repo_cache`) | `images` = ảnh snapshot (`VIETTSC_CITY_DATASET`) · `repo_cache` = index+cache detection YOLO11x của repo | J02E |
| `SCENARIO_WINDOWS` | `[("medium", "5cam_14p", "13:06", "13:21")]` (source: `None` = 3 cửa sổ repo) | Danh sách `(nhãn, session, "HH:MM", "HH:MM")` → quyết định `EVAL_MODES`. `images` bắt buộc khai báo; `None` chỉ dùng với `repo_cache` | `flow_estimation/window.py` |
| `SPLIT` / `SEED` | `test` / `0` | split eval (seed demand train 1001 / val 2002 / test 3003) / seed SUMO | `run_tsc.py`, `build_demand.py:48` (`SPLIT_SEEDS`) |
| `RUN_COUNTS` | `3600` | giây mô phỏng / episode | `tsc/config.py` ENV_CONF |
| `SCALE` | `0.25` | hệ số demand của source (KHÔNG nâng lên 1.0) | `build_demand.py:50` (`DEFAULT_SCALE`) |
| `DEMAND_MODE` | `j02_mixed` | `j02_mixed` (ROI theo hướng) hoặc `baseline_5050` (chia 50/50 gốc) | J02 |
| `CONTROLLERS` | `["fixedtime","maxpressure","mplight"]` | controller có sẵn trong source | `run_tsc.py:98` |
| `EVAL_EPISODES` | `1` | số episode eval mỗi (mode, controller); seed SUMO = `SEED`, `SEED+1`, … | `run_tsc.py eval --episodes` |
| `RUN_TRAIN` | `True` | train MPLight trước khi eval | `run_tsc.py train` |
| `TRAIN_ROUNDS` | `100` | số round train | `tsc/config.py` EXP_CONF `NUM_ROUNDS` |
| `TRAIN_GENERATORS` | `1` | số episode thu thập / round | EXP_CONF `NUM_GENERATORS` |
| `TRAIN_TEST_EVERY` | `1` | greedy test trên split **val** mỗi N round (phải ≥ 1: dùng để chọn checkpoint) | EXP_CONF `TEST_EVERY` |
| `MPLIGHT_TRAIN_MODES` | `None` (= mọi `EVAL_MODES`) | tập con `EVAL_MODES` cần train | — |
| `MPLIGHT_CKPT` | `None` | `None` = dùng checkpoint tự chọn · `"x.pt"` (mọi mode) · `{"light": "x.pt", ...}` (ghi đè thủ công theo mode) | `run_tsc.py eval --model` |
| `INCLUDE_STRESS` | `False` | thêm mode stress `test-heavy` (demand random-density của `build_trips.py`, KHÔNG phải demand camera) | `run_tsc.py:15-16` |
| `STRESS_MPLIGHT_FROM` | `"heavy"` | MPLight trên stress dùng checkpoint của mode này (test-heavy giống hệt ở mọi split nên không tách được train/val) | — |
| `INSTALL_LIBSUMO` | `True` | thử `pip install libsumo` nếu thiếu (lỗi => dùng `traci`, chậm hơn ~10x) | `tsc/env.py:75-84`, `requirements.txt` |
| `BENCH_TRIPINFO` | `True` | J05: eval chạy với `--tripinfo-dir` ⇒ thêm cột delay / waiting / trip_duration / số xe unfinished-undeparted (từ tripinfo SUMO). `False` ⇒ bảng như J04b | `run_tsc.py eval --tripinfo-dir`, `tsc/tripinfo.py` |
| `MPLIGHT_TRAIN_SEEDS` | `None` (= `[SEED]`, 1 trial) | J05: danh sách seed (số nguyên ≥ 0, không trùng); mỗi seed = 1 trial train độc lập (`run_tsc.py train --seed s`) và 1 lượt eval MPLight. Không dùng cùng `MPLIGHT_CKPT` thủ công khi > 1 seed | `run_tsc.py train --seed` |
| `REPORT_BASELINE` | `"fixedtime"` | J05: controller làm mốc cho `bench/improvement.csv` (% cải thiện; dương = tốt hơn mốc). Không có trong `CONTROLLERS` ⇒ bỏ qua bước này (có in chú thích) | `tsc/benchmark.py::improvement_table` |
| `LEARNING_CURVES` | `True` | J05: vẽ `train/<mode>/learning_curve.png` từ `progress.csv` (cần matplotlib; thiếu ⇒ chỉ in cảnh báo, run vẫn PASS) | `tsc/benchmark.py::plot_learning_curves` |
| `ADAPTIVE_BETA` | `"auto"` | J07: `"auto"` = bật nếu calibration đang dùng có khối `adaptive_beta`, không thì tắt (có in chú thích, β cố định) · `True` = bắt buộc (thiếu khối ⇒ lỗi) · `False` = tắt (β cố định). Bật ⇒ cell flow infer + directional infer thêm `--adaptive-beta` | `flow_estimation/adaptive_beta.py`, `infer.py`, `directional.py` |

Checkpoint MPLight tự chọn: hàng `progress.csv` có `test_travel_time` nhỏ nhất (greedy trên **val**, không bao giờ dùng test); hoà → ít `test_teleports` hơn → round nhỏ hơn. Mỗi (mode, train seed) có thư mục riêng `train/<mode>/seed<s>/` (run_tsc tạo `mplight_<mode>_<stamp>/` bên trong) và `train/<mode>/seed<s>/mplight_selection.json`; learning curve: `train/<mode>/learning_curve.png`. Output `bench/`: `results_episodes.csv` (mỗi episode 1 dòng, có `train_seed`), `results.csv` (trung bình/độ lệch chuẩn mẫu theo (mode, controller), GỘP mọi trial: `n = n_trials × EVAL_EPISODES`, cột `n_trials`, `train_seeds`; `n=1` => std = NaN), `results_trials.csv` (theo (mode, controller, train_seed); chỉ khi có > 1 trial), `improvement.csv/json` (% so `REPORT_BASELINE`). Định nghĩa metric + khác biệt RESCO/LibSignal: mục **Benchmark protocol (J05)** (sau cell 09).

### J07 — Power-law Adaptive Beta
β theo từng frame thay vì cố định: `β = β*[gap][c] · clip(D/D̄[gap][c], 0.25, 3)^γ[c]` (5 gap 5/10/15/20/30 s; `c` = motorcycle/car/truck/bus).
- **D** = số detection **không tĩnh** của class `c` trong frame (toàn frame, không ROI; cùng ngưỡng conf với calibration). **D̄** = D trung bình lúc calibration.
- `calibrate.py` tự fit: γ theo class bằng CV theo camera, chỉ nhận khi WAPE CV giảm trung bình ≥ 0.5 điểm (không thì γ = 0 = β cố định). Kết quả YOLO11x: γ motorcycle 0.2, các class khác 0. Khối `adaptive_beta` nằm trong calibration JSON (`form`, `gamma`, `dbar`, `beta_star`, `cv`, `density_stats`, `gamma_source`).
- Knob `ADAPTIVE_BETA` (cell 00a): `"auto"` (mặc định) / `True` / `False` — xem bảng tham số. Bật ⇒ `<FLOW_DIR>/adaptive_beta_used.json` + `perframe.json` có thêm `density`, `beta_factor`; cell calibration report in bảng `dbar` / `β*` / CV.
- **Công thức J07-a — YOLO11x + adaptive:**
```python
os.environ["VIETTSC_CALIBRATION_ARTIFACT"] = "<REPO>/flow_estimation/calibration/yolo11x_1280_adaptive.json"   # sinh bằng calibrate.py (có sau bước Kaggle); đặt TRƯỚC cell 00a
DETECTOR_BACKEND = "yolo11x"
CITY_SOURCE = "repo_cache"
SCENARIO_WINDOWS = None
```
- **Công thức J07-b — Hybrid:** chạy `RUN_MODE = "calibration_only"` trước (calibrate.py tự ghi khối `adaptive_beta`), rồi đặt `os.environ["VIETTSC_CALIBRATION_ARTIFACT"]` = thư mục calibration run đó và chạy lại ở mode downstream. Calibration không có khối này + `ADAPTIVE_BETA="auto"` ⇒ chạy β cố định như cũ.

### Công thức (dán đè các dòng tương ứng ở cell 00a)
Công thức 2, 6, 7 dùng cửa sổ của repo (light/heavy) ⇒ phải kèm **công thức 9** (YOLO11x + `repo_cache`). Với mặc định Hybrid chỉ có mode `medium`, nên `INCLUDE_STRESS=True` cần `STRESS_MPLIGHT_FROM = "medium"`.
**1. Mặc định** (không đổi gì = Hybrid, 1 mode medium; đo Kaggle: round train ~70–150 s ⇒ 100 round ≈ 2–3,5 h — smoke: `RUN_COUNTS = 300`, `TRAIN_ROUNDS = 3`, Kaggle `j05_hybrid_synth_small` E2E PASS). Chi phí công thức 9 (YOLO11x đủ 3 mode): training chiếm gần hết thời gian (3 mode × 100 round × (1 train + 1 val episode) mô phỏng 3600 s). Đo trên Kaggle: `traci` ≈ 40 s/episode medium (chưa có số đo libsumo; libsumo nhanh hơn, xem README). Ước lượng thô ~ vài giờ tới ~7 giờ — đặt `TRAIN_ROUNDS` nhỏ nếu cần.

**2. Smoke nhanh**
```python
RUN_COUNTS = 300
TRAIN_ROUNDS = 2
MPLIGHT_TRAIN_MODES = ["light"]
EVAL_EPISODES = 1
SCENARIO_WINDOWS = [("light", "2026-11-12_0530-0700", "05:30", "06:30")]   # chỉ 1 cửa sổ repo (None = cả 3)
```
**3. Chỉ baseline, không train**
```python
RUN_TRAIN = False
CONTROLLERS = ["fixedtime", "maxpressure"]      # hoặc giữ "mplight" nhưng không có MPLIGHT_CKPT => mplight bị bỏ qua (ghi rõ)
```
**4. Eval MPLight từ checkpoint có sẵn** (không train)
```python
RUN_TRAIN = False
MPLIGHT_CKPT = {"light": "/kaggle/input/ckpt/light_round_57.pt", "medium": "/kaggle/input/ckpt/medium_round_80.pt", "heavy": "/kaggle/input/ckpt/heavy_round_91.pt"}   # hoặc 1 đường dẫn cho mọi mode
```
**5. Nhiều seed**
```python
EVAL_EPISODES = 5       # seed SUMO = SEED..SEED+4
```
Seed chỉ đổi tính ngẫu nhiên của SUMO; demand/routes cố định bởi `SPLIT_SEEDS` của `build_demand.py` (cùng scenario cho mọi episode), nên std phản ánh dao động mô phỏng, không phải dao động demand.

**6. Stress test-heavy**
```python
INCLUDE_STRESS = True
STRESS_MPLIGHT_FROM = "heavy"     # phải thuộc mode được train (hoặc có trong MPLIGHT_CKPT)
```
Demand của test-heavy là random-density (`build_trips.py`, ×12), KHÔNG phải demand ước lượng từ camera; MPLight chạy với checkpoint của `STRESS_MPLIGHT_FROM`. Ước lượng ~4–5 phút/episode/controller.

**7. Chỉ 1 mode (ví dụ heavy)**
```python
SCENARIO_WINDOWS = [("heavy", "2026-10-29_0850-1250", "10:50", "11:50")]
```
**8. Demand baseline 50/50**
```python
DEMAND_MODE = "baseline_5050"
```
**9. YOLO11x, đủ 3 mode light/medium/heavy (default của source; cache + calibration thật của repo)**
```python
DETECTOR_BACKEND = "yolo11x"
CITY_SOURCE = "repo_cache"
SCENARIO_WINDOWS = None      # 3 cửa sổ gốc của repo
```
(Không cần weights/dataset/calibration ngoài repo. Đây là cấu hình đã nghiệm thu J04/J05, Kaggle `j05_kaggle_smoke_final` E2E PASS.)

**9b. Hybrid: đặt calibration (BẮT BUỘC) / đổi dataset** (weights + ảnh: Kaggle tự điền, đặt tay để ghi đè; TRƯỚC khi chạy cell 00a)
```python
os.environ["VIETTSC_CITY_DATASET"] = "<thư mục hoặc .zip ảnh>"                    # đổi ảnh thì đổi cả SCENARIO_WINDOWS cho khớp
os.environ["VIETTSC_CALIBRATION_ARTIFACT"] = "<thư mục/ file calibration hybrid>"  # hoặc VIETTSC_HYBRID_CACHE_DIR
os.environ["VIETTSC_CITY_DETS_CACHE"] = "<cache detection city, tuỳ chọn: bỏ qua detect>"
os.environ["VIETTSC_YOLO11X"], os.environ["VIETTSC_BMD45"] = "<yolo11x.pt>", "<yolov12x_bmd45.pt>"
```
(Calibration hybrid thật: `RUN_MODE="calibration_only"` + `VIETTSC_CALIB_SET` — xem `contexte/J02E_RUNBOOK.md` mục C. `RUN_MODE="image_review"` không cần calibration.)

**10. Báo cáo (5 trial MPLight)**
```python
MPLIGHT_TRAIN_SEEDS = [0, 1, 2, 3, 4]    # 5 trial train độc lập mỗi mode (MPLIGHT_CKPT phải = None)
EVAL_EPISODES = 5                          # seed SUMO eval = SEED..SEED+4
```
Chi phí: thời gian training = (công thức ở mục 1) × số trial (5 trial ⇒ gấp 5; mặc định 100 round × 3 mode ≈ vài giờ/trial), eval MPLight cũng chạy 1 lượt/trial. `results.csv` gộp 5 trial × 5 episode = n 25 cho MPLight (fixedtime/maxpressure: n 5); xem từng trial ở `results_trials.csv`. Smoke: `MPLIGHT_TRAIN_SEEDS = [0, 1]` + `TRAIN_ROUNDS = 2` + `RUN_COUNTS = 300`.


## J01 — Running original YOLO11x
`DETECTOR_BACKEND = "yolo11x"` (mặc định, cell 00a). Đường chạy gốc không đổi: `flow_estimation.detect` (COCO, chỉ YOLO11x) → `calibrate` → `infer`/`window` → `build_demand` → SUMO. Không cần BMD45, không cần cleanup config.

| RUN_MODE | Input bắt buộc | Artifact (OUT_DIR = `.../out/<RUN_ID>`) |
|---|---|---|
| `baseline_cache` | cache + calibration có sẵn của repo (`data/detections/*yolo11x*`, `flow_estimation/calibration/yolo11x_1280.json`) | không detect/calibrate lại; flow/demand/SUMO/benchmark |
| `full_recompute` | ảnh calibration + ảnh city + weights YOLO11x **thật** (`VIETTSC_YOLO11X`; `models/yolo11x.pt` của repo chỉ là LFS pointer) + GPU + `fork` | `detections/{calib,city}_yolo11x.jsonl.gz`, `calibration/yolo11x_recomputed.json`, … |
| `calibration_only` | ảnh calibration + weights thật + GPU | `calibration_runs/<RUN_ID>/{detections.jsonl.gz, calibration.json, …}` |


## J01 — Running Hybrid Cleaned
`DETECTOR_BACKEND = "hybrid_cleaned"`. Nguồn từng lớp: **car/truck/bus = YOLO11x** (COCO 2/5/7, NMS gốc), **motorcycle = YOLO12-X BMD45 lớp `Two-wheeler`** (id lấy từ `model.names` lúc chạy), mọi lớp khác bị loại. Sau đó **cleanup S0-G** (chỉ box motorcycle của BMD45: rule R1 `ENCLOSES_CLUSTER`; size-cap chỉ gắn cờ, KHÔNG xóa) rồi late fusion → cache VietTSC bốn lớp, cùng schema `detect.py` (JSONL.GZ, 1920×1080, conf 3 chữ số). Downstream (calibrate/infer/window/build_demand/SUMO) là code gốc, không đổi.

- **Hai checkpoint thật**: `VIETTSC_YOLO11X` (yolo11x.pt, SHA `7bc158aa…`) và `VIETTSC_BMD45` hoặc `REPO/models/yolov12x_bmd45.pt` (SHA `4c3bc036…`, HF `iisc-aim/BMD-45` rev `ec06f3df…`). Notebook copy vào `STAGE/models/`, từ chối LFS pointer / sai size / sai SHA, **không tải tự động**. Kaggle: `/kaggle/input` chỉ đọc; `/kaggle/working` không bền → đặt 2 file vào một Dataset.
- **Cách ly artifact**: `calib_hybrid_cleaned.jsonl.gz`, `city_hybrid_cleaned.jsonl.gz`, `hybrid_cleaned_calibration.json` (+ `*.provenance.json`), RUN_ID mặc định `run_hybrid_cleaned_*`. β/ngưỡng của YOLO11x **không** được dùng lại.
- **Nhãn cũ**: `calibrate.py` ghi cứng `"yolo11x (COCO, Ultralytics)"` vào calibration JSON. Với hybrid, nhãn đó **không** là provenance; dùng `<calibration>.provenance.json` và `experiment_metadata.json`.
- **Diagnostics** (tuỳ chọn): `EXPORT_HYBRID_DIAGNOSTICS = True` (cần ảnh; tối đa `DIAGNOSTIC_MAX_IMAGES`) → `OUT_DIR/diagnostics/<RUN_ID>/hybrid_cleaned/` (FINAL theo ngưỡng, before/after cleanup, box bị xóa/cờ). Không phải ground truth.
- **Giới hạn**: box BMD45 bị chặn kích thước (≈67.7×75.3 px ở ảnh 512×288, chưa rõ nguyên nhân); confidence YOLO11x và BMD45 không tương đương; cleanup chỉ được duyệt bằng 12 ảnh; chưa có ground truth ⇒ chưa thể nói hybrid chính xác hơn. Bicycle không được phát hiện (downstream không cần). `J00a` F1/F2/F3 đã merge vào notebook này (qua J01); notebook original chưa freeze.

| RUN_MODE | Input bắt buộc (hybrid_cleaned) | Artifact |
|---|---|---|
| `baseline_cache` | `VIETTSC_HYBRID_CACHE_DIR` chứa `calib_hybrid_cleaned.jsonl.gz`, `city_hybrid_cleaned.jsonl.gz`, `hybrid_cleaned_calibration.json` + 3 `*.provenance.json` hợp lệ (SHA khớp); thiếu/lệch ⇒ **dừng**, không dùng cache YOLO11x | không detect/calibrate lại; flow/demand/SUMO/benchmark |
| `full_recompute` | ảnh calibration + ảnh city + 2 checkpoint thật + GPU + `fork` | `detections/{calib,city}_hybrid_cleaned.jsonl.gz(+provenance)`, `calibration/hybrid_cleaned_recomputed.json(+provenance)`, audit CSV trong `logs/` |
| `calibration_only` | ảnh calibration + 2 checkpoint thật + GPU | `calibration_runs/<RUN_ID>/{detections.jsonl.gz(+.provenance.json), calibration.json(+.provenance.json), …}` |

**Chạy nhỏ không calibration**: dùng CLI trực tiếp trong STAGE — `python -m flow_estimation.hybrid_detector --list <jsonl> --root <ảnh> --out <cache.jsonl.gz> --weights-yolo11x … --weights-bmd45 … --audit-dir <dir>` rồi `python -m flow_estimation.hybrid_diagnostics --cache … --audit-dir … --root … --out … --max-images 12`.


In [ ]:
import os
import sys
import torch

print("Python:", sys.executable)
print("Kaggle:", os.path.exists("/kaggle/input"))
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 00 — Setup & Preflight

In [ ]:
# 00a. CẤU HÌNH CHUNG (sửa ở đây). Mọi thứ khác suy ra từ cell này.
import os, sys, json, shutil, subprocess, platform, hashlib, importlib, importlib.util, argparse, time, csv, re
import importlib.metadata as im
import datetime as dt
from datetime import datetime as _datetime, timezone as _tz   # dùng cho metadata: biến `dt` có thể bị cell khác ghi đè
import xml.etree.ElementTree as ET
from pathlib import Path
try: sys.stdout.reconfigure(encoding="utf-8")
except Exception: pass

# ---- chế độ chạy ----
RUN_MODE = "baseline_cache"        # "baseline_cache" | "full_recompute" | "calibration_only" | "image_review"   # J02E: image_review = detect city + ROI từng frame, không calibration/flow/SUMO
# ---- benchmark (giữ cố định giữa các controller) ----
SPLIT = "test"                     # train/val/test = seed demand 1001/2002/3003 (build_demand.py:39)
# J02E: EVAL_MODES suy ra từ SCENARIO_WINDOWS (khối "J02E" bên dưới), theo đúng thứ tự SCENARIO_WINDOWS
SEED = 0                           # SUMO seed (run_tsc --seed)
RUN_COUNTS = 3600                  # giây mô phỏng / episode (run_tsc --run-counts)
SCALE = 0.25                       # baseline của source (build_demand.py:41). KHÔNG nâng lên 1.0
# ---- controllers ----
CONTROLLERS = ["fixedtime", "maxpressure", "mplight"]   # J04: mặc định source (cả 3). Chỉ baseline => bỏ "mplight" VÀ đặt RUN_TRAIN = False
# ---- J04: MPLight = train từng mode (split train) -> tự chọn checkpoint theo split val -> eval trên split test ----
MPLIGHT_CKPT = None                # J04: None => dùng checkpoint tự chọn sau training | "path.pt" (dùng cho MỌI mode) | {"light": "a.pt", ...} (ghi đè thủ công theo mode, ưu tiên hơn ckpt tự chọn)
RUN_TRAIN = True                   # J04: mặc định source: train MPLight rồi eval. False => không train (MPLight chỉ chạy nếu MPLIGHT_CKPT có)
TRAIN_ROUNDS = 100                 # J04: tsc/config.py EXP_CONF["NUM_ROUNDS"]
TRAIN_GENERATORS = 1               # J04: EXP_CONF["NUM_GENERATORS"] (run_tsc.py train --generators)
TRAIN_TEST_EVERY = 1               # J04: EXP_CONF["TEST_EVERY"]; greedy test trên VAL mỗi N round. PHẢI >= 1 vì checkpoint được chọn theo kết quả val
MPLIGHT_TRAIN_MODES = None         # J04: None => train mọi mode trong EVAL_MODES | ["light", ...] (tập con của EVAL_MODES). Train = split train; val = split val CÙNG mode. KHÔNG train trên test
TRAIN_MODE = None                  # (tương thích cũ, KHÔNG cấu hình ở đây) suy ra = MPLIGHT_TRAIN_MODES[0]
# ---- J04: số episode / stress / libsumo ----
EVAL_EPISODES = 1                  # J04: run_tsc.py eval --episodes (source mặc định 1). Seed SUMO = SEED, SEED+1, ...; demand/routes KHÔNG đổi giữa các episode
INCLUDE_STRESS = False             # J04: True => thêm mode stress "test-heavy" vào benchmark (demand random-density của build_trips.py, KHÔNG phải demand camera)
STRESS_MODE = "test-heavy"         # J04: run_tsc.py chỉ có mode stress này
STRESS_MPLIGHT_FROM = "heavy"      # J04: MPLight trên stress dùng checkpoint của mode này (test-heavy giống hệt ở mọi split nên không tách được train/val)
BENCH_TRIPINFO = True              # J05: True => run_tsc.py eval --tripinfo-dir (thêm cột delay/waiting/trip_duration/n_trips_unfinished/undeparted vào bảng). False => như J04b
MPLIGHT_TRAIN_SEEDS = None         # J05: None => [SEED] (1 trial) | [0, 1, 2, 3, 4] => mỗi seed 1 trial train riêng (run_tsc.py train --seed s) + 1 lượt eval; chi phí training × số trial
REPORT_BASELINE = "fixedtime"      # J05: controller làm mốc cho improvement.csv (% cải thiện); không có trong CONTROLLERS => bỏ qua có chú thích
LEARNING_CURVES = True             # J05: vẽ train/<mode>/learning_curve.png (cần matplotlib; thiếu => chỉ cảnh báo)
ADAPTIVE_BETA = "auto"   # J07: "auto" = on if the active calibration JSON has an "adaptive_beta" block, else off (note printed) | True = required (error if missing) | False = off (fixed beta)
INSTALL_LIBSUMO = True             # J04: thử pip install libsumo nếu thiếu (lỗi => run_tsc dùng traci, chậm hơn ~10x)
# Controller của bạn: module .py có factory(junctions) -> agent với act(obs, explore=False) -> {junction_id: phase_index}
USER_AGENT_MODULE = None           # vd "my_tsc"  (None = không dùng)
USER_AGENT_FACTORY = "build_agent" # tên hàm/class trong module: factory(junctions) -> agent
USER_AGENT_NAME = "user"
USER_AGENT_PATHS = []              # thư mục thêm vào sys.path để import module trên
# ---- dữ liệu nằm NGOÀI repo (chỉ cần cho full_recompute) ----
# Kaggle: dataset ở /kaggle/input/<tên>/...; đặt os.environ["VIETTSC_CALIB_IMAGES"] = "<đường dẫn>" (và các biến VIETTSC_* khác) TRƯỚC khi chạy cell này
def _p(env): v = os.environ.get(env); return Path(v) if v else None
# MẶC ĐỊNH HYBRID (2026-11-22): trên Kaggle tự điền đường dẫn WEIGHTS + ẢNH cho biến VIETTSC_* CHƯA đặt (chỉ khi file/thư mục tồn tại; biến đã đặt thì giữ nguyên)
if Path("/kaggle").exists():
    _KAGGLE_DEFAULTS = {
        "VIETTSC_YOLO11X": ["/kaggle/input/datasets/rykanny/yolo11-x/yolo11x.pt", "/kaggle/input/yolo11-x/yolo11x.pt"],
        "VIETTSC_BMD45": ["/kaggle/input/datasets/rykanny/weight-yolo12xbmd45/yolov12x_bmd45.pt", "/kaggle/input/weight-yolo12xbmd45/yolov12x_bmd45.pt"],
        "VIETTSC_CITY_DATASET": ["/kaggle/input/datasets/rykanny/5cam-14p", "/kaggle/input/5cam-14p"],
        # KHÔNG tự điền VIETTSC_CALIBRATION_ARTIFACT: calibration phải do người chạy chọn rõ (thật hoặc giả lập) — xem cell hướng dẫn
    }
    for _k, _cands in _KAGGLE_DEFAULTS.items():
        if not os.environ.get(_k):
            _hit = next((c for c in _cands if Path(c).exists()), None)
            if _hit: os.environ[_k] = _hit; print(f"Kaggle mặc định: {_k} = {_hit}")
            else: print(f"Kaggle mặc định: {_k} KHÔNG tìm thấy (đã thử {_cands}) — đặt os.environ['{_k}'] trước cell này")
CITY_IMAGES  = _p("VIETTSC_CITY_IMAGES")      # <root>/<session>/Cam_*/frame_*.jpg
CALIB_IMAGES = _p("VIETTSC_CALIB_IMAGES")     # <root>/<video>/<frame>.jpg
YOLO_WEIGHTS = _p("VIETTSC_YOLO11X")          # None => weights repo nếu là file thật, ngược lại "yolo11x.pt" (Ultralytics tự tải, cần Internet)
REQUIRE_GPU = True                 # calibration_only: detection trên ~35k ảnh cần GPU (Kaggle: bật GPU)
ALLOW_WEIGHTS_DOWNLOAD = True      # không có file weights thật -> cho Ultralytics tải 'yolo11x.pt' (cần Internet)
# ============================================================
# J01 — DETECTOR BACKEND SELECTION
# ============================================================
# J01: "yolo11x"        = detector gốc của VietTSC (flow_estimation.detect, COCO, chỉ YOLO11x). Default của source.
# J01: "hybrid_cleaned" = YOLO11x (car/truck/bus) + YOLO12-X BMD45 (Two-wheeler -> motorcycle) + cleanup S0-G (flow_estimation.hybrid_detector).
DETECTOR_BACKEND = "hybrid_cleaned"   # MẶC ĐỊNH (người dùng chọn 2026-11-22). YOLO11x đủ 3 mode: xem công thức 9 ở cell hướng dẫn
# DETECTOR_BACKEND = "yolo11x"        # default của source
# J01: hybrid_cleaned cần HAI checkpoint THẬT (không phải Git LFS pointer; SHA-256 được ghim trong flow_estimation.hybrid_detector):
#      VIETTSC_YOLO11X (yolo11x.pt thật, biến đã có ở trên) và VIETTSC_BMD45 (yolov12x_bmd45.pt thật; mặc định REPO/models/yolov12x_bmd45.pt).
#      Kaggle: /kaggle/input chỉ đọc, /kaggle/working KHÔNG giữ qua phiên -> đưa 2 file vào một Dataset và đặt hai biến môi trường. Không tải tự động.
BMD45_WEIGHTS = _p("VIETTSC_BMD45")
# J01: baseline_cache + hybrid_cleaned dùng thư mục artifact hybrid ĐÃ XÁC MINH (calib_hybrid_cleaned.jsonl.gz, city_hybrid_cleaned.jsonl.gz,
#      hybrid_cleaned_calibration.json + các *.provenance.json); không bao giờ rơi về cache/β của YOLO11x.
HYBRID_CACHE_DIR = _p("VIETTSC_HYBRID_CACHE_DIR")
# J01: ảnh diagnostic (opt-in, có giới hạn; chỉ hybrid_cleaned; vẽ từ cache của chính detector đang chọn). KHÔNG phải ground truth.
EXPORT_HYBRID_DIAGNOSTICS = False
DIAGNOSTIC_MAX_IMAGES = 12
DIAGNOSTIC_CONF_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
# ---- dựng lại calibration set bằng build_calibration_set.py (opt-in; INPUT KHÔNG có trong repo) ----
TRACKS_DIR    = _p("VIETTSC_TRACKS")        # --tracks: 1 JSON/video có "objects" (tracker KHÔNG nằm trong repo)
MAPKF_DIR     = _p("VIETTSC_MAPKF")         # --mapkf : 1 CSV/video (frame_id, timestamp)
KEYFRAMES_DIR = _p("VIETTSC_KEYFRAMES")     # --keyframes: <lot>/<video>/<frame_id>.webp
CAMERA_FOLDS  = _p("VIETTSC_CAMERA_FOLDS")  # calibrate.py:98 cần camera_folds.json; build_calibration_set KHÔNG tạo file này
REBUILD_CALIB_SET = False                   # True chỉ hợp lệ với RUN_MODE="full_recompute"
# ---- pipeline đo flow (giữ như README / window.py) ----
# J02E: nguồn ảnh city + cửa sổ kịch bản (thay SESSIONS / EXCLUDE_CAMS / SETTING_WINDOWS cố định)
CITY_SOURCE = "images"            # MẶC ĐỊNH Hybrid: ảnh snapshot (VIETTSC_CITY_DATASET) ("repo_cache" | "images")
# CITY_SOURCE = "repo_cache"       # default của source (index + cache detection YOLO11x của repo)
CITY_DATASET = _p("VIETTSC_CITY_DATASET")          # J02E: thư mục hoặc .zip (images). Zip được giải nén vào WORK/inputs/<RUN_ID>/
CITY_DETS_CACHE = _p("VIETTSC_CITY_DETS_CACHE")    # J02E: (tuỳ chọn, chỉ images) tái dùng cache detection city có sẵn thay vì detect; khoá = đường dẫn ảnh tương đối trong CITY_INDEX
CITY_EXCLUDE = ["Cam_To_Hien_Thanh_Su_Van_Hanh"]   # J02E: camera loại (ID chuẩn) -> EXCLUDE_CAMS
# J02E: SCENARIO_WINDOWS = [(nhãn light|medium|heavy, session, "HH:MM", "HH:MM"), ...]. repo_cache: None => 3 cửa sổ gốc của repo. images: BẮT BUỘC khai báo (không có mặc định).
SCENARIO_WINDOWS = [("medium", "5cam_14p", "13:06", "13:21")]   # MẶC ĐỊNH Hybrid: cửa sổ của dataset 5cam-14p (nhãn 'medium' do người dùng chọn, D35)
# SCENARIO_WINDOWS = None         # repo_cache: None => 3 cửa sổ gốc của repo (light/medium/heavy)
# J02E: calibration / ground truth từ ngoài repo
CALIBRATION_ARTIFACT = _p("VIETTSC_CALIBRATION_ARTIFACT")   # J02E: thư mục run calibration_only/full_recompute trước đó, hoặc file calibration .json (dùng cho baseline_cache)
CALIB_SET_DIR = _p("VIETTSC_CALIB_SET")      # J02E: videos.json, frames.jsonl, camera_folds.json, images/, (tuỳ chọn) ground_truth_provenance.json. None => data/calibration của repo
REVIEW_CONF = 0.25                 # J02E: ngưỡng xem lại khi KHÔNG có calibration (image_review); nhãn REVIEW_THRESHOLD_NOT_CALIBRATED
ROI_OVERLAYS_PER_CAMERA = 3        # J02E: số overlay ROI / camera (roi_frames)
SHOW_OVERLAYS = 4                  # J02E: số overlay hiện trong notebook
WORKERS = 8
# ============================================================
# J02 — DEMAND MODE (directional ROI). Mapping camera->edge GỐC giữ nguyên (sumo/config/camera_segments.json).
# ============================================================
# "j02_mixed"     = Cam1 full-frame 50/50; Cam2/3/4 ROI FORWARD/REVERSE (EXCLUDE bỏ) -> rate×SCALE, KHÔNG chia 2.
# "baseline_5050" = hành vi gốc của source (mọi camera /2).
DEMAND_MODE = "j02_mixed"
J02_TAG = "j02"                     # build_demand --tag -> STAGE/sumo/scenarios_j02 (không đụng sumo/scenarios)
INSTALL_MISSING = Path("/kaggle").exists()     # chỉ pip install package CÒN THIẾU (không cài lại torch/SUMO nếu đã có)

# ---- suy ra ----
assert RUN_MODE in ("baseline_cache", "full_recompute", "calibration_only", "image_review"), RUN_MODE   # J02E
assert DEMAND_MODE in ("j02_mixed", "baseline_5050"), DEMAND_MODE   # J02
J02 = DEMAND_MODE == "j02_mixed"
# J01: backend (hybrid_cleaned = YOLO11x car/truck/bus + BMD45 motorcycle + cleanup S0-G)
assert DETECTOR_BACKEND in ("yolo11x", "hybrid_cleaned"), DETECTOR_BACKEND
HYBRID = DETECTOR_BACKEND == "hybrid_cleaned"
_DTAG = DETECTOR_BACKEND                     # tên artifact theo backend; "yolo11x" giữ nguyên tên cũ
assert not EXPORT_HYBRID_DIAGNOSTICS or HYBRID, "EXPORT_HYBRID_DIAGNOSTICS chỉ dùng với DETECTOR_BACKEND='hybrid_cleaned'"
assert 1 <= DIAGNOSTIC_MAX_IMAGES <= 100, "DIAGNOSTIC_MAX_IMAGES phải trong 1..100 (không xuất hàng nghìn PNG)"
if HYBRID and RUN_MODE == "baseline_cache" and HYBRID_CACHE_DIR is None and CALIBRATION_ARTIFACT is None:   # J02E: hoặc artifact calibration hybrid
    raise RuntimeError("J01: DETECTOR_BACKEND='hybrid_cleaned' + RUN_MODE='baseline_cache' cần VIETTSC_HYBRID_CACHE_DIR (artifact hybrid đã xác minh) hoặc VIETTSC_CALIBRATION_ARTIFACT (calibration hybrid). KHÔNG dùng cache/β của YOLO11x thay thế.")
RECOMPUTE = RUN_MODE in ("full_recompute", "calibration_only")      # chạy lại detect (+ calibrate)
DOWNSTREAM = RUN_MODE not in ("calibration_only", "image_review")   # J02E: flow inference, routeSampler, SUMO, benchmark (image_review: bỏ qua có chủ đích)
# ---- J02E: nguồn city, cửa sổ kịch bản, cờ chế độ ----
assert CITY_SOURCE in ("repo_cache", "images"), CITY_SOURCE
IMAGES = CITY_SOURCE == "images"
IMAGE_REVIEW = RUN_MODE == "image_review"
assert not IMAGE_REVIEW or IMAGES, "J02E: RUN_MODE='image_review' cần CITY_SOURCE='images' (ảnh + CITY_DATASET)"
assert CITY_DETS_CACHE is None or (IMAGES and RUN_MODE != "calibration_only"), "J02E: VIETTSC_CITY_DETS_CACHE chỉ dùng với CITY_SOURCE='images' (và RUN_MODE != calibration_only)"
if HYBRID and RUN_MODE == "baseline_cache" and not IMAGES and HYBRID_CACHE_DIR is None:
    raise RuntimeError("J02E: hybrid_cleaned + baseline_cache + CITY_SOURCE='repo_cache' cần VIETTSC_HYBRID_CACHE_DIR (cache city hybrid); calibration artifact không thay cache city")
if SCENARIO_WINDOWS is None:
    if IMAGES: raise RuntimeError("J02E: CITY_SOURCE='images' cần SCENARIO_WINDOWS tường minh [(nhãn, session, 'HH:MM', 'HH:MM'), ...] (không có mặc định)")
    SCENARIO_WINDOWS = [("light", "2026-11-12_0530-0700", "05:30", "06:30"), ("medium", "2026-11-07_1306-1706", "13:06", "14:06"), ("heavy", "2026-10-29_0850-1250", "10:50", "11:50")]   # cửa sổ gốc của repo
SCENARIO_WINDOWS = [tuple(_w) for _w in SCENARIO_WINDOWS]
if not SCENARIO_WINDOWS: raise RuntimeError("J02E: SCENARIO_WINDOWS rỗng — cần ít nhất một (nhãn, session, 'HH:MM', 'HH:MM')")
for _w in SCENARIO_WINDOWS:
    assert len(_w) == 4 and all(isinstance(_x, str) for _x in _w), f"J02E: cửa sổ không hợp lệ {_w!r} (cần (nhãn, session, 'HH:MM', 'HH:MM'))"
    assert _w[0] in ("light", "medium", "heavy"), f"J02E: nhãn {_w[0]!r} phải thuộc light/medium/heavy (build_demand --mode)"
    assert all(re.fullmatch(r"\d{1,2}:\d{2}", _x) for _x in _w[2:]), f"J02E: HH:MM không hợp lệ trong {_w!r}"
assert len({_w[0] for _w in SCENARIO_WINDOWS}) == len(SCENARIO_WINDOWS), "J02E: nhãn trong SCENARIO_WINDOWS phải duy nhất"
EVAL_MODES = [_w[0] for _w in SCENARIO_WINDOWS]
SESSIONS = list(dict.fromkeys(_w[1] for _w in SCENARIO_WINDOWS))
SETTING_WINDOWS = SCENARIO_WINDOWS             # alias cho code downstream
EXCLUDE_CAMS = list(CITY_EXCLUDE)
# ---- J04: mode train / eval / stress + kiểm tra cấu hình ----
def _j04_posint(name, v):
    assert isinstance(v, int) and not isinstance(v, bool) and v >= 1, f"J04: {name} phải là số nguyên >= 1, nhận {v!r}"
_j04_posint("EVAL_EPISODES", EVAL_EPISODES); _j04_posint("TRAIN_ROUNDS", TRAIN_ROUNDS); _j04_posint("TRAIN_GENERATORS", TRAIN_GENERATORS)
assert isinstance(TRAIN_TEST_EVERY, int) and not isinstance(TRAIN_TEST_EVERY, bool) and TRAIN_TEST_EVERY >= 1, \
    f"J04: TRAIN_TEST_EVERY phải >= 1 (checkpoint được chọn theo kết quả val; 0 = không có val), nhận {TRAIN_TEST_EVERY!r}"
assert STRESS_MODE == "test-heavy", f"J04: STRESS_MODE chỉ có thể là 'test-heavy' (run_tsc.py --mode), nhận {STRESS_MODE!r}"
assert STRESS_MPLIGHT_FROM in ("light", "medium", "heavy"), f"J04: STRESS_MPLIGHT_FROM phải thuộc light/medium/heavy, nhận {STRESS_MPLIGHT_FROM!r}"
MPLIGHT_TRAIN_MODES = list(EVAL_MODES) if MPLIGHT_TRAIN_MODES is None else list(MPLIGHT_TRAIN_MODES)
assert MPLIGHT_TRAIN_MODES and len(set(MPLIGHT_TRAIN_MODES)) == len(MPLIGHT_TRAIN_MODES) and all(m in EVAL_MODES for m in MPLIGHT_TRAIN_MODES), \
    f"J04: MPLIGHT_TRAIN_MODES {MPLIGHT_TRAIN_MODES} phải là tập con (không trùng, không rỗng) của EVAL_MODES {EVAL_MODES} (mỗi mode train cần cửa sổ trong SCENARIO_WINDOWS)"
TRAIN_MODE = MPLIGHT_TRAIN_MODES[0]                # tương thích cũ
BENCH_MODES = list(EVAL_MODES) + ([STRESS_MODE] if INCLUDE_STRESS else [])    # EVAL_MODES = mode demand camera (engine dùng); BENCH_MODES thêm stress
# MPLIGHT_CKPT: None | str/Path (mọi mode) | dict {mode: path}. Chuẩn hoá; MPLIGHT_CKPT (biến cũ, cell 01b băm 1 đường dẫn) chỉ giữ dạng 1 đường dẫn chung.
MPLIGHT_CKPT_USER = MPLIGHT_CKPT
if isinstance(MPLIGHT_CKPT, dict):
    assert all(k in ("light", "medium", "heavy", STRESS_MODE) for k in MPLIGHT_CKPT), f"J04: khoá MPLIGHT_CKPT không hợp lệ: {list(MPLIGHT_CKPT)}"
    MPLIGHT_CKPT_BY_MODE, MPLIGHT_CKPT_ALL = {str(k): str(v) for k, v in MPLIGHT_CKPT.items() if v}, None
else:
    MPLIGHT_CKPT_BY_MODE, MPLIGHT_CKPT_ALL = {}, (str(MPLIGHT_CKPT) if MPLIGHT_CKPT else None)
MPLIGHT_CKPT = MPLIGHT_CKPT_ALL
# ---- J05: kiểm tra knob benchmark protocol ----
assert isinstance(BENCH_TRIPINFO, bool) and isinstance(LEARNING_CURVES, bool), f"J05: BENCH_TRIPINFO / LEARNING_CURVES phải là bool, nhận {BENCH_TRIPINFO!r} / {LEARNING_CURVES!r}"
assert ADAPTIVE_BETA in ("auto", True, False) and (ADAPTIVE_BETA == "auto" or isinstance(ADAPTIVE_BETA, bool)), f"J07: ADAPTIVE_BETA phải là 'auto' | True | False, nhận {ADAPTIVE_BETA!r}"   # J07
if MPLIGHT_TRAIN_SEEDS is None: MPLIGHT_TRAIN_SEEDS_EFF = [SEED]
else:
    assert isinstance(MPLIGHT_TRAIN_SEEDS, (list, tuple)) and len(MPLIGHT_TRAIN_SEEDS) >= 1 and all(isinstance(_s, int) and not isinstance(_s, bool) and _s >= 0 for _s in MPLIGHT_TRAIN_SEEDS), \
        f"J05: MPLIGHT_TRAIN_SEEDS phải là None hoặc list số nguyên >= 0, nhận {MPLIGHT_TRAIN_SEEDS!r}"
    assert len(set(MPLIGHT_TRAIN_SEEDS)) == len(MPLIGHT_TRAIN_SEEDS), f"J05: MPLIGHT_TRAIN_SEEDS có seed trùng: {MPLIGHT_TRAIN_SEEDS!r}"
    MPLIGHT_TRAIN_SEEDS_EFF = list(MPLIGHT_TRAIN_SEEDS)
assert isinstance(REPORT_BASELINE, str) and REPORT_BASELINE, f"J05: REPORT_BASELINE phải là tên controller (str), nhận {REPORT_BASELINE!r}"
if REPORT_BASELINE not in CONTROLLERS: print(f"J05: REPORT_BASELINE={REPORT_BASELINE!r} không có trong CONTROLLERS {CONTROLLERS} => bỏ qua improvement.csv")
def _j05_manual(_m):      # mode _m dùng checkpoint thủ công? (cùng thứ tự ưu tiên như mplight_ckpt() ở cell 08a)
    _s = STRESS_MPLIGHT_FROM if _m == STRESS_MODE else _m
    return bool(MPLIGHT_CKPT_BY_MODE.get(_m) or MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE.get(_s))
if len(MPLIGHT_TRAIN_SEEDS_EFF) > 1 and "mplight" in CONTROLLERS and DOWNSTREAM:
    _j05_bad = [_m for _m in BENCH_MODES if _j05_manual(_m)]
    if _j05_bad: raise RuntimeError(f"J05: MPLIGHT_TRAIN_SEEDS={MPLIGHT_TRAIN_SEEDS_EFF} (nhiều trial) KHÔNG dùng được cùng MPLIGHT_CKPT thủ công cho mode {_j05_bad} (checkpoint thủ công = 1 trial, không có train seed). Bỏ MPLIGHT_CKPT hoặc đặt MPLIGHT_TRAIN_SEEDS = None.")
CITY_FROM_IMAGES = IMAGES and RUN_MODE != "calibration_only"             # city_index + detection city từ ảnh
CITY_REUSE = CITY_FROM_IMAGES and CITY_DETS_CACHE is not None            # tái dùng cache detection city có sẵn (không detect)
CITY_DETECT = (RUN_MODE == "full_recompute" or CITY_FROM_IMAGES) and not CITY_REUSE    # chạy detector trên ảnh city
NEED_MODEL = RECOMPUTE or CITY_DETECT                                    # cần ultralytics + weights
HAS_CITY_DETS = RUN_MODE != "calibration_only"
assert SPLIT in ("train", "val", "test") and all(m in ("light", "medium", "heavy") for m in EVAL_MODES)
assert TRAIN_MODE in ("light", "medium", "heavy"), TRAIN_MODE
if not DOWNSTREAM and RUN_TRAIN:     # J04: training cần scenarios SUMO => calibration_only / image_review bỏ qua có chủ đích (không báo lỗi dù RUN_TRAIN mặc định True)
    print(f"J04: RUN_MODE={RUN_MODE} không chạy SUMO => RUN_TRAIN tự đặt False"); RUN_TRAIN = False
assert not RUN_TRAIN or "mplight" in CONTROLLERS, "J04: RUN_TRAIN=True nhưng 'mplight' không có trong CONTROLLERS => đặt RUN_TRAIN = False (chỉ baseline) hoặc thêm 'mplight' vào CONTROLLERS"
if DOWNSTREAM and "mplight" in CONTROLLERS and (MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE or RUN_TRAIN):
    def _j04_mpl_src(_m):     # nguồn checkpoint tĩnh cho mode _m (thứ tự giống mplight_ckpt() ở cell 08a)
        _s = STRESS_MPLIGHT_FROM if _m == STRESS_MODE else _m
        if MPLIGHT_CKPT_BY_MODE.get(_m) or MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE.get(_s): return "manual"
        return "train" if (RUN_TRAIN and _s in MPLIGHT_TRAIN_MODES) else None
    _j04_bad = [_m for _m in BENCH_MODES if _j04_mpl_src(_m) is None]
    if _j04_bad:
        raise RuntimeError(f"J04: MPLight không có checkpoint cho mode {_j04_bad}: mỗi mode trong BENCH_MODES {BENCH_MODES} cần (a) MPLIGHT_CKPT (str hoặc dict có mode đó), hoặc (b) RUN_TRAIN=True với mode đó trong MPLIGHT_TRAIN_MODES {MPLIGHT_TRAIN_MODES}. "
                           f"Với stress '{STRESS_MODE}': STRESS_MPLIGHT_FROM='{STRESS_MPLIGHT_FROM}' phải thuộc MPLIGHT_TRAIN_MODES (RUN_TRAIN=True) hoặc có trong MPLIGHT_CKPT; hoặc bỏ 'mplight' khỏi CONTROLLERS / INCLUDE_STRESS=False.")
ON_KAGGLE = Path("/kaggle").exists()
HERE = Path.cwd()
RUN_ID = os.environ.get("VIETTSC_RUN_ID", ("run_hybrid_cleaned_" if HYBRID else "run_") + dt.datetime.now().strftime("%Y%m%d_%H%M%S"))   # J01: backend nằm trong RUN_ID mặc định

assert re.fullmatch(r"[A-Za-z0-9._-]+", RUN_ID), f"RUN_ID chỉ gồm chữ/số/._-: {RUN_ID!r}"
def _now(): return _datetime.now(_tz.utc).isoformat(timespec="seconds")
T_START = _now()

def find_repo():
    c = [os.environ.get("VIETTSC_REPO")]
    if ON_KAGGLE:     # repo là Kaggle Dataset (read-only) hoặc đã clone vào /kaggle/working
        c += ["/kaggle/working/VietTSC"] + [str(p.parent) for pat in ("*/build_demand.py", "*/*/build_demand.py")
                                           for p in Path("/kaggle/input").glob(pat)]
    else:             # notebook nằm ở "Agent workspace/kaggle/"
        c += [str(HERE.parent.parent / "VietTSC-Benchmark-main"), str(HERE.parent / "VietTSC-Benchmark-main"),
              str(HERE / "VietTSC-Benchmark-main")]
    for p in c:
        if p and (Path(p) / "build_demand.py").is_file():
            return Path(p).resolve()
    raise FileNotFoundError("Không thấy REPO (build_demand.py). Đặt VIETTSC_REPO hoặc thêm repo làm Kaggle Dataset. Đã thử: %s" % [x for x in c if x])

REPO_DIR = find_repo()                                   # source gốc: CHỈ ĐỌC
WORK = Path("/kaggle/working") if ON_KAGGLE else HERE.parent / "scratch"
OUT_DIR = WORK / "out" / RUN_ID                          # artifacts của run này (mỗi run 1 thư mục, không ghi đè run cũ)
STAGE = WORK / "stage" / RUN_ID                          # working copy để chạy source gốc
# FAIL-FAST: RUN_ID đã có artifact => dừng TRƯỚC mọi thao tác ghi. Không overwrite / merge / resume / xoá run cũ.
def _files_under(d): d = Path(d); return [p for p in d.rglob("*") if p.is_file()] if d.exists() else []
_used = {str(d): len(_files_under(d)) for d in (OUT_DIR, STAGE) if _files_under(d)}
if _used:
    raise FileExistsError(f"RUN_ID '{RUN_ID}' đã có artifact {_used}. Không ghi đè/merge/resume: đặt os.environ['VIETTSC_RUN_ID'] khác (hoặc xoá thủ công run cũ nếu bạn chắc chắn).")
for sub in ("detections", "calibration", "flow", "bench", "logs"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
CAL_RUN_DIR = OUT_DIR / "calibration_runs" / RUN_ID          # calibration_only: toàn bộ artifacts + log của lần chạy này
if RUN_MODE == "calibration_only": CAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
META_PATH = (CAL_RUN_DIR if RUN_MODE == "calibration_only" else OUT_DIR) / "experiment_metadata.json"   # một file metadata / run

BASE = {  # dữ liệu baseline trong repo (đọc-chỉ)
    "calib_dir":       REPO_DIR / "data/calibration",
    "calib_dets":      REPO_DIR / "data/detections/calibration_yolo11x_1280.jsonl.gz",
    "city_dets":       REPO_DIR / "data/detections/city_cam_yolo11x_1280.jsonl.gz",
    "city_index":      REPO_DIR / "data/detections/city_cam_frames.jsonl",
    "calibration":     REPO_DIR / "flow_estimation/calibration/yolo11x_1280.json",
    "demand_settings": REPO_DIR / "results/flow/demand_settings.json",
    "weights":         REPO_DIR / "models/yolo11x.pt",
}
# calibration set đang dùng (videos.json, frames.jsonl, camera_folds.json, images/)
CALIB_DIR = (OUT_DIR / "calib_set") if REBUILD_CALIB_SET else (CALIB_SET_DIR if CALIB_SET_DIR else BASE["calib_dir"])   # J02E: CALIB_SET_DIR
CALIB_IMAGES_EFF = (CALIB_DIR / "images") if REBUILD_CALIB_SET else (CALIB_IMAGES or ((CALIB_SET_DIR / "images") if CALIB_SET_DIR and (CALIB_SET_DIR / "images").is_dir() else None))   # J02E: mặc định CALIB_SET_DIR/images
# detection cache + calibration ĐANG DÙNG — một nguồn sự thật cho mọi stage sau
# J02E: dets city: images => OUT_DIR/detections/city_<backend>.jsonl.gz (hoặc CITY_DETS_CACHE khi tái dùng); repo_cache => như cũ
_CITY_IMG = CITY_DETS_CACHE if CITY_REUSE else OUT_DIR / f"detections/city_{_DTAG}.jsonl.gz"
if IMAGE_REVIEW:     # J02E
    ACTIVE = dict(label=f"IMAGE_REVIEW ({DETECTOR_BACKEND}): detection city + đếm ROI từng frame; không calibration/flow/SUMO/benchmark",
                  dets_calib=None, dets_city=_CITY_IMG, calibration=None)
elif HYBRID and RUN_MODE == "baseline_cache":     # J01: artifact hybrid ĐÃ XÁC MINH (cell hybrid_backend / calibration_artifact); không bao giờ rơi về cache/β YOLO11x
    ACTIVE = dict(label=("HYBRID_CLEANED (baseline_cache): cache + calibration hybrid đã xác minh từ VIETTSC_HYBRID_CACHE_DIR" if not (IMAGES or CALIBRATION_ARTIFACT) else
                         "HYBRID_CLEANED (baseline_cache, J02E): " + ("calibration hybrid từ VIETTSC_CALIBRATION_ARTIFACT" if CALIBRATION_ARTIFACT else "calibration hybrid từ VIETTSC_HYBRID_CACHE_DIR") + ("; detection city từ ảnh CITY_DATASET" if IMAGES else "")),
                  dets_calib=(HYBRID_CACHE_DIR / "calib_hybrid_cleaned.jsonl.gz") if HYBRID_CACHE_DIR else None,
                  dets_city=_CITY_IMG if IMAGES else HYBRID_CACHE_DIR / "city_hybrid_cleaned.jsonl.gz",
                  calibration=CALIBRATION_ARTIFACT if CALIBRATION_ARTIFACT else HYBRID_CACHE_DIR / "hybrid_cleaned_calibration.json")
elif RUN_MODE == "baseline_cache":
    ACTIVE = dict(label=("BASELINE: YOLO11x cache của repo + calibration baseline của repo" if not (IMAGES or CALIBRATION_ARTIFACT) else
                         "BASELINE (J02E): " + ("calibration yolo11x từ VIETTSC_CALIBRATION_ARTIFACT" if CALIBRATION_ARTIFACT else "calibration yolo11x của repo") + ("; detection city từ ảnh CITY_DATASET" if IMAGES else "; cache city của repo")),
                  dets_calib=BASE["calib_dets"], dets_city=_CITY_IMG if IMAGES else BASE["city_dets"], calibration=CALIBRATION_ARTIFACT or BASE["calibration"])
elif RUN_MODE == "calibration_only":
    ACTIVE = dict(label=("CALIBRATION-ONLY (HYBRID_CLEANED): YOLO11x car/truck/bus + BMD45 motorcycle + cleanup S0-G chạy lại trên calibration snapshots + calibration mới (calibration_runs/<RUN_ID>)" if HYBRID
                         else "CALIBRATION-ONLY: YOLO11x chạy lại trên calibration snapshots + calibration mới (calibration_runs/<RUN_ID>)"),
                  dets_calib=CAL_RUN_DIR / "detections.jsonl.gz", dets_city=None, calibration=CAL_RUN_DIR / "calibration.json")
else:
    ACTIVE = dict(label=("RECOMPUTED (HYBRID_CLEANED): YOLO11x car/truck/bus + BMD45 motorcycle + cleanup S0-G chạy lại + calibration mới (trong OUT_DIR)" if HYBRID
                         else "RECOMPUTED: YOLO11x chạy lại + calibration mới (trong OUT_DIR)"),
                  dets_calib=OUT_DIR / f"detections/calib_{_DTAG}.jsonl.gz", dets_city=_CITY_IMG,
                  calibration=OUT_DIR / f"calibration/{_DTAG}_recomputed.json")
ACTIVE["calibration_provenance"] = Path(str(ACTIVE["calibration"]) + ".provenance.json") if (HYBRID and ACTIVE["calibration"] is not None and not Path(ACTIVE["calibration"]).is_dir()) else None   # J01: sidecar xác định detector (nhãn trong calibration JSON là nhãn cũ); J02E: dir artifact => cell 01c
FLOW_DIR = OUT_DIR / "flow"
STATUS = {k: "NOT RUN" for k in ("preflight", "stage", "calib_set", "detection", "calibration", "calibration_report", "infer", "settings", "routes",
                                 "scenario_check", "controllers", "benchmark")}
STATUS["directional"] = "NOT RUN" if J02 else "NOT REQUESTED (DEMAND_MODE=baseline_5050)"   # J02
STATUS["directional_targets"] = "NOT RUN" if J02 else "NOT REQUESTED (DEMAND_MODE=baseline_5050)"   # J02
STATUS["training"] = "NOT RUN" if RUN_TRAIN else "NOT REQUESTED (RUN_TRAIN=False)"
STATUS["hybrid_backend"] = "NOT RUN" if HYBRID else "NOT REQUESTED (DETECTOR_BACKEND=yolo11x)"          # J01
STATUS["hybrid_diagnostics"] = ("NOT RUN" if EXPORT_HYBRID_DIAGNOSTICS else "SKIPPED (EXPORT_HYBRID_DIAGNOSTICS=False)") if HYBRID else "NOT REQUESTED (DETECTOR_BACKEND=yolo11x)"   # J01
STATUS["input_discovery"] = "NOT RUN" if CITY_FROM_IMAGES else ("NOT REQUESTED (calibration_only)" if RUN_MODE == "calibration_only" else "NOT REQUESTED (CITY_SOURCE=repo_cache)")   # J02E
STATUS["roi_frames"] = "NOT RUN" if HAS_CITY_DETS else f"SKIPPED ({RUN_MODE}: by design)"   # J02E
STATUS["adaptive_beta"] = "NOT RUN" if DOWNSTREAM else f"SKIPPED ({RUN_MODE}: by design)"   # J07
ADAPTIVE_BETA_EFF = False   # J07: quyết định hiệu lực (đặt bởi j07_adaptive_on() ở cell flow infer); reset mỗi lần chạy cell 00a
if IMAGE_REVIEW:     # J02E: bỏ qua có chủ đích (không phải lỗi); cell cuối KHÔNG coi các stage này là bắt buộc
    for _k in ("calib_set", "calibration", "calibration_report", "infer", "settings", "routes", "scenario_check", "controllers", "benchmark", "directional", "directional_targets"):
        STATUS[_k] = "SKIPPED (image_review: by design)"
for _v in ("SCEN", "BENCH_SCEN", "TRAIN_SCEN", "TRAIN_SCEN_BY_MODE", "TRAIN_RUN_DIRS", "TRAIN_SUMMARY", "MPLIGHT_SELECTED", "MPLIGHT_SELECTION", "PLAN_RUNS", "SKIPPED", "RUN_DIRS", "EXPECTED_RUNS", "TRAIN_RUN_DIR", "MPLIGHT_SELECTED_TRIALS", "MPLIGHT_SELECTION_TRIALS", "TRAIN_RUN_DIRS_TRIALS", "TRAIN_BINDING_TRIALS", "LEARNING_CURVE_FILES", "IMPROVEMENT_WRITTEN", "TRIALS_TABLE_WRITTEN", "WEIGHTS_USED", "WEIGHTS_SOURCE", "HYBRID_WEIGHTS", "HYBRID_PROV", "DIAG_DIR", "J02_SETTINGS", "J02_TARGETS", "EXPECT_BINDING_PATH", "TRAIN_BINDING", "CITY_INDEX", "CITY_ROOT", "CITY_REPORT", "CAL_RESOLVED", "CAL_PATH_IN", "GROUND_TRUTH_KIND", "ROI_OUT", "SYNC_STAMP"):   # J02E
    globals().pop(_v, None)   # reset biến run-specific: kernel dùng lại cho RUN_ID mới không được mang scenario/controller của run trước vào metadata
FAILED = {}        # stage -> lý do lỗi (do stage() ghi)
BACKENDS = {}      # run -> backend SUMO thực tế (traci|libsumo), đọc từ run_tsc

def tree_sha(root):
    root = Path(root); h = {}
    for p in sorted(root.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            h[p.relative_to(root).as_posix()] = hashlib.sha1(p.read_bytes()).hexdigest()
    return h
SRC_SNAPSHOT = tree_sha(REPO_DIR)           # để chứng minh REPO không bị đổi (cell 10)

def require(p, what=""):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"THIẾU {what or 'file'}: {p}")
    return p

def _env():
    e = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    if globals().get("SUMO_HOME"): e["SUMO_HOME"] = SUMO_HOME
    return e

def sh(args, cwd=None, log=None):
    """Chạy lệnh con (mặc định trong STAGE), stream output vào cell, dừng nếu lỗi."""
    args = [str(a) for a in args]
    print(">", " ".join(args), flush=True)
    lines = []
    with subprocess.Popen(args, cwd=str(cwd or STAGE), env=_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as pr:
        for line in pr.stdout:
            print(line.rstrip(), flush=True); lines.append(line)
    if log: Path(log).parent.mkdir(parents=True, exist_ok=True); Path(log).write_text("$ " + " ".join(args) + "\n" + "".join(lines) + f"\n[exit code {pr.returncode}]\n", encoding="utf-8")   # stdout+stderr gộp, ghi cả khi lỗi
    if pr.returncode:
        raise RuntimeError(f"lệnh lỗi (exit {pr.returncode}): {' '.join(args)}")
    return "".join(lines)

def pymod(mod, *a): return sh([sys.executable, "-m", mod, *a])
def pyscript(script, *a, log=None): return sh([sys.executable, script, *a], log=log)
print("RUN_MODE :", RUN_MODE, "|", ACTIVE["label"]); print("BACKEND  :", DETECTOR_BACKEND)   # J01
print("DEMAND   :", DEMAND_MODE, "(J02)")
print("CITY     :", CITY_SOURCE, ("| dataset " + str(CITY_DATASET) + (" | dets cache tái dùng " + str(CITY_DETS_CACHE) if CITY_REUSE else "") if IMAGES else ""), "| windows", SCENARIO_WINDOWS, "| EVAL_MODES", EVAL_MODES)   # J02E
print("REPO_DIR :", REPO_DIR, "(chỉ đọc)"); print("STAGE    :", STAGE); print("OUT_DIR  :", OUT_DIR)
print("J04      : BENCH_MODES", BENCH_MODES, "| EVAL_EPISODES", EVAL_EPISODES, "| controllers", CONTROLLERS,
      "| RUN_TRAIN", RUN_TRAIN, ("(modes %s, %d round, test_every %d)" % (MPLIGHT_TRAIN_MODES, TRAIN_ROUNDS, TRAIN_TEST_EVERY)) if RUN_TRAIN else "",
      "| MPLIGHT_CKPT", MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE or "auto/none", "| INCLUDE_STRESS", INCLUDE_STRESS)
print("J05      : BENCH_TRIPINFO", BENCH_TRIPINFO, "| MPLIGHT_TRAIN_SEEDS", MPLIGHT_TRAIN_SEEDS_EFF, "| REPORT_BASELINE", REPORT_BASELINE, "| LEARNING_CURVES", LEARNING_CURVES)
print("J07      : ADAPTIVE_BETA", repr(ADAPTIVE_BETA), "(auto = bật nếu calibration có khối adaptive_beta; quyết định hiệu lực in ở stage infer)")   # J07

# ============================================================
# J00a — F1/F2/F3 helpers (độ tin cậy; thuật toán calibrate, multiprocessing.Pool và tham số --workers KHÔNG đổi)
# ============================================================
# J00a F1: probe chạy trong Python SUBPROCESS (cùng sys.executable, env=_env(), cwd như lệnh calibrate). KHÔNG kiểm tra start method của Jupyter kernel.
# calibrate.py dùng global module `_G` + `Pool(workers)` + hàm worker ở module scope => interpreter con phải mặc định 'fork'. Probe tạo Pool 2 worker rất nhỏ cùng mẫu đó.
_J00A_PROBE = r'''
import json, multiprocessing, sys
_G = {}
def _w(x): return _G["k"] + x
if __name__ == "__main__":
    sm = multiprocessing.get_start_method()
    out = {"start_method": sm, "python": sys.executable, "ok": False}
    if sm != "fork":
        out["error"] = "default start method of this interpreter is %r; calibrate.py (module-global _G + Pool) needs 'fork'" % sm
    else:
        try:
            _G["k"] = 10
            with multiprocessing.Pool(2) as p: out["result"] = p.map(_w, [1, 2, 3], chunksize=1)
            out["ok"] = out["result"] == [11, 12, 13]
        except Exception as e: out["error"] = type(e).__name__ + ": " + str(e)[:200]
    print(json.dumps(out))
'''
def _j00a_probe_mp_fork(cwd):
    try:
        r = subprocess.run([sys.executable, "-c", _J00A_PROBE], cwd=str(cwd), env=_env(), capture_output=True, text=True, timeout=60)
        if r.returncode != 0: return {"ok": False, "start_method": None, "python": sys.executable, "error": f"exit {r.returncode}: {(r.stderr or r.stdout)[-300:]}"}
        out = json.loads(r.stdout.strip().splitlines()[-1])
    except Exception as e:
        return {"ok": False, "start_method": None, "python": sys.executable, "error": f"{type(e).__name__}: {e}"}
    return out

# J00a F3: weights thật (không phải LFS pointer), không bao giờ truyền đường dẫn thiếu cho Ultralytics (nó có thể tự tải)
def _j00a_require_real_weights(path, label):
    p = Path(path)
    if not p.is_file(): raise RuntimeError(f"{label}: không phải file / không tồn tại: {p} (không tải tự động, không fallback)")
    if p.stat().st_size < 4096 and p.read_bytes()[:64].startswith(b"version https://git-lfs"): raise RuntimeError(f"{label}: {p} là Git LFS pointer, không phải weights thật")
    return p

# J00a F2: recorder sớm, dùng được từ preflight/staging (trước khi cell 01b định nghĩa stage()/write_run_metadata). Ghi atomic; KHÔNG che lỗi gốc (caller vẫn raise).
def _j00a_record_failure(key, et, ev):
    msg = f"{et.__name__}: {ev}"
    try:
        FAILED[key] = msg; STATUS[key] = "FAILED: " + msg
        if "write_run_metadata" in globals():           # cell 01b đã chạy: dùng helper đầy đủ
            try: write_run_metadata(f"FAILED at {key}: {msg}"); return
            except Exception: pass
        rec = {"run_id": RUN_ID, "run_mode": RUN_MODE, "status": "FAILED", "execution_status": f"FAILED at {key}: {msg}"[:1500], "failed_stage": key,
               "error_type": et.__name__, "error_message": str(ev)[:1500], "timestamp": _now(), "recorder": "J00a early_stage (provenance đầy đủ chỉ có khi cell 01b đã chạy)"}
        META_PATH.parent.mkdir(parents=True, exist_ok=True); tmp = META_PATH.with_name(META_PATH.name + ".tmp")
        tmp.write_text(json.dumps(rec, indent=1, ensure_ascii=False), encoding="utf-8"); os.replace(tmp, META_PATH)
    except Exception as e2:
        print("J00a: không ghi được metadata FAILED sớm:", repr(e2))

class early_stage:
    def __init__(self, key): self.key = key
    def __enter__(self): return self
    def __exit__(self, et, ev, tb):
        if et is not None and not issubclass(et, (KeyboardInterrupt, SystemExit)): _j00a_record_failure(self.key, et, ev)
        return False

# ============================================================
# J07 — Power-law Adaptive Beta: quyết định bật/tắt theo calibration ĐANG DÙNG cho inference (ACTIVE["calibration"])
# ============================================================
def j07_adaptive_on(cal_path, quiet=False):   # J07
    """Đọc calibration JSON dùng cho inference; trả về True/False theo knob ADAPTIVE_BETA, lưu ADAPTIVE_BETA_EFF + STATUS["adaptive_beta"].
    True + thiếu khối "adaptive_beta" => RuntimeError. "auto" + thiếu => tắt, in chú thích (beta cố định)."""
    global ADAPTIVE_BETA_EFF
    p = Path(cal_path)
    if not p.is_file(): raise FileNotFoundError(f"J07: không thấy calibration JSON: {p}")
    cal = json.load(open(p, encoding="utf-8"))
    blk = cal.get("adaptive_beta") if isinstance(cal, dict) else None
    has = isinstance(blk, dict) and isinstance(blk.get("gamma"), dict)
    if ADAPTIVE_BETA is False:
        eff, st = False, "off (ADAPTIVE_BETA=False) — fixed beta"
    elif has:
        eff = True
        st = "on (gamma " + ", ".join(f"{k} {v}" for k, v in blk["gamma"].items()) + f"; gamma_source {blk.get('gamma_source')})"
    elif ADAPTIVE_BETA is True:
        raise RuntimeError(f"J07: ADAPTIVE_BETA=True nhưng calibration {p} không có khối 'adaptive_beta'. Dùng calibration có khối này (vd yolo11x_1280_adaptive.json hoặc calibration_only mới) hoặc đặt ADAPTIVE_BETA='auto'/False.")
    else:
        eff, st = False, "off (calibration has no adaptive_beta block) — fixed beta"
        if not quiet: print("J07 adaptive beta: OFF (calibration has no adaptive_beta block) — fixed beta")
    ADAPTIVE_BETA_EFF = eff; STATUS["adaptive_beta"] = st
    if not quiet: print("J07 adaptive beta:", st, "|", p)
    return eff

In [ ]:
# J00a: F2 — lỗi cài package được ghi vào metadata FAILED
with early_stage("preflight"):
    # 00b. (tuỳ chọn) cài CHỈ các package còn thiếu. Không đụng torch / SUMO nếu đã có.
    NEEDED = [("numpy", "numpy"), ("scipy", "scipy")]
    if DOWNSTREAM: NEEDED += [("sumolib", "sumolib"), ("traci", "traci"), ("sumo", "eclipse-sumo")]
    if NEED_MODEL: NEEDED += [("ultralytics", "ultralytics")]      # J02E: cả detection city từ ảnh
    NEEDED += [("PIL", "pillow")]                                  # J02E: city_index / roi_frames
    import importlib.metadata as _imd      # J02E fix4: find_spec("sumo") thấy namespace package STAGE/sumo của lần chạy trước => không dùng cho eclipse-sumo
    def _j02e_present(mod, pip):
        if mod != "sumo": return importlib.util.find_spec(mod) is not None
        try: _imd.version("eclipse-sumo"); return True
        except Exception: pass
        _sh = os.environ.get("SUMO_HOME")
        return bool(_sh) and any((Path(_sh) / "bin" / _n).is_file() for _n in ("sumo", "sumo.exe"))
    missing = [pip for mod, pip in NEEDED if not _j02e_present(mod, pip)]
    if missing and INSTALL_MISSING:
        sh([sys.executable, "-m", "pip", "install", "-q", *missing], cwd=HERE)
        importlib.invalidate_caches()
    else:
        print("thiếu:", missing or "không", "| INSTALL_MISSING =", INSTALL_MISSING)
    # J04: libsumo (tuỳ chọn, tăng tốc SUMO). Cài trong try/except RIÊNG: thất bại KHÔNG làm hỏng preflight.
    LIBSUMO_AVAILABLE = None
    if DOWNSTREAM:
        def _j04_libsumo_ok():     # kiểm tra trong subprocess (không nạp libsumo vào kernel)
            import tempfile
            _pd = tempfile.mkdtemp(prefix="viettsc_libsumo_probe_")      # J04b: cwd rỗng — tránh file/thư mục 'libsumo' trong cwd che package thật
            try: return subprocess.run([sys.executable, "-c", "import libsumo"], cwd=_pd, env=_env(), capture_output=True, timeout=120).returncode == 0
            except Exception: return False
            finally: shutil.rmtree(_pd, ignore_errors=True)
        LIBSUMO_AVAILABLE = _j04_libsumo_ok()
        if not LIBSUMO_AVAILABLE and INSTALL_LIBSUMO and INSTALL_MISSING:
            try:
                sh([sys.executable, "-m", "pip", "install", "-q", "libsumo"], cwd=HERE)
                importlib.invalidate_caches(); LIBSUMO_AVAILABLE = _j04_libsumo_ok()
            except Exception as _j04e:
                print("CẢNH BÁO (J04): không cài được libsumo:", repr(_j04e)[:300])
        if not LIBSUMO_AVAILABLE:
            print("CẢNH BÁO (J04): libsumo không có => run_tsc dùng traci (tsc/env.py:75-84), chậm hơn ~10x theo README. Training 100 round sẽ rất lâu.")
        print("libsumo:", "available" if LIBSUMO_AVAILABLE else "not available")


In [ ]:
# J00a: F2 — preflight bọc để lỗi được ghi vào experiment metadata (FAILED)
with early_stage("preflight"):
    # 00c. Preflight: môi trường + dependency BẮT BUỘC + dữ liệu. Thiếu gì -> dừng với lỗi rõ ràng (Run All không "thành công giả").
    def ver(pkg):
        try: return im.version(pkg)
        except Exception: return None
    print("OS:", platform.platform(), "| python", platform.python_version(), "| Kaggle:", ON_KAGGLE)
    for p in ("torch", "numpy", "scipy", "ultralytics", "eclipse-sumo", "sumolib", "traci", "libsumo"):
        print(f"  {p:13s}", ver(p) or ("(có thể nằm trong SUMO_HOME/tools)" if importlib.util.find_spec(p.replace('-', '_')) else "-- chưa cài"))
    try:
        import torch
        gpu = torch.cuda.is_available()
        print("  GPU:", torch.cuda.get_device_name(0) if gpu else "không có",
              "| GPU chỉ phục vụ detection/training; SUMO mô phỏng chạy CPU headless (không GUI)")
    except Exception as e: print("  torch lỗi:", e)

    SUMO_HOME = os.environ.get("SUMO_HOME")
    if not SUMO_HOME:
        try: import sumo; SUMO_HOME = getattr(sumo, "SUMO_HOME", None)      # J02E fix3: namespace 'sumo' (STAGE/sumo trên sys.path) không có SUMO_HOME
        except Exception: pass
    if not SUMO_HOME:      # J02E fix4: import sumo có thể là namespace (STAGE/sumo) -> tìm thư mục eclipse-sumo đã cài qua metadata
        try:
            _cand = Path(im.distribution("eclipse-sumo").locate_file("sumo"))
            if any((_cand / "bin" / _n).is_file() for _n in ("sumo", "sumo.exe")): SUMO_HOME = str(_cand)
        except Exception: pass
    if SUMO_HOME: os.environ["SUMO_HOME"] = SUMO_HOME
    def which_sumo(name):
        return shutil.which(name) or next((str(p) for p in [Path(SUMO_HOME or ".") / "bin" / name, Path(SUMO_HOME or ".") / "bin" / (name + ".exe")] if p.is_file()), None)
    tools = Path(SUMO_HOME or ".") / "tools"
    hard = {"torch": importlib.util.find_spec("torch") is not None, "numpy": importlib.util.find_spec("numpy") is not None,
            "scipy": importlib.util.find_spec("scipy") is not None}
    if DOWNSTREAM:      # SUMO chỉ cần khi có stage sau calibration
        hard.update({
            "SUMO_HOME": bool(SUMO_HOME), "sumo (binary)": which_sumo("sumo"), "duarouter (binary, randomTrips)": which_sumo("duarouter"),
            "tools/routeSampler.py": (tools / "routeSampler.py").is_file(), "tools/randomTrips.py": (tools / "randomTrips.py").is_file(),
            "sumolib": importlib.util.find_spec("sumolib") is not None,
            "traci|libsumo": importlib.util.find_spec("traci") is not None or importlib.util.find_spec("libsumo") is not None,
        })
    if NEED_MODEL: hard["ultralytics"] = importlib.util.find_spec("ultralytics") is not None      # J02E: cả detection city từ ảnh
    if CITY_FROM_IMAGES:      # J02E
        hard["CITY_DATASET (images)"] = bool(CITY_DATASET and Path(CITY_DATASET).exists())
        if CITY_DETS_CACHE is not None: hard["VIETTSC_CITY_DETS_CACHE"] = Path(CITY_DETS_CACHE).is_file()
        hard["PIL (city_index/roi_frames)"] = importlib.util.find_spec("PIL") is not None
    if RECOMPUTE:
        if DOWNSTREAM and not IMAGES: hard["CITY_IMAGES"] = bool(CITY_IMAGES and CITY_IMAGES.exists())      # J02E: images dùng CITY_DATASET
        if REBUILD_CALIB_SET:
            hard.update({"VIETTSC_TRACKS": bool(TRACKS_DIR and TRACKS_DIR.exists()), "VIETTSC_MAPKF": bool(MAPKF_DIR and MAPKF_DIR.exists()),
                         "VIETTSC_KEYFRAMES": bool(KEYFRAMES_DIR and KEYFRAMES_DIR.exists()), "VIETTSC_CAMERA_FOLDS": bool(CAMERA_FOLDS and CAMERA_FOLDS.is_file())})
        else:
            hard["CALIB_IMAGES (effective: VIETTSC_CALIB_IMAGES hoặc CALIB_SET_DIR/images)"] = bool(CALIB_IMAGES_EFF and Path(CALIB_IMAGES_EFF).exists())      # J02E fix1
    if RECOMPUTE:      # J00a: F1 — calibrate.py dùng global _G + Pool => cần 'fork' TRONG Python subprocess chạy calibrate; probe subprocess (không phải kernel), TRƯỚC detection
        _mp = _j00a_probe_mp_fork(OUT_DIR)
        print("J00a probe multiprocessing (subprocess):", json.dumps(_mp, ensure_ascii=False))
        hard[f"calibrate subprocess: multiprocessing fork + Pool (start_method={_mp.get('start_method')}; {_mp.get('error', 'ok')})"] = bool(_mp.get("ok"))
    if RUN_MODE == "calibration_only":
        hard["GPU/CUDA cho detection (REQUIRE_GPU)"] = (not REQUIRE_GPU) or bool(globals().get("gpu"))
        if not REBUILD_CALIB_SET:
            for _n in ("videos.json", "frames.jsonl", "camera_folds.json"): hard[f"calibration set: {_n}"] = (CALIB_DIR / _n).is_file()
            _fr0 = json.loads(open(CALIB_DIR / "frames.jsonl", encoding="utf-8").readline())["image"] if (CALIB_DIR / "frames.jsonl").is_file() else None
            hard["ảnh calibration mẫu (frames.jsonl dòng 1)"] = bool(_fr0 and CALIB_IMAGES_EFF and (CALIB_IMAGES_EFF / _fr0).is_file())
    src_need = {"flow_estimation/": REPO_DIR / "flow_estimation"}
    if HYBRID: src_need.update({f"flow_estimation/{n}": REPO_DIR / "flow_estimation" / n for n in ("hybrid_detector.py", "hybrid_motorcycle_cleanup.py", "hybrid_diagnostics.py", "hybrid_cleanup_config.json")})   # J01
    if REBUILD_CALIB_SET: src_need["build_calibration_set.py"] = REPO_DIR / "flow_estimation/build_calibration_set.py"
    if DOWNSTREAM: src_need.update({"build_demand.py": REPO_DIR / "build_demand.py", "run_tsc.py": REPO_DIR / "run_tsc.py", "tsc/": REPO_DIR / "tsc",
                "sumo/map/hcmc.net.xml.gz": REPO_DIR / "sumo/map/hcmc.net.xml.gz", "sumo/config/junctions.json": REPO_DIR / "sumo/config/junctions.json",
                "sumo/config/camera_segments.json": REPO_DIR / "sumo/config/camera_segments.json",
                "calibration (ACTIVE)": ACTIVE["calibration"] if RUN_MODE == "baseline_cache" else (BASE["calib_dir"] / "videos.json" if not REBUILD_CALIB_SET else REPO_DIR / "flow_estimation/build_calibration_set.py")})
    if DOWNSTREAM and not IMAGES: src_need.update({"city_index": BASE["city_index"], "dets_city (ACTIVE)": ACTIVE["dets_city"] if RUN_MODE == "baseline_cache" else BASE["city_index"]})   # J02E: images không dùng index/cache của repo
    if HAS_CITY_DETS: src_need.update({f"flow_estimation/{n}": REPO_DIR / "flow_estimation" / n for n in ("city_index.py", "camera_aliases.json", "roi_frames.py", "calibration_artifact.py", "roi.py", "directional_roi.json")})   # J02E
    hard.update({k: v.exists() for k, v in src_need.items()})
    print("\n-- kiểm tra bắt buộc")
    for k, v in hard.items(): print(f"  [{'ok' if v else 'THIẾU'}] {k}" + (f"  {v}" if isinstance(v, str) else ""))
    bad = [k for k, v in hard.items() if not v]
    if bad: raise RuntimeError("Preflight FAIL — thiếu bắt buộc: " + ", ".join(bad))
    w = BASE["weights"]; w_ptr = w.is_file() and w.stat().st_size < 1024 and w.read_bytes().startswith(b"version https://git-lfs")
    print("\nweights YOLO11x trong repo:", "LFS pointer (không dùng được; detect sẽ dùng 'yolo11x.pt' tải về)" if w_ptr else ("file thật" if w.is_file() else "không có"))
    if NEED_MODEL and not HYBRID:     # J02E: (J00a F3) cả detection city từ ảnh — mọi RUN_MODE có detection mới (calibration_only VÀ full_recompute) dùng cùng quy tắc weights; ALLOW_WEIGHTS_DOWNLOAD=False => thiếu weights thật = fail-fast; weights YOLO11x thật, hoặc tải được (J01: hybrid_cleaned không tải gì, kiểm riêng bên dưới)
        if YOLO_WEIGHTS:
            if not YOLO_WEIGHTS.is_file(): raise RuntimeError(f"VIETTSC_YOLO11X không phải file: {YOLO_WEIGHTS}")
            if YOLO_WEIGHTS.stat().st_size < 1024 and YOLO_WEIGHTS.read_bytes().startswith(b"version https://git-lfs"): raise RuntimeError("VIETTSC_YOLO11X là LFS pointer, không phải weights thật")
            print("weights:", YOLO_WEIGHTS)
        elif w.is_file() and not w_ptr: print("weights: models/yolo11x.pt trong repo (file thật)")
        else:
            if not ALLOW_WEIGHTS_DOWNLOAD: raise RuntimeError("Không có weights YOLO11x thật: đặt VIETTSC_YOLO11X hoặc ALLOW_WEIGHTS_DOWNLOAD=True")
            import urllib.request
            try: urllib.request.urlopen("https://github.com", timeout=8)
            except Exception as e: raise RuntimeError(f"Không có weights thật và không có Internet để tải yolo11x.pt ({e}). Đặt VIETTSC_YOLO11X hoặc bật Internet.")
            print("weights: sẽ tải 'yolo11x.pt' (Ultralytics) -- Internet OK")
    # J01: hybrid_cleaned — hai checkpoint THẬT phải có (không tải tự động, không fallback). SHA-256 + size được kiểm sau staging bằng flow_estimation.hybrid_detector --verify-only.
    if HYBRID and NEED_MODEL:      # J02E
        _bm = BMD45_WEIGHTS or (REPO_DIR / "models/yolov12x_bmd45.pt")
        for _lbl, _pth in (("YOLO11x (VIETTSC_YOLO11X)", YOLO_WEIGHTS), ("BMD45 (VIETTSC_BMD45 hoặc REPO/models/yolov12x_bmd45.pt)", _bm)):
            if not _pth or not Path(_pth).is_file(): raise RuntimeError(f"J01 hybrid_cleaned: thiếu weights thật {_lbl}: {_pth}")
            if Path(_pth).stat().st_size < 4096 and Path(_pth).read_bytes()[:64].startswith(b"version https://git-lfs"): raise RuntimeError(f"J01 hybrid_cleaned: {_lbl} là Git LFS pointer, không phải weights thật")
        print("J01 hybrid_cleaned: 2 checkpoint có mặt; ALLOW_WEIGHTS_DOWNLOAD không áp dụng (hybrid không tải weights)")
    STATUS["preflight"] = "done"

In [ ]:
# J00a: F2 — staging bọc để lỗi được ghi vào experiment metadata (FAILED)
with early_stage("stage"):
    # 00d. Staging copy: code của REPO (giữ nguyên byte) + sumo/{map,config,bicycle}. KHÔNG copy sumo/scenarios, sumo/demand_cache, results, runs, data, models.
    STAGE_INCLUDE = ["build_demand.py", "run_tsc.py", "requirements.txt", "flow_estimation", "tsc", "sumo/map", "sumo/config", "sumo/bicycle"]
    if any(p.is_file() for p in STAGE.rglob("*")):      # STAGE đã có file => không reuse/ghi đè
        raise FileExistsError(f"STAGE đã có file: {STAGE}. Không tái sử dụng; đổi VIETTSC_RUN_ID.")
    for rel in STAGE_INCLUDE:
        s, d = REPO_DIR / rel, STAGE / rel
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(s, d, ignore=shutil.ignore_patterns("__pycache__")) if s.is_dir() else shutil.copy2(s, d)
    # bằng chứng: mọi file đã stage == file trong REPO (không patch source)
    src_h = SRC_SNAPSHOT
    stage_h = {k: v for k, v in tree_sha(STAGE).items() if k in src_h}
    diff = [k for k, v in stage_h.items() if v != src_h[k]]
    assert not diff, f"file trong STAGE khác REPO: {diff}"
    assert not (STAGE / "sumo/scenarios").exists() or RUN_ID in str(STAGE), "STAGE không được chứa scenarios cũ"
    print(f"STAGE ok: {len(stage_h)} file giống hệt REPO; scenarios/results cũ KHÔNG có trong STAGE:",
          not (STAGE / "sumo/scenarios").exists() or "(scenarios của run này)")
    STATUS["stage"] = "done"

### 01b — Experiment metadata helper
Glue ghi `experiment_metadata.json` (chỉ calibration_only). Đọc hằng số `GAPS`/`CONFS` **từ source** (`flow_estimation.calibrate`) và số liệu từ artifacts; thông tin không có → `NOT_AVAILABLE`/`null`. `detector_name` lấy từ JSON nhưng nhãn này do `calibrate.py:112` ghi cứng, không suy ra từ weights — dùng `weights_path` + `weights_sha256` để phân biệt detector.

In [ ]:
# 01b. Metadata + provenance (MỘT cơ chế) và guard `stage()`.
# experiment_metadata.json: calibration_only -> <OUT_DIR>/calibration_runs/<RUN_ID>/ ; mode khác -> <OUT_DIR>/. Gọi nhiều lần: RUNNING -> COMPLETE | FAILED ...
NA = "NOT_AVAILABLE"

def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

def _hash(p):                       # sha256 nếu là file thật, ngược lại NOT_AVAILABLE (không đoán, không hash giả)
    try:
        p = Path(p); return _sha256(p) if p.is_file() else NA
    except Exception: return NA

def _nlines(p):
    try:
        with open(p, encoding="utf-8") as fh: return sum(1 for _ in fh)
    except OSError: return None

def _ver(pkg):
    try: return im.version(pkg)
    except Exception: return NA

def _env_meta():
    sumo_v = _ver("eclipse-sumo")
    if sumo_v == NA and globals().get("which_sumo"):        # không có metadata của package -> hỏi binary (--version, không mô phỏng)
        try:
            b = which_sumo("sumo")
            if b: sumo_v = subprocess.run([b, "--version"], capture_output=True, text=True, timeout=20).stdout.splitlines()[0].strip()
        except Exception: pass
    cuda = NA
    try:
        import torch; cuda = torch.version.cuda or NA
    except Exception: pass
    return dict(python=platform.python_version(), platform=platform.platform(), on_kaggle=ON_KAGGLE,
                torch=_ver("torch"), torch_cuda=cuda, ultralytics=_ver("ultralytics"), numpy=_ver("numpy"), scipy=_ver("scipy"),
                sumo=sumo_v, sumolib=_ver("sumolib"), traci_package=_ver("traci"), libsumo_package=_ver("libsumo"),
                sumo_backend_by_run=dict(BACKENDS))          # backend thực tế mỗi run (NOT_AVAILABLE nếu chưa chạy)

_SRC_KEYS = ["flow_estimation/detect.py", "flow_estimation/calibrate.py", "flow_estimation/infer.py", "flow_estimation/window.py", "flow_estimation/common.py",
             "build_demand.py", "run_tsc.py", "flow_estimation/roi.py", "flow_estimation/city_index.py", "flow_estimation/camera_aliases.json", "flow_estimation/roi_frames.py", "flow_estimation/calibration_artifact.py", "flow_estimation/directional.py", "flow_estimation/directional_roi.json", "sumo/config/camera_directions.json", "sumo/config/camera_segments.json", "tsc/env.py", "tsc/agents.py", "tsc/config.py", "tsc/junctions.py"]

def _user_ctrl():
    if not USER_AGENT_MODULE: return None
    try:
        sp = importlib.util.find_spec(USER_AGENT_MODULE); o = sp.origin if sp else None
        return dict(name=USER_AGENT_NAME, module=USER_AGENT_MODULE, factory=USER_AGENT_FACTORY, module_file=o or NA, module_sha256=_hash(o) if o else NA)
    except Exception: return dict(name=USER_AGENT_NAME, module=USER_AGENT_MODULE, factory=USER_AGENT_FACTORY, module_sha256=NA)

def _j01_hybrid_meta():
    """J01: provenance đầy đủ của backend hybrid_cleaned (đọc các sidecar; nhãn detector trong calibration JSON là nhãn cũ hard-code của calibrate.py)"""
    if DETECTOR_BACKEND != "hybrid_cleaned": return NA
    def _rd(p):
        try: return json.load(open(p, encoding="utf-8")) if p and Path(p).is_file() else NA
        except Exception as e: return f"không đọc được ({e})"
    return dict(detection_cache_calibration_provenance=_rd(str(ACTIVE["dets_calib"]) + ".provenance.json"),
                detection_cache_city_provenance=_rd(str(ACTIVE["dets_city"]) + ".provenance.json") if ACTIVE["dets_city"] else NA,
                calibration_provenance=_rd(ACTIVE.get("calibration_provenance")),
                verified_weights=(globals().get("HYBRID_PROV") or NA))


def _j02e_meta():
    """J02E: nguồn city, index, cửa sổ, calibration artifact (kết quả calibration_artifact), ground truth, roi_frames."""
    g = globals(); rp = g.get("CITY_REPORT")
    return dict(city_source=CITY_SOURCE, city_dataset=str(CITY_DATASET) if CITY_DATASET else NA, scenario_windows=[list(w) for w in SCENARIO_WINDOWS],
                city_index=str(g.get("CITY_INDEX")), city_index_sha256=_hash(g.get("CITY_INDEX")),
                city_index_report=dict(n_frames=rp["n_frames"], index_sha256=rp["index_sha256"], errors=rp["errors"], warnings=rp["warnings"]) if rp else NA,
                city_dets_cache_reused=str(CITY_DETS_CACHE) if CITY_REUSE else NA, review_conf=REVIEW_CONF,
                calibration_artifact=g.get("CAL_RESOLVED") or NA, ground_truth_kind=g.get("GROUND_TRUTH_KIND", NA), sync_stamp=g.get("SYNC_STAMP") or NA,
                roi_frames_dir=str(OUT_DIR / "roi_frames"), roi_frames_provenance_sha256=_hash(OUT_DIR / "roi_frames" / "roi_frames_provenance.json"))


def write_run_metadata(status):
    g = globals()
    cj = json.load(open(ACTIVE["calibration"], encoding="utf-8")) if (ACTIVE["calibration"] is not None and Path(ACTIVE["calibration"]).is_file()) else None   # J02E: image_review không có calibration
    wts = g.get("WEIGHTS_USED")
    wp = next((p for p in ([Path(str(wts))] + [STAGE / str(wts)]) if wts is not None and p.is_file()), None)   # Ultralytics tải về cwd (STAGE)
    try:
        sys.dont_write_bytecode = True
        if str(STAGE) not in sys.path: sys.path.insert(0, str(STAGE))
        from flow_estimation.calibrate import GAPS as _GAPS, CONFS as _CONFS      # hằng số của source, không khai báo lại
        src_gaps, src_confs = list(_GAPS), list(_CONFS)
    except Exception: src_gaps = src_confs = NA
    nv = nc = None
    if cj: nv, nc = cj["n_videos"], cj["n_cameras"]
    elif (CALIB_DIR / "videos.json").is_file():
        _V = json.load(open(CALIB_DIR / "videos.json", encoding="utf-8")); nv, nc = len(_V), len({v["camera"] for v in _V.values()})
    scen = (g.get("SCEN") or {}) if DOWNSTREAM else {}; tscen = (g.get("TRAIN_SCEN") or {}) if (DOWNSTREAM and RUN_TRAIN) else {}   # chỉ lấy scenario đúng mode của run này
    bench = OUT_DIR / "bench"
    meta = dict(
        run_id=RUN_ID, run_mode=RUN_MODE, active=ACTIVE["label"], detector_backend=DETECTOR_BACKEND,
        detector_name=("hybrid_cleaned (provenance sidecar; ver hybrid_provenance)" if DETECTOR_BACKEND == "hybrid_cleaned" else (cj["detector"]["model"] if cj else NA)),
        legacy_detector_label_in_calibration_json=cj["detector"]["model"] if cj else NA,
        detector_name_note="nhãn do calibrate.py:112 ghi cứng vào JSON, KHÔNG suy ra từ weights; phân biệt detector bằng weights_path/weights_sha256",
        weights_path=str(wp) if wp else (str(wts) if wts is not None else NA),
        weights_source=g.get("WEIGHTS_SOURCE", NA), weights_sha256=_sha256(wp) if wp else NA,
        image_size=cj["detector"]["imgsz"] if cj else NA,
        image_size_note="detect.py --imgsz mặc định 1280 (notebook không đổi); giá trị này cũng do calibrate.py:112 ghi cứng",
        hybrid_provenance=_j01_hybrid_meta(),
        j02e=_j02e_meta(),      # J02E
        calibration_images_path=str(CALIB_IMAGES_EFF) if CALIB_IMAGES_EFF else None,
        calibration_metadata_path=str(CALIB_DIR / "videos.json"), calibration_frames_path=str(CALIB_DIR / "frames.jsonl"),
        camera_folds_path=str(CALIB_DIR / "camera_folds.json"),
        number_of_frames=_nlines(CALIB_DIR / "frames.jsonl"), number_of_videos=nv, number_of_cameras=nc,
        supported_gaps_s=cj["gaps_s"] if cj else src_gaps, supported_extra_schedules=["mix (10/15 s)"],
        unsupported_gaps_s={"25": "NOT_IMPLEMENTED_IN_SOURCE"},
        confidence_search_grid=src_confs, confidence_search_grid_source="flow_estimation.calibrate.CONFS",
        detection_cache_path=str(ACTIVE["dets_calib"]), calibration_json_path=str(ACTIVE["calibration"]),
        started_at=T_START, finished_at=None if status == "RUNNING" else _now(),
        execution_status=status, stage_status=dict(STATUS), failed_stages=dict(FAILED),
        environment=_env_meta(),
        input_provenance=dict(
            calibration_index_sha256=_hash(CALIB_DIR / "frames.jsonl"), reference_metadata_sha256=_hash(CALIB_DIR / "videos.json"),
            camera_folds_sha256=_hash(CALIB_DIR / "camera_folds.json"),
            detection_cache_calibration_sha256=_hash(ACTIVE["dets_calib"]),
            detection_cache_city_sha256=_hash(ACTIVE["dets_city"]) if ACTIVE["dets_city"] else NA,
            calibration_json_sha256=_hash(ACTIVE["calibration"]),
            demand_settings_sha256=_hash(FLOW_DIR / "demand_settings.json") if DOWNSTREAM else NA,
            source_tree_sha256=hashlib.sha256(json.dumps(SRC_SNAPSHOT, sort_keys=True).encode()).hexdigest(), source_tree_files=len(SRC_SNAPSHOT),
            source_file_sha256={k: _hash(REPO_DIR / k) for k in _SRC_KEYS},
            mplight_checkpoint=dict(path=str(MPLIGHT_CKPT), sha256=_hash(MPLIGHT_CKPT)) if MPLIGHT_CKPT else None,
            user_controller=_user_ctrl()),
        demand=dict(demand_mode=DEMAND_MODE, scenario_root=str(g.get("SCEN_ROOT", NA)), expected_binding=str(g.get("EXPECT_BINDING_PATH", NA)),
                    j02_demand_settings_directional_sha256=_hash(FLOW_DIR / "demand_settings_directional.json") if J02 else NA,
                    j02_perframe_directional_sha256=_hash(FLOW_DIR / "perframe_directional.json") if J02 else NA,
                    split=SPLIT, modes=EVAL_MODES, scale=SCALE, duration_s=RUN_COUNTS, sumo_seed=SEED,
                    demand_seed_by_mode={m: s["meta"]["seed"] for m, s in scen.items()} or NA,
                    source_windows={m: s["meta"].get("source_window") for m, s in scen.items()} or NA) if DOWNSTREAM else NA,
        scenario_fingerprints={**{f"{SPLIT}/{m}": s["fingerprint"] for m, s in scen.items()}, **{f"{k}/{TRAIN_MODE}": s["fingerprint"] for k, s in tscen.items()}} or NA,
        output_provenance=dict(
            out_dir=str(OUT_DIR), stage_dir=str(STAGE), logs_dir=str(OUT_DIR / "logs"),
            detection_cache_calibration=str(ACTIVE["dets_calib"]), detection_cache_city=str(ACTIVE["dets_city"]) if ACTIVE["dets_city"] else NA,
            calibration_json=str(ACTIVE["calibration"]),
            flow_dir=str(FLOW_DIR) if DOWNSTREAM else NA, perframe=str(FLOW_DIR / "perframe.json") if (FLOW_DIR / "perframe.json").is_file() else NA,
            demand_settings=str(FLOW_DIR / "demand_settings.json") if (FLOW_DIR / "demand_settings.json").is_file() else NA,
            scenarios={f"{SPLIT}/{m}": str(s["cfg"]) for m, s in scen.items()} or NA,
            train_scenarios={k: str(s["cfg"]) for k, s in tscen.items()} or NA,
            controllers_requested=list(CONTROLLERS), controllers_run=g.get("PLAN_RUNS", NA) if DOWNSTREAM else NA, controllers_skipped=g.get("SKIPPED", NA) if DOWNSTREAM else NA,
            benchmark_results_csv=str(bench / "results.csv") if (bench / "results.csv").is_file() else NA,
            benchmark_results_json=str(bench / "results.json") if (bench / "results.json").is_file() else NA,
            training_run_dir=str(g["TRAIN_RUN_DIR"]) if (RUN_TRAIN and g.get("TRAIN_RUN_DIR")) else NA))
    META_PATH.parent.mkdir(parents=True, exist_ok=True)
    META_PATH.write_text(json.dumps(meta, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    return meta

class stage:
    """with stage("<STATUS key>"): ...  Lỗi -> STATUS/FAILED + metadata 'FAILED at <stage>' (nếu ghi được), rồi RAISE LẠI (không nuốt, không PASS giả)."""
    def __init__(self, key): self.key = key
    def __enter__(self): return self
    def __exit__(self, et, ev, tb):
        if et is not None and not issubclass(et, (KeyboardInterrupt, SystemExit)):
            msg = f"{et.__name__}: {ev}"; FAILED[self.key] = msg; STATUS[self.key] = "FAILED: " + msg
            try: write_run_metadata(f"FAILED at {self.key}: {msg}")
            except Exception as e2: print("không ghi được metadata FAILED:", e2)
        return False
print("metadata:", META_PATH)

### 01c — J02E: nguồn đồng bộ · khám phá ảnh city · calibration artifact (fail-fast)
Chỉ gọi CLI của REPO (chạy trong `STAGE`): (1) nếu `REPO_DIR` là cây đã đồng bộ (`SYNC_MANIFEST.json`, từ `scripts/j02e_sync.py`) thì mọi file đã stage phải khớp SHA-256 manifest (lệch ⇒ **FAIL "stale source"**); (2) `CITY_SOURCE="images"`: `flow_estimation.city_index` (alias camera, timestamp, hình học 512:288, cửa sổ `SCENARIO_WINDOWS`) → `OUT_DIR/inputs/city_index.jsonl` + report; `repo_cache` → index của repo; (3) `baseline_cache`: `flow_estimation.calibration_artifact` kiểm calibration cho đúng backend **trước** khi detect (INVALID ⇒ stage `calibration` FAILED ⇒ các stage sau BLOCKED, không bao giờ dùng β backend khác).

In [ ]:
# J02E fix5
# J02E 01c — (1) kiểm tra nguồn đồng bộ, (2) khám phá ảnh city + city_index, (3) calibration artifact (fail-fast, TRƯỚC detection). Chỉ gọi CLI của REPO (STAGE).
CITY_INDEX = CITY_ROOT = CITY_REPORT = CAL_RESOLVED = CAL_PATH_IN = SYNC_STAMP = None
GROUND_TRUTH_KIND = NA

def j02e_ground_truth():
    """J02E: nguồn ground truth của calibration set ĐANG DÙNG (ghi vào provenance). Set của repo = REFERENCE; fixture tổng hợp = SYNTHETIC_GT_CODE_TEST (CODE TEST, KHÔNG có giá trị khoa học)."""
    p = CALIB_DIR / "ground_truth_provenance.json"
    if p.is_file():
        d = json.load(open(p, encoding="utf-8")); assert d.get("kind"), f"{p}: thiếu 'kind'"
        return d
    return {"kind": "REFERENCE"}

# (1) cây nguồn đồng bộ (Kaggle): file đã stage phải khớp manifest
with stage("stage"):
    _sm = REPO_DIR / "SYNC_MANIFEST.json"
    if _sm.is_file():
        _M = json.load(open(_sm, encoding="utf-8")); SYNC_STAMP = _M.get("stamp"); _mf = _M["files"]
        _rel = [r for r in _mf if r in STAGE_INCLUDE or any(r.startswith(x + "/") for x in STAGE_INCLUDE)]
        _bad = [r for r in _rel if not (STAGE / r).is_file() or _sha256(STAGE / r) != _mf[r]]
        _extra = [r for r in tree_sha(STAGE) if r not in _mf]
        if _bad: raise RuntimeError(f"J02E: stale source — {len(_bad)} file đã stage khác SYNC_MANIFEST.json (stamp {SYNC_STAMP}): {_bad[:8]}")
        print(f"J02E sync: stamp {SYNC_STAMP} | {len(_rel)} file đã stage khớp SYNC_MANIFEST.json" + (f" | cảnh báo: {len(_extra)} file stage không có trong manifest: {_extra[:5]}" if _extra else ""))
    else:
        print("J02E sync: không có SYNC_MANIFEST.json (REPO_DIR không phải cây đã đồng bộ) — bỏ qua kiểm tra stale")

# (2) khám phá ảnh city
if CITY_FROM_IMAGES:
    with stage("input_discovery"):
        _seg = json.load(open(STAGE / "sumo/config/camera_segments.json", encoding="utf-8"))
        CITY_REQUIRE = [c for c in _seg if not c.startswith("_")]            # 4 camera của camera_segments.json
        _src = require(CITY_DATASET, "CITY_DATASET (VIETTSC_CITY_DATASET)")
        if _src.is_file():                                                   # .zip -> WORK/inputs/<RUN_ID>/
            CITY_ROOT = WORK / "inputs" / RUN_ID
            assert not _files_under(CITY_ROOT), f"{CITY_ROOT} đã có file — dùng RUN_ID mới"
            CITY_ROOT.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(_src), str(CITY_ROOT))
            print("đã giải nén", _src, "->", CITY_ROOT)
        else:
            CITY_ROOT = _src
        CITY_INDEX = OUT_DIR / "inputs" / "city_index.jsonl"; _rep = OUT_DIR / "inputs" / "city_index_report.json"
        CITY_INDEX.parent.mkdir(parents=True, exist_ok=True)
        _a = ["--root", CITY_ROOT, "--exclude", *CITY_EXCLUDE, "--require", *CITY_REQUIRE, "--roi", STAGE / "flow_estimation/directional_roi.json", "--out", CITY_INDEX, "--report", _rep]
        for _m, _s, _f, _t in SCENARIO_WINDOWS: _a += ["--window", _s, _f, _t]
        pymod("flow_estimation.city_index", *_a)
        CITY_REPORT = json.load(open(_rep, encoding="utf-8"))
        assert not CITY_REPORT["errors"], f"city_index errors: {CITY_REPORT['errors']}"
        _miss = [x for x in SESSIONS if x not in CITY_REPORT["sessions"]]
        assert not _miss, f"J02E: session trong SCENARIO_WINDOWS không có trong ảnh: {_miss} (có: {list(CITY_REPORT['sessions'])})"
        print(f"\n{'session':14s}{'camera':32s}{'frames':>7s}  {'first_t':31s}{'last_t':31s}{'size':>9s} {'gaps>35s':>8s}  ts_source")
        for _sn, _sv in CITY_REPORT["sessions"].items():
            for _cn, _c in _sv["cameras"].items():
                print(f"{_sn:14s}{_cn:32s}{_c['n_frames']:>7d}  {_c['first_t']:31s}{_c['last_t']:31s}{str(_c['width']) + 'x' + str(_c['height']):>9s} {_c['n_gaps_gt_35s']:>8d}  {_c['timestamp_source']}")
        STATUS["input_discovery"] = f"done ({CITY_REPORT['n_frames']} frame, {len(CITY_REPORT['sessions'])} session, index sha256 {CITY_REPORT['index_sha256'][:12]})"
    print(STATUS["input_discovery"])
else:
    CITY_INDEX, CITY_ROOT = BASE["city_index"], CITY_IMAGES              # repo_cache: index + ảnh (nếu có) như cũ

# (3) calibration artifact (baseline_cache): kiểm TRƯỚC detection; INVALID => stage 'calibration' FAILED, không fallback
if RUN_MODE == "baseline_cache":
    with stage("calibration"):
        CAL_PATH_IN = CALIBRATION_ARTIFACT if CALIBRATION_ARTIFACT else (HYBRID_CACHE_DIR if HYBRID else BASE["calibration"])
        require(CAL_PATH_IN, "calibration artifact")
        _res = OUT_DIR / "calibration" / "calibration_artifact.json"
        try:
            pymod("flow_estimation.calibration_artifact", "--path", CAL_PATH_IN, "--backend", DETECTOR_BACKEND, "--out", _res)
        except RuntimeError:
            _r = json.load(open(_res, encoding="utf-8")) if _res.is_file() else {}      # J02E fix5
            GROUND_TRUTH_KIND = _r.get("ground_truth_kind") or NA                      # J02E fix5: giữ ground truth của artifact để báo cáo cuối (kể cả khi INVALID)
            _pb = _r.get("problems") or ["(không đọc được kết quả)"]
            if _r.get("problems") and all("non-finite/invalid beta" in _p for _p in _pb):      # J02E fix5: chỉ thiếu dữ liệu (beta NaN) => BLOCKED, KHÔNG phải FAILED chung chung (cùng ngữ nghĩa j02e_blocked_msg, fix2)
                raise RuntimeError("BLOCKED: INSUFFICIENT_DATA — " + "; ".join(_pb) + f" | calibration artifact backend '{DETECTOR_BACKEND}' ({CAL_PATH_IN}); KHÔNG fallback sang calibration backend khác")
            raise RuntimeError(f"J02E: calibration artifact INVALID cho backend '{DETECTOR_BACKEND}' ({CAL_PATH_IN}): " + "; ".join(_pb) + " — KHÔNG fallback sang calibration backend khác")
        CAL_RESOLVED = json.load(open(_res, encoding="utf-8"))
        assert CAL_RESOLVED["status"] == "OK" and CAL_RESOLVED["backend"] == DETECTOR_BACKEND, CAL_RESOLVED
        ACTIVE["calibration"] = Path(os.path.join(str(STAGE), CAL_RESOLVED["calibration"])).resolve()
        if HYBRID: ACTIVE["calibration_provenance"] = Path(os.path.join(str(STAGE), CAL_RESOLVED["provenance"])).resolve() if CAL_RESOLVED.get("provenance") else None
        GROUND_TRUTH_KIND = CAL_RESOLVED["ground_truth_kind"]
        print(f"calibration artifact: status {CAL_RESOLVED['status']} | backend {CAL_RESOLVED['backend']} | layout {CAL_RESOLVED['layout']} | ground_truth_kind {GROUND_TRUTH_KIND} | scientific_validity: {CAL_RESOLVED['scientific_validity']}")
        print("  calibration:", ACTIVE["calibration"], "| sha256", CAL_RESOLVED["calibration_sha256"][:12], "| conf", CAL_RESOLVED["conf"])
        STATUS["calibration"] = ("done (baseline calibration của repo, không chạy lại)" if not (HYBRID or CALIBRATION_ARTIFACT) else
                                 "done (hybrid_cleaned calibration đã xác minh từ VIETTSC_HYBRID_CACHE_DIR, không chạy lại)" if (HYBRID and not CALIBRATION_ARTIFACT) else
                                 f"done (calibration artifact {CAL_RESOLVED['layout']} đã kiểm bằng calibration_artifact, không chạy lại)") + f" | ground truth {GROUND_TRUTH_KIND}"
elif RECOMPUTE:
    GROUND_TRUTH_KIND = j02e_ground_truth()["kind"]
    print("ground truth của calibration set:", GROUND_TRUTH_KIND, "| scientific:", f"KHÔNG ({GROUND_TRUTH_KIND}, synthetic — chỉ CODE/INTEGRATION TEST)" if str(GROUND_TRUTH_KIND).startswith("SYNTHETIC") else "theo reference của set")      # J02E fix6


### J01 — hybrid_cleaned backend: checkpoint staging + provenance
Chỉ chạy khi `DETECTOR_BACKEND='hybrid_cleaned'`. Recompute: copy 2 checkpoint thật vào `STAGE/models/` và kiểm size/SHA-256 (không tải). `baseline_cache`: xác minh artifact hybrid (backend, SHA-256 cache/weights/config/calibration).


In [ ]:
# J01 — hybrid_cleaned backend: stage + verify the two REAL checkpoints (recompute modes) or validate the supplied hybrid artifact folder (baseline_cache).
# J01: /kaggle/input là chỉ đọc; weights được COPY vào STAGE/models (ghi được). /kaggle/working không bền qua phiên -> giữ checkpoint trong Dataset.
HYBRID_WEIGHTS = None; HYBRID_PROV = {}
def _j01_hd():
    if str(STAGE) not in sys.path: sys.path.insert(0, str(STAGE))
    sys.dont_write_bytecode = True
    from flow_estimation import hybrid_detector as hd          # chỉ import numpy; KHÔNG import ultralytics
    return hd

def _j01_prov(p): return json.load(open(require(p, "provenance sidecar"), encoding="utf-8"))

def _j01_validate_hybrid_artifacts():
    """baseline_cache + hybrid_cleaned: provenance chính xác của backend. KHÔNG bao giờ chấp nhận cache/β của YOLO11x."""
    hd = _j01_hd(); cfg_sha = hd.hc.config_sha256(); problems = []; prov = {}
    for k, p in (("calib", ACTIVE["dets_calib"]),) + ((("city", ACTIVE["dets_city"]),) if not IMAGES else ()):   # J02E: city mới (images) được kiểm ở cell detection bằng calibration_artifact --dets-provenance
        pr = _j01_prov(str(require(p, f"hybrid cache {k}")) + ".provenance.json"); prov[k] = pr
        if pr.get("backend") != "hybrid_cleaned": problems.append(f"{k}: backend={pr.get('backend')!r} (cần 'hybrid_cleaned')")
        if pr["cache"]["sha256"] != _sha256(p): problems.append(f"{k}: SHA-256 cache ≠ provenance")
        if pr["weights"]["yolo11x"]["sha256"] != hd.YOLO11X["sha256"] or pr["weights"]["bmd45"]["sha256"] != hd.BMD45["sha256"]: problems.append(f"{k}: SHA-256 weights ≠ pin")
        if pr["cleanup_config_sha256"] != cfg_sha: problems.append(f"{k}: cleanup config SHA ≠ bản trong repo")
        P = pr["parameters"]
        if (P["conf_floor"], P["nms_iou"], P["imgsz"], P["max_det"]) != (hd.DETECTION_CONF_FLOOR, hd.NMS_IOU, hd.IMAGE_SIZE, hd.MAX_DET): problems.append(f"{k}: tham số detect khác pin: {P}")
    cp = _j01_prov(ACTIVE["calibration_provenance"]); prov["calibration"] = cp
    if cp.get("backend") != "hybrid_cleaned": problems.append("calibration: backend ≠ hybrid_cleaned")
    if cp.get("calibration_json_sha256") != _sha256(ACTIVE["calibration"]): problems.append("calibration: SHA-256 JSON ≠ provenance")
    if cp.get("detection_cache_calibration_sha256") != prov["calib"]["cache"]["sha256"]: problems.append("calibration: fit trên cache khác với cache calib đang dùng")
    if cp.get("cleanup_config_sha256") != cfg_sha: problems.append("calibration: cleanup config SHA ≠ repo")
    if problems: raise RuntimeError("J01: artifact hybrid không hợp lệ — " + "; ".join(problems))
    return prov

def j01_write_calibration_provenance():
    """Sidecar xác định detector của calibration hybrid_cleaned. calibrate.py ghi cứng nhãn 'yolo11x (COCO, Ultralytics)' vào JSON: nhãn đó KHÔNG phải provenance."""
    hd = _j01_hd(); dpp = str(ACTIVE["dets_calib"]) + ".provenance.json"; dp = _j01_prov(dpp)
    cal = json.load(open(require(ACTIVE["calibration"], "calibration.json"), encoding="utf-8"))
    side = dict(backend="hybrid_cleaned", created=_now(), calibration_json=str(ACTIVE["calibration"]), calibration_json_sha256=_sha256(ACTIVE["calibration"]),
                detection_cache_calibration=str(ACTIVE["dets_calib"]), detection_cache_calibration_sha256=_sha256(ACTIVE["dets_calib"]), detection_cache_calibration_provenance_sha256=_sha256(dpp),
                weights=dp["weights"], class_names=dp["class_names"], detection_parameters=dp["parameters"], versions=dp["versions"], cleanup_config_sha256=dp["cleanup_config_sha256"],
                cleanup_config_status=dp["cleanup_config_status"], cleanup_reference=dp["cleanup_reference"],
                calibration_set=dict(videos_sha256=_hash(CALIB_DIR / "videos.json"), frames_sha256=_hash(CALIB_DIR / "frames.jsonl"), camera_folds_sha256=_hash(CALIB_DIR / "camera_folds.json")),
                conf_selected=cal.get("conf"), legacy_detector_label_in_json=cal["detector"]["model"], legacy_label_note=hd.LEGACY_LABEL_WARNING,
                beta_provenance="beta/conf fitted by flow_estimation.calibrate on THIS hybrid cache; never reuse YOLO11x-only beta",
                ground_truth=j02e_ground_truth())   # J02E: REFERENCE (set của repo) | SYNTHETIC_GT_CODE_TEST (fixture)
    Path(ACTIVE["calibration_provenance"]).write_text(json.dumps(side, indent=1, ensure_ascii=False), encoding="utf-8")
    print("J01 calibration provenance ->", ACTIVE["calibration_provenance"]); return side

if HYBRID:
    with stage("hybrid_backend"):
        if RECOMPUTE or CITY_DETECT:      # J02E: detection city từ ảnh cũng cần 2 checkpoint thật
            (STAGE / "models").mkdir(parents=True, exist_ok=True); HYBRID_WEIGHTS = {}
            for _n, _s in (("yolo11x.pt", YOLO_WEIGHTS), ("yolov12x_bmd45.pt", BMD45_WEIGHTS or (REPO_DIR / "models/yolov12x_bmd45.pt"))):
                _d = STAGE / "models" / _n
                if _d.exists(): raise FileExistsError(f"STAGE/models/{_n} đã tồn tại — STAGE phải mới cho mỗi RUN_ID")
                shutil.copy2(require(_s, f"weights {_n}"), _d); HYBRID_WEIGHTS[_n] = _d
            _o = sh([sys.executable, "-m", "flow_estimation.hybrid_detector", "--verify-only", "--weights-yolo11x", HYBRID_WEIGHTS["yolo11x.pt"], "--weights-bmd45", HYBRID_WEIGHTS["yolov12x_bmd45.pt"]])
            HYBRID_PROV = json.loads(_o[_o.index("{"):])
            STATUS["hybrid_backend"] = "done (2 checkpoint thật đã stage; size + SHA-256 khớp pin; không tải tự động)"
        elif HYBRID_CACHE_DIR is not None and CALIBRATION_ARTIFACT is None:      # J02E: đường cũ; có CALIBRATION_ARTIFACT => calibration_artifact (cell 01c) kiểm
            HYBRID_PROV = _j01_validate_hybrid_artifacts()
            STATUS["hybrid_backend"] = "done (artifact hybrid đã xác minh: backend, SHA-256 cache/weights/config/calibration)"
        else:
            STATUS["hybrid_backend"] = "done (J02E: calibration hybrid đã kiểm bằng calibration_artifact ở cell 01c; cache city tái dùng, không cần checkpoint)"
    print(STATUS["hybrid_backend"])
else:
    print(STATUS["hybrid_backend"])


## 01a — Calibration set (bước 1–2: reference flow + snapshots)
Source: `python -m flow_estimation.build_calibration_set --tracks --mapkf --keyframes --out [--workers]` (ground truth = số xe *moving* từ tracking, snapshot 5 s, 512×288 JPEG q88).
**BLOCKED** nếu thiếu input: tracks/mapkf/keyframes **không có trong repo** và bước tracking (video → tracks) **không có trong source**. `camera_folds.json` không do script này tạo.
Mặc định (`REBUILD_CALIB_SET=False`): dùng calibration set có sẵn trong repo (artifact, KHÔNG dựng lại).

In [ ]:
if IMAGE_REVIEW:   # J02E: không cần calibration
    STATUS["calib_set"] = "SKIPPED (image_review: by design)"
elif RUN_MODE == "baseline_cache" and (CALIBRATION_ARTIFACT or HYBRID):   # J02E fix7: calibration lấy từ artifact/cache => calibration set KHÔNG được đọc, không in ground truth của nó
    _csrc = CALIBRATION_ARTIFACT if CALIBRATION_ARTIFACT else (HYBRID_CACHE_DIR / "hybrid_cleaned_calibration.json" if HYBRID_CACHE_DIR else "cache hybrid")
    STATUS["calib_set"] = f"SKIPPED (baseline_cache + {'CALIBRATION_ARTIFACT' if CALIBRATION_ARTIFACT else 'HYBRID_CACHE_DIR'}: calibration set không dùng/không được đọc; calibration từ artifact {_csrc})"
else:
    with stage("calib_set"):
        if REBUILD_CALIB_SET:
            assert RECOMPUTE, "REBUILD_CALIB_SET chỉ dùng với RUN_MODE='full_recompute' hoặc 'calibration_only'"
            pymod("flow_estimation.build_calibration_set", "--tracks", require(TRACKS_DIR, "VIETTSC_TRACKS"), "--mapkf", require(MAPKF_DIR, "VIETTSC_MAPKF"),
                  "--keyframes", require(KEYFRAMES_DIR, "VIETTSC_KEYFRAMES"), "--out", CALIB_DIR, "--workers", WORKERS)
            shutil.copy2(require(CAMERA_FOLDS, "VIETTSC_CAMERA_FOLDS"), CALIB_DIR / "camera_folds.json")   # input của calibrate.py:98
            for n in ("videos.json", "frames.jsonl", "camera_folds.json"): require(CALIB_DIR / n, n)
            STATUS["calib_set"] = "done (rebuilt từ tracks)"
        else:
            for n in ("videos.json", "frames.jsonl", "camera_folds.json"): require(CALIB_DIR / n, n)
            _src = f"CALIB_SET_DIR (set ngoài) = {CALIB_SET_DIR}" if CALIB_SET_DIR else f"data/calibration của repo = {CALIB_DIR}"      # J02E fix2: nguồn thật
            if RUN_MODE == "baseline_cache":      # J02E fix7: calibration là của repo (yolo11x); calibration set không chạy lại => không in ground truth kind
                STATUS["calib_set"] = f"loaded (baseline_cache: calibration của repo; calibration set không chạy lại, chỉ kiểm tra file tồn tại tại {_src})"
            else:
                try: _gtk = j02e_ground_truth()["kind"]
                except Exception: _gtk = "unknown"
                STATUS["calib_set"] = f"loaded (artifact có sẵn từ {_src}, không dựng lại) | ground truth {_gtk}"
print("calibration set:", CALIB_DIR); print(STATUS["calib_set"])

## 01 — Detection
`baseline_cache`: dùng cache YOLO11x trong repo (đã nêu rõ ở `ACTIVE`). `full_recompute`: chạy `flow_estimation.detect` ghi vào `ACTIVE[...]` (cần ảnh gốc + GPU; thiếu thì **dừng**, không fallback sang cache cũ). Backend J01 `hybrid_cleaned`: xem section J01 (cell hybrid backend); không fallback sang YOLO11x.

**J02E:** `CITY_SOURCE="images"` → detection city chạy trên ảnh của `CITY_INDEX` (cell 01c; `--device 0` nếu có GPU) hoặc tái dùng `VIETTSC_CITY_DETS_CACHE` (mọi ảnh của index phải có trong cache; backend khớp `DETECTOR_BACKEND` qua `<cache>.provenance.json`; hybrid bắt buộc có provenance). `image_review` chỉ chạy bước này cho city (không calibration).


In [ ]:
# J02E: helper detection city (CITY_INDEX/CITY_ROOT) — images (mọi RUN_MODE trừ calibration_only) và full_recompute
def _j02e_gpu_args(): return ["--device", "0"] if globals().get("gpu") else []      # J02E: GPU => --device 0

def _j02e_pick_yolo_weights():
    """Cùng quy tắc weights như nhánh recompute (J00a F3): weights thật; không để Ultralytics tự tải khi ALLOW_WEIGHTS_DOWNLOAD=False."""
    if YOLO_WEIGHTS:
        _j00a_require_real_weights(YOLO_WEIGHTS, "VIETTSC_YOLO11X"); return YOLO_WEIGHTS, "VIETTSC_YOLO11X (file)"
    if BASE["weights"].is_file() and not w_ptr: return BASE["weights"], "repo models/yolo11x.pt (file thật)"
    if not ALLOW_WEIGHTS_DOWNLOAD: raise RuntimeError("Không có weights YOLO11x thật (VIETTSC_YOLO11X thiếu; models/yolo11x.pt thiếu hoặc là LFS pointer) và ALLOW_WEIGHTS_DOWNLOAD=False — không tải tự động")
    return "yolo11x.pt", "Ultralytics download (cần Internet)"

def _j02e_verify_reused_cache():
    """CITY_DETS_CACHE (tái dùng): MỌI ảnh của CITY_INDEX phải có trong cache; backend khớp DETECTOR_BACKEND (<cache>.provenance.json)."""
    import gzip
    c = require(CITY_DETS_CACHE, "VIETTSC_CITY_DETS_CACHE")
    with (gzip.open(c, "rt", encoding="utf-8") if c.suffix == ".gz" else open(c, encoding="utf-8")) as fh:
        have = {json.loads(l)["image"] for l in fh if l.strip()}
    need = [json.loads(l)["image"] for l in open(CITY_INDEX, encoding="utf-8") if l.strip()]
    miss = [k for k in need if k not in have]
    if miss: raise RuntimeError(f"J02E: CITY_DETS_CACHE thiếu {len(miss)}/{len(need)} ảnh của CITY_INDEX (khoá = đường dẫn tương đối so với CITY_ROOT), ví dụ: {miss[:5]}")
    pp = Path(str(c) + ".provenance.json")
    if pp.is_file():
        be = json.load(open(pp, encoding="utf-8")).get("backend")
        if be != DETECTOR_BACKEND: raise RuntimeError(f"J02E: backend của cache tái dùng ({be!r}) khác DETECTOR_BACKEND={DETECTOR_BACKEND!r} — không dùng")
        note = f"cache provenance: backend={be}"
    elif HYBRID: raise RuntimeError(f"J02E: hybrid_cleaned cần {pp} (không có provenance => không chấp nhận cache tái dùng)")
    else: note = "cache provenance: none (yolo11x legacy)"
    return f"reused cache {c.name} sha256 {_sha256(c)[:12]}, {len(need)} ảnh khớp CITY_INDEX; {note}"

def _j02e_city_detect(log_dir):
    """Detection city trên CITY_INDEX/CITY_ROOT (images hoặc full_recompute); CITY_REUSE => chỉ xác minh cache có sẵn. Trả về mô tả cho STATUS."""
    if CITY_REUSE: return _j02e_verify_reused_cache()
    _root = require(CITY_ROOT, "CITY_IMAGES / CITY_DATASET")
    if HYBRID:
        _hy = ["--weights-yolo11x", HYBRID_WEIGHTS["yolo11x.pt"], "--weights-bmd45", HYBRID_WEIGHTS["yolov12x_bmd45.pt"]]
        sh([sys.executable, "-m", "flow_estimation.hybrid_detector", "--list", CITY_INDEX, "--root", _root, "--out", ACTIVE["dets_city"], *_hy, *_j02e_gpu_args(),
            "--audit-dir", log_dir / "hybrid_audit_city"], log=log_dir / "detection_city_run.log")
        require(ACTIVE["dets_city"], "detections city (hybrid) vừa tạo"); require(str(ACTIVE["dets_city"]) + ".provenance.json", "provenance hybrid (city)")
    else:
        sh([sys.executable, "-m", "flow_estimation.detect", "--list", CITY_INDEX, "--root", _root, "--out", ACTIVE["dets_city"], "--weights", WEIGHTS_USED, *_j02e_gpu_args()],
           log=log_dir / "detection_city_run.log")
        require(ACTIVE["dets_city"], "detections city vừa tạo")
    return "city detection chạy trên ảnh của CITY_INDEX"

def _j02e_check_cal_vs_dets():
    """baseline_cache: calibration artifact vs provenance của dets city THỰC SỰ dùng (backend; hybrid: SHA weights). Sai => stage 'calibration' FAILED."""
    _sc = Path(str(ACTIVE["dets_city"]) + ".provenance.json")
    if RUN_MODE == "baseline_cache" and CAL_RESOLVED and _sc.is_file():
        with stage("calibration"):
            pymod("flow_estimation.calibration_artifact", "--path", CAL_PATH_IN, "--backend", DETECTOR_BACKEND, "--dets-provenance", _sc, "--out", OUT_DIR / "calibration" / "calibration_artifact_vs_dets.json")
            print("J02E: calibration khớp provenance của dets city:", _sc.name)

if RUN_MODE == "baseline_cache" and not IMAGES:      # J02E: hành vi cũ (cache city của repo / hybrid)
    with stage("detection"):
        require(ACTIVE["dets_city"], "detections city (baseline cache)")
        if ACTIVE["dets_calib"] is not None: require(ACTIVE["dets_calib"], "detections calibration (baseline cache)")
        STATUS["detection"] = "done (baseline cache, không chạy YOLO)" if not HYBRID else "done (hybrid_cleaned cache đã xác minh, không chạy model)"   # J01
elif IMAGE_REVIEW or RUN_MODE == "baseline_cache":     # J02E: ảnh city (CITY_SOURCE=images), KHÔNG detection calibration
    with stage("detection"):
        DET_LOG_DIR = OUT_DIR / "logs"
        if not CITY_REUSE:
            if HYBRID: WEIGHTS_USED, WEIGHTS_SOURCE = HYBRID_WEIGHTS["yolo11x.pt"], "hybrid_cleaned: yolo11x.pt thật + yolov12x_bmd45.pt thật (staged; SHA-256 ghim)"
            else: WEIGHTS_USED, WEIGHTS_SOURCE = _j02e_pick_yolo_weights()
            write_run_metadata("RUNNING")
        STATUS["detection"] = "done (" + _j02e_city_detect(DET_LOG_DIR) + ")"
elif HYBRID:       # J01: hybrid_cleaned = YOLO11x (car/truck/bus) + BMD45 (Two-wheeler -> motorcycle) + cleanup S0-G -> cache VietTSC cùng schema. Weights thật đã được xác minh ở cell hybrid_backend.
    with stage("detection"):
        WEIGHTS_USED, WEIGHTS_SOURCE = HYBRID_WEIGHTS["yolo11x.pt"], "hybrid_cleaned: yolo11x.pt thật + yolov12x_bmd45.pt thật (staged; SHA-256 ghim)"
        DET_LOG_DIR = CAL_RUN_DIR if RUN_MODE == "calibration_only" else OUT_DIR / "logs"
        write_run_metadata("RUNNING")
        _hy = ["--weights-yolo11x", HYBRID_WEIGHTS["yolo11x.pt"], "--weights-bmd45", HYBRID_WEIGHTS["yolov12x_bmd45.pt"]]
        sh([sys.executable, "-m", "flow_estimation.hybrid_detector", "--list", CALIB_DIR / "frames.jsonl", "--root", require(CALIB_IMAGES_EFF, "CALIB_IMAGES"),
            "--out", ACTIVE["dets_calib"], *_hy, "--audit-dir", DET_LOG_DIR / "hybrid_audit_calib"], log=DET_LOG_DIR / "detection_calib_run.log")
        require(ACTIVE["dets_calib"], "detections calibration (hybrid) vừa tạo"); require(str(ACTIVE["dets_calib"]) + ".provenance.json", "provenance hybrid (calibration)")
        _cd = _j02e_city_detect(DET_LOG_DIR) if DOWNSTREAM else None       # J02E: calibration_only không cần detection của camera thành phố
        STATUS["detection"] = "done (recomputed)" + (f"; city: {_cd}" if CITY_REUSE else "")
else:
    with stage("detection"):
        if YOLO_WEIGHTS:
            _j00a_require_real_weights(YOLO_WEIGHTS, "VIETTSC_YOLO11X")      # J00a: F3 — file thật, không phải LFS pointer
            WEIGHTS_USED, WEIGHTS_SOURCE = YOLO_WEIGHTS, "VIETTSC_YOLO11X (file)"
        elif BASE["weights"].is_file() and not w_ptr: WEIGHTS_USED, WEIGHTS_SOURCE = BASE["weights"], "repo models/yolo11x.pt (file thật)"
        else:
            # J00a: F3 — không để Ultralytics tự tải khi ALLOW_WEIGHTS_DOWNLOAD=False (defense in depth; preflight có thể bị bỏ qua)
            if not ALLOW_WEIGHTS_DOWNLOAD: raise RuntimeError("Không có weights YOLO11x thật (VIETTSC_YOLO11X thiếu; models/yolo11x.pt thiếu hoặc là LFS pointer) và ALLOW_WEIGHTS_DOWNLOAD=False — không tải tự động")
            WEIGHTS_USED, WEIGHTS_SOURCE = "yolo11x.pt", "Ultralytics download (cần Internet)"
        DET_LOG_DIR = CAL_RUN_DIR if RUN_MODE == "calibration_only" else OUT_DIR / "logs"
        write_run_metadata("RUNNING")
        sh([sys.executable, "-m", "flow_estimation.detect", "--list", CALIB_DIR / "frames.jsonl", "--root", require(CALIB_IMAGES_EFF, "CALIB_IMAGES"),
            "--out", ACTIVE["dets_calib"], "--weights", WEIGHTS_USED], log=DET_LOG_DIR / "detection_calib_run.log")
        require(ACTIVE["dets_calib"], "detections calibration vừa tạo")
        _cd = _j02e_city_detect(DET_LOG_DIR) if DOWNSTREAM else None       # J02E: calibration_only không cần detection của camera thành phố
        STATUS["detection"] = "done (recomputed)" + (f"; city: {_cd}" if CITY_REUSE else "")
_j02e_check_cal_vs_dets()      # J02E
print("detections city  :", ACTIVE["dets_city"]); print("detections calib :", ACTIVE["dets_calib"]); print(STATUS["detection"])

## 02 — Calibration (β + confidence)
`baseline_cache`: dùng `flow_estimation/calibration/yolo11x_1280.json` (baseline, không ghi đè). `full_recompute`: `flow_estimation.calibrate` trên **đúng** detections của bước 01 → `ACTIVE["calibration"]`. Lưu ý `calibrate.py:38,106-109` dùng biến global với `multiprocessing.Pool` nên chỉ đúng với `fork` (Linux/Kaggle) — trên Windows notebook dừng với lỗi rõ ràng, không sửa source.

**J02E:** `baseline_cache` resolve calibration ở cell 01c bằng `flow_estimation.calibration_artifact` (fail-fast, trước detection): yolo11x mặc định = JSON repo, hoặc `VIETTSC_CALIBRATION_ARTIFACT`; hybrid cần artifact có provenance khớp backend (INVALID ⇒ stage FAILED, **không** dùng β yolo11x). `image_review`: bỏ qua.


In [ ]:
def j02e_validate_calibration(path, outdir):      # J02E fix2: CLI production calibration_artifact (exit 2 khi INVALID => bắt để vẫn ghi report); không fallback
    out = Path(outdir) / "calibration_artifact_check.json"
    if out.is_file(): out.unlink()
    try: pymod("flow_estimation.calibration_artifact", "--path", path, "--backend", DETECTOR_BACKEND, "--out", out)
    except RuntimeError as e: print("calibration_artifact exit != 0:", e)
    try: r = json.load(open(out, encoding="utf-8"))
    except Exception as e: r = {"status": "INVALID", "problems": [f"không đọc được kết quả kiểm artifact: {e}"]}
    r["_check_file"] = str(out)
    return r
def j02e_blocked_msg(r): return "BLOCKED: INSUFFICIENT_DATA — " + "; ".join(r.get("problems") or ["artifact INVALID"])
if IMAGE_REVIEW:      # J02E
    print(STATUS["calibration"])
elif RUN_MODE == "baseline_cache":
    with stage("calibration"):      # J02E: calibration đã resolve + kiểm bằng calibration_artifact ở cell 01c (fail-fast, trước detection); STATUS["calibration"] đặt ở đó
        require(ACTIVE["calibration"], "calibration baseline")
        assert CAL_RESOLVED and CAL_RESOLVED["status"] == "OK", "J02E: calibration artifact chưa được xác minh (cell 01c)"
else:
    CAL_LOG = (CAL_RUN_DIR / "calibration_run.log") if RUN_MODE == "calibration_only" else (OUT_DIR / "logs" / "calibration_run.log")
    with stage("calibration"):
        _mp = _j00a_probe_mp_fork(STAGE)       # J00a: F1 — Python subprocess thật (cùng interpreter/env/cwd như lệnh calibrate bên dưới), không kiểm tra kernel
        if not _mp.get("ok"): raise RuntimeError("calibrate.py (module-global _G + multiprocessing.Pool) cần 'fork' trong Python subprocess; probe: " + json.dumps(_mp, ensure_ascii=False) + " | Linux/Kaggle OK; Windows dùng RUN_MODE='baseline_cache'")
        require(ACTIVE["dets_calib"], "detections calibration (do bước 01 vừa tạo — không dùng cache cũ thay thế)")
        sh([sys.executable, "-m", "flow_estimation.calibrate", "--calib", CALIB_DIR, "--dets", ACTIVE["dets_calib"],
            "--out", ACTIVE["calibration"], "--workers", WORKERS], log=CAL_LOG)           # stdout+stderr+exit code -> log
        require(ACTIVE["calibration"], "calibration vừa tạo")
        if HYBRID: j01_write_calibration_provenance()      # J01: sidecar xác định detector (nhãn trong JSON là nhãn cũ hard-code)
        if RUN_MODE == "full_recompute":      # J02E fix2: INVALID => stage calibration FAILED (BLOCKED), downstream không chạy
            CAL_CHECK = j02e_validate_calibration(OUT_DIR, OUT_DIR / "calibration")
            print("calibration_artifact check:", CAL_CHECK["status"], "|", CAL_CHECK["_check_file"])
            if CAL_CHECK["status"] != "OK": raise RuntimeError(j02e_blocked_msg(CAL_CHECK))
        STATUS["calibration"] = "done (recomputed)"
if not IMAGE_REVIEW:   # J02E
    cal = json.load(open(ACTIVE["calibration"], encoding="utf-8"))
    print("calibration:", ACTIVE["calibration"]); print("  detector (nhãn cũ trong JSON" + (", KHÔNG phải provenance của hybrid_cleaned" if HYBRID else "") + "):", cal["detector"]["model"], "| conf:", cal["conf"], "| WAPE CV mix:", cal["cv"]["mix"]["wape"]); print(STATUS["calibration"])

### J01 — Hybrid diagnostics (opt-in)
`EXPORT_HYBRID_DIAGNOSTICS=True`: ảnh bounding box từ cache hybrid đang dùng (+ before/after cleanup, box bị xóa/cờ). Giới hạn `DIAGNOSTIC_MAX_IMAGES`. Không phải ground truth.


In [ ]:
# J01 — diagnostic images (opt-in, bounded). Drawn from the ACTIVE hybrid cache (+ audit CSVs written by the same hybrid_detector run). NOT ground truth, NOT an accuracy measure.
if HYBRID and EXPORT_HYBRID_DIAGNOSTICS:
    with stage("hybrid_diagnostics"):
        DIAG_DIR = OUT_DIR / "diagnostics" / RUN_ID / DETECTOR_BACKEND
        _audit = (HYBRID_CACHE_DIR if (RUN_MODE == "baseline_cache" and not IMAGES) else (CAL_RUN_DIR if RUN_MODE == "calibration_only" else OUT_DIR / "logs"))
        _jobs = [("calibration", ACTIVE["dets_calib"], CALIB_IMAGES_EFF, _audit / "hybrid_audit_calib")] if (ACTIVE["dets_calib"] is not None and CALIB_IMAGES_EFF) else []   # J02E
        if HAS_CITY_DETS and ACTIVE["dets_city"] and CITY_ROOT and Path(CITY_ROOT).exists(): _jobs.append(("city", ACTIVE["dets_city"], CITY_ROOT, _audit / "hybrid_audit_city"))
        for _n, _c, _r, _a in _jobs:
            sh([sys.executable, "-m", "flow_estimation.hybrid_diagnostics", "--cache", require(_c, f"cache {_n}"), "--root", require(_r, f"ảnh {_n} (cần ảnh thật; không vẽ từ cache không có ảnh)"),
                "--out", DIAG_DIR / _n, "--max-images", DIAGNOSTIC_MAX_IMAGES, "--confs", *DIAGNOSTIC_CONF_THRESHOLDS] + (["--audit-dir", _a] if Path(_a).is_dir() else []))
        STATUS["hybrid_diagnostics"] = "done"
        print("diagnostics ->", DIAG_DIR)
else:
    print("J01 diagnostics:", STATUS["hybrid_diagnostics"])


### 02b — Bảng calibration (chỉ đọc JSON, không tính lại)
Hiển thị **đúng các trường** `calibrate.py` đã ghi vào JSON (`conf`, `beta`, `cv`, `folds`, `n_videos`). Với `baseline_cache` đây là artifact **có sẵn** (không phải kết quả chạy lại — xem dòng `provenance`).
**Source không có / không lưu** (không tự thêm): gap 25 s (`calibrate.GAPS` chỉ 5/10/15/20/30); WAPE theo từng confidence ứng viên; ngưỡng và β per-fold cho gap ≠ 10 s; sai số per-video/per-camera; precision/recall/mAP/MAE/RMSE; manual ground truth (Y = tracking-derived, `build_calibration_set.py:15-16`).

In [ ]:
if IMAGE_REVIEW:   # J02E
    print("SKIPPED (image_review): bảng calibration")
else:
    # 02b. Bảng calibration: chỉ trình bày các trường JSON do calibrate.py ghi.
    cal = json.load(open(require(ACTIVE["calibration"], "calibration ACTIVE"), encoding="utf-8"))
    f = lambda v, nd=2: "N/A" if v is None else f"{v:.{nd}f}"
    CL, cv, beta = cal["classes"], cal["cv"], cal["beta"]
    print("nguồn     :", ACTIVE["calibration"]); print("provenance:", STATUS["calibration"])
    print("detector  :", cal["detector"], ("<-- NHÃN CŨ hard-code của calibrate.py; provenance thật: " + str(ACTIVE["calibration_provenance"])) if HYBRID else "", "| n_videos", cal["n_videos"], "| n_cameras", cal["n_cameras"], "| gaps_s", cal["gaps_s"])
    print("conf cuối :", cal["conf"], "(fit trên TẤT CẢ video, ngưỡng chọn ở gap 10 s và dùng cho mọi gap: calibrate.py:145-147)")
    SHOW = [5, 10, 15, 20, 25, 30]
    print("\n-- β final-fit = median(Y/N) theo gap x class")
    print(f"{'gap':>5s} " + " ".join(f"{c:>10s}" for c in CL))
    for g in SHOW:
        b = beta.get(str(g))
        print(f"{g:>5d} " + (" ".join(f"{f(x, 4):>10s}" for x in b) if b else "NOT IMPLEMENTED IN SOURCE (gap không nằm trong calibrate.GAPS)"))
    print("\n-- cross-validation theo camera (held-out): WAPE %  |  bias %      [moto car truck bus total]")
    for g in [str(x) for x in SHOW] + ["mix"]:
        c = cv.get(g)
        if c is None: print(f"{g:>5s}  N/A (không có trong source)"); continue
        print(f"{g:>5s}  " + " ".join(f"{f(x):>7s}" for x in c["wape"]) + "   |   " + " ".join(f"{f(x):>7s}" for x in c["bias"]))
    print("\n-- per-fold (source chỉ lưu gap 10 s)")
    for k, fo in cal["folds"].items():
        print(f"fold {k}: train {fo['train_videos']} / test {fo['test_videos']} video ({fo['test_cameras']} cam) | conf moto {fo['conf_motorcycle']} other {fo['conf_other']} | "
              f"β10 {fo['beta_10s']} | test WAPE {fo['test_wape']}")
    if Path(ACTIVE["calibration"]).resolve() != BASE["calibration"].resolve():     # full_recompute: đối chiếu với baseline của repo
        base = json.load(open(BASE["calibration"], encoding="utf-8"))
        print("\n-- Δ WAPE total (mới − baseline repo), CV")
        for g in [str(x) for x in SHOW] + ["mix"]:
            if g in cv and g in base["cv"]: print(f"{g:>5s}  mới {f(cv[g]['wape'][-1])} | baseline {f(base['cv'][g]['wape'][-1])} | Δ {f(cv[g]['wape'][-1] - base['cv'][g]['wape'][-1])}")
    print("\nKHÔNG LƯU Ở ARTIFACT: WAPE theo từng conf ứng viên; ngưỡng/β per-fold của gap != 10 s; sai số per-video/camera. CV gap != 10 s/mix dùng ngưỡng chọn riêng cho gap đó nên chỉ gần đúng cấu hình triển khai.")
    # J07: khối adaptive_beta (power-law) của calibration, nếu có
    _ab = cal.get("adaptive_beta")   # J07
    if not _ab: print("\nadaptive_beta: not in this calibration")   # J07
    else:   # J07
        print("\n-- J07 adaptive beta | form:", _ab.get("form"), "| clip", _ab.get("clip"))
        print("   gamma", _ab.get("gamma"), "| gamma_source", _ab.get("gamma_source"), "| gamma_gain", _ab.get("gamma_gain"), "| min_gain", _ab.get("min_gain"))
        _gs = sorted(_ab["dbar"], key=int)
        print("\n   dbar[gap][class] (D trung bình/frame, full frame)"); print(f"   {'gap':>5s} " + " ".join(f"{c:>10s}" for c in CL))
        for g in _gs: print(f"   {g:>5s} " + " ".join(f"{f(_ab['dbar'][g][c], 3):>10s}" for c in CL))
        print("\n   beta_star[gap][class] (beta tại D = dbar)"); print(f"   {'gap':>5s} " + " ".join(f"{c:>10s}" for c in CL))
        for g in _gs: print(f"   {g:>5s} " + " ".join(f"{f(x, 4):>10s}" for x in _ab["beta_star"][g]))
        print("\n   CV (camera held-out) fixed -> adaptive: WAPE % | bias %")
        for g in _gs:
            _cv = _ab.get("cv", {}).get(g)
            if not _cv: continue
            print(f"   gap {g:>2s}: " + " | ".join(f"{k} WAPE {f(_cv[k]['wape_fixed'])}->{f(_cv[k]['wape_adaptive'])} bias {f(_cv[k]['bias_fixed'])}->{f(_cv[k]['bias_adaptive'])}" for k in ("motorcycle", "total") if k in _cv))
        print("\n   density_stats motorcycle (frames/ p50 / p95)")
        for g in _gs:
            _ds = _ab.get("density_stats", {}).get(g, {}).get("motorcycle")
            if _ds: print(f"   gap {g:>2s}: n_frames {_ds.get('n_frames')} | p50 {_ds.get('p50')} | p95 {_ds.get('p95')}")

### 02c — Calibration report & experiment logs (chỉ `calibration_only`)
Sinh **tự động** từ `calibration.json` của lần chạy này (không hard-code số YOLO11x): `calibration_summary.csv` (final-fit conf/β **tách khỏi** cột CV `cv_*`), `calibration_per_fold.csv` (source chỉ lưu per-fold cho gap 10 s), `experiment_metadata.json`. Log `calibration_run.log` đã được ghi bởi bước 02 (stdout+stderr+exit code của `flow_estimation.calibrate`). Gap 25 s ghi `NOT_IMPLEMENTED_IN_SOURCE`.

In [ ]:
# 02c. Report + logs. Chỉ đọc calibration.json của LẦN CHẠY NÀY; không tính lại thuật toán nào.
if RUN_MODE != "calibration_only":
    STATUS["calibration_report"] = "SKIPPED (chỉ dùng với RUN_MODE='calibration_only')"
    print(STATUS["calibration_report"])
else:
    with stage("calibration_report"):
        NI = "NOT_IMPLEMENTED_IN_SOURCE"
        cal = json.load(open(require(ACTIVE["calibration"], "calibration.json"), encoding="utf-8"))
        CL = cal["classes"]; nC = len(CL)
        cols = (["gap_s", "status", "final_fit_conf_motorcycle", "final_fit_conf_other"] + [f"final_fit_beta_{c}" for c in CL]
                + [f"cv_wape_{c}" for c in CL] + ["cv_wape_total"] + [f"cv_bias_{c}" for c in CL] + ["cv_bias_total", "note"])
        with open(CAL_RUN_DIR / "calibration_summary.csv", "w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh); w.writerow(cols)
            for g in (5, 10, 15, 20, 25, 30, "mix"):
                b, c = cal["beta"].get(str(g)), cal["cv"].get(str(g))
                if g == "mix":
                    assert len(c["wape"]) == nC + 1 and len(c["bias"]) == nC + 1
                    row = [g, "ok", cal["conf"]["motorcycle"], cal["conf"]["other"]] + ["N/A_MIX_USES_BETA_10_AND_15"] * nC + c["wape"] + c["bias"] + [
                        "mix: beta của từng cặp = β(10 s) hoặc β(15 s); CV fit ngưỡng ở gap 10 s"]
                elif b is None or c is None:
                    row = [g, NI] + [NI] * (2 + nC + (nC + 1) + (nC + 1)) + ["gap không nằm trong calibrate.GAPS; không có β/CV"]
                else:
                    assert len(c["wape"]) == nC + 1 and len(c["bias"]) == nC + 1 and len(b) == nC
                    row = [g, "ok", cal["conf"]["motorcycle"], cal["conf"]["other"]] + b + c["wape"] + c["bias"] + [
                        "final_fit_* = fit trên TẤT CẢ video (ngưỡng chọn ở gap 10 s, dùng cho mọi gap); cv_* = camera giữ riêng, fit lại từng fold"]
                assert len(row) == len(cols), (g, len(row), len(cols))
                w.writerow(row)
        fcols = (["fold", "gap_s", "train_videos", "test_videos", "test_cameras", "conf_motorcycle", "conf_other"]
                 + [f"beta_10s_{c}" for c in CL] + [f"test_wape_{c}" for c in CL] + ["scope_note"])
        with open(CAL_RUN_DIR / "calibration_per_fold.csv", "w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh); w.writerow(fcols)
            for k, fo in cal["folds"].items():
                assert len(fo["beta_10s"]) == nC and len(fo["test_wape"]) == nC
                w.writerow([k, 10, fo["train_videos"], fo["test_videos"], fo["test_cameras"], fo["conf_motorcycle"], fo["conf_other"],
                            *fo["beta_10s"], *fo["test_wape"], "source chỉ lưu per-fold cho gap 10 s (calibrate.py:134-138); không có per-fold cho gap khác"])
        CAL_CHECK = j02e_validate_calibration(CAL_RUN_DIR, CAL_RUN_DIR)      # J02E fix2: kiểm artifact của run này SAU khi report đã ghi
        CAL_OK = CAL_CHECK["status"] == "OK"
        print("calibration_artifact check:", CAL_CHECK["status"], "| layout", CAL_CHECK.get("layout"), "| ground_truth_kind", CAL_CHECK.get("ground_truth_kind"), "|", CAL_CHECK["_check_file"])
        write_run_metadata("COMPLETE" if CAL_OK else j02e_blocked_msg(CAL_CHECK))        # tạm thời; cell 37 xác nhận lần cuối
        REQ = ["detections.jsonl.gz", "calibration.json", "calibration_run.log", "calibration_summary.csv", "calibration_per_fold.csv", "experiment_metadata.json"]
        if HYBRID: REQ += ["detections.jsonl.gz.provenance.json", "calibration.json.provenance.json"]      # J01
        missing = [n for n in REQ if not (CAL_RUN_DIR / n).is_file() or (CAL_RUN_DIR / n).stat().st_size == 0]
        if missing:
            raise RuntimeError("Calibration KHÔNG hoàn chỉnh, thiếu output: " + ", ".join(missing))
        STATUS["calibration_report"] = "done" if CAL_OK else j02e_blocked_msg(CAL_CHECK)      # J02E fix2
        print("calibration run dir:", CAL_RUN_DIR)
        for n in REQ: print(f"  {n:28s} {(CAL_RUN_DIR / n).stat().st_size / 1e3:10.1f} KB")
        if CAL_OK: print("\nCALIBRATION COMPLETE")
        else: print("\nCALIBRATION BLOCKED/INSUFFICIENT_DATA — " + "; ".join(CAL_CHECK.get("problems") or []))      # J02E fix2

## 03 — Per-frame ROI counts & overlays (J02E, `flow_estimation.roi_frames`)
Chạy trên dets city ĐANG DÙNG (`ACTIVE`) + `CITY_INDEX`: mỗi ảnh → số xe trong từng ROI (FORWARD / REVERSE / EXCLUDE; camera không có ROI = FULL_FRAME) + overlay (cần ảnh). Ngưỡng: calibration `ACTIVE` (nhãn `CALIBRATED:<sha>`) hoặc `REVIEW_CONF` (nhãn `REVIEW_THRESHOLD_NOT_CALIBRATED`, image_review). **Đây là số đếm TỨC THỜI trong 1 ảnh — KHÔNG phải flow, KHÔNG phải veh/h** (flow veh/h = cell 04c). Output: `OUT_DIR/roi_frames/`. ROI/directional (J02): `flow_estimation/directional_roi.json`, mapping `sumo/config/camera_directions.json`.


In [ ]:
# J02E 03 — roi_frames: đếm xe theo ROI từng frame + overlay (instantaneous count per frame, NOT flow, NOT veh/h)
print("DEMAND_MODE:", DEMAND_MODE, "| J02 tag:", J02_TAG if J02 else "-")
if RUN_MODE == "calibration_only":
    print(STATUS["roi_frames"])
else:
    with stage("roi_frames"):
        ROI_OUT = OUT_DIR / "roi_frames"
        _idx = CITY_INDEX
        if not IMAGES:      # repo_cache: chỉ các session của SCENARIO_WINDOWS (index của repo chứa cả session khác)
            _idx = OUT_DIR / "inputs" / "city_index_active.jsonl"; _idx.parent.mkdir(parents=True, exist_ok=True)
            with open(CITY_INDEX, encoding="utf-8") as _fi, open(_idx, "w", encoding="utf-8") as _fo:
                for _l in _fi:
                    if _l.strip() and json.loads(_l)["session"] in SESSIONS: _fo.write(_l)
        _a = ["--index", _idx, "--dets", require(ACTIVE["dets_city"], "dets city ACTIVE"), "--roi", require(STAGE / "flow_estimation/directional_roi.json", "ROI config"),
              "--exclude", *EXCLUDE_CAMS, "--out", ROI_OUT]
        _cal_ok = ACTIVE.get("calibration") is not None and Path(ACTIVE["calibration"]).is_file()
        _a += ["--calibration", ACTIVE["calibration"]] if _cal_ok else ["--review-conf", REVIEW_CONF]
        _sc = Path(str(ACTIVE["dets_city"]) + ".provenance.json")
        _a += ["--dets-provenance", _sc] if _sc.is_file() else ["--backend", DETECTOR_BACKEND]
        if CITY_ROOT and Path(CITY_ROOT).is_dir(): _a += ["--root", CITY_ROOT, "--overlays", ROI_OVERLAYS_PER_CAMERA]
        else: print("J02E: không có ảnh (CITY_IMAGES / CITY_DATASET) — overlay bị bỏ qua (chỉ đếm)")
        pymod("flow_estimation.roi_frames", *_a)
        _rp = json.load(open(require(ROI_OUT / "roi_frames_provenance.json", "roi_frames_provenance.json"), encoding="utf-8"))
        for _n in ("counts_per_frame.csv", "summary_camera_roi.csv", "detections.csv"): require(ROI_OUT / _n, _n)
        _rows = list(csv.DictReader(open(ROI_OUT / "summary_camera_roi.csv", encoding="utf-8")))
        print("\nĐẾM TỨC THỜI TRONG 1 ẢNH (instantaneous count, NOT flow / NOT veh/h) — sum_* = tổng các frame, mean_* = trung bình mỗi frame; theo camera × ROI")
        _cols = ["session", "camera", "roi", "n_frames", "sum_motorcycle", "sum_car", "sum_truck", "sum_bus", "sum_total", "mean_total"]
        print(f"{'session':22s}{'camera':32s}{'roi':12s}" + "".join(f"{c:>15s}" for c in _cols[3:]))
        for _r in _rows:
            print(f"{_r['session']:22s}{_r['camera']:32s}{_r['roi']:12s}" + "".join((f"{float(_r[c]):>15.2f}" if c == "mean_total" else f"{int(float(_r[c])):>15d}") for c in _cols[3:]))
        _pngs = sorted((ROI_OUT / "overlays").rglob("*.png")) if (ROI_OUT / "overlays").is_dir() else []
        _show = _pngs[::max(1, len(_pngs) // max(SHOW_OVERLAYS, 1))][:SHOW_OVERLAYS]
        try:
            from IPython.display import Image as _IPImage, display as _ipdisplay
            for _png in _show: print(_png.relative_to(ROI_OUT)); _ipdisplay(_IPImage(filename=str(_png)))
        except ImportError:
            print("overlays (không có IPython):", [str(x.relative_to(ROI_OUT)) for x in _show])
        STATUS["roi_frames"] = f"done ({_rp['n_frames']} frame, {_rp['n_detections']} detection, threshold {_rp['threshold']['label']}, {len(_pngs)} overlay)"
    print(STATUS["roi_frames"])


## 04 — Flow Inference + Demand Settings
`flow_estimation.infer` (từ cache + frame index, không cần ảnh) → `perframe.json` → `flow_estimation.window` → `demand_settings.json`. Dùng **đúng** `ACTIVE` dets + calibration. Sau đó file settings được đặt vào `STAGE/results/flow/demand_settings.json` (nơi `build_demand.py:35` đọc).

**J02E:** `--index` = `CITY_INDEX` (repo_cache: `city_cam_frames.jsonl`; images: index do `city_index` dựng, đã lọc theo `SCENARIO_WINDOWS`); các mode = `EVAL_MODES`; so với `results/flow/demand_settings.json` của repo chỉ khi `CITY_SOURCE=repo_cache`.


In [ ]:
if DOWNSTREAM:
    with stage("infer"):
        j07_adaptive_on(ACTIVE["calibration"])   # J07: quyết định ADAPTIVE_BETA_EFF theo calibration dùng cho inference
        pymod("flow_estimation.infer", "--sessions", *SESSIONS, "--index", CITY_INDEX, "--exclude", *EXCLUDE_CAMS,
              "--dets", require(ACTIVE["dets_city"], "dets city ACTIVE"), "--calibration", require(ACTIVE["calibration"], "calibration ACTIVE"),
              "--out", FLOW_DIR, *(["--adaptive-beta"] if ADAPTIVE_BETA_EFF else []))   # J07: --adaptive-beta chỉ khi ADAPTIVE_BETA_EFF
        require(FLOW_DIR / "perframe.json", "perframe.json"); STATUS["infer"] = "done"
    with stage("settings"):
        args = ["--perframe", FLOW_DIR / "perframe.json", "--out", FLOW_DIR / "demand_settings.json"]
        for name, s, a, b in SETTING_WINDOWS: args += ["--setting", name, s, a, b]
        pymod("flow_estimation.window", *args)
        SETTINGS_OUT = require(FLOW_DIR / "demand_settings.json", "demand_settings.json vừa sinh")
        # nối sang build_demand.py: đặt đúng vị trí nó đọc, TRONG STAGE (không đụng results/flow của REPO)
        dst = STAGE / "results/flow/demand_settings.json"; dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(SETTINGS_OUT, dst)
        cur = json.load(open(dst, encoding="utf-8")); ref = json.load(open(BASE["demand_settings"], encoding="utf-8")) if not IMAGES else None   # J02E: so với repo chỉ khi repo_cache
        print("settings dùng cho route generation:", dst)
        for m in EVAL_MODES: print(f"  {m:7s}", cur[m]["mean_of_cameras"], "| giống repo baseline:", (cur[m] == ref[m]) if ref is not None else "n/a (CITY_SOURCE=images)")   # J02E
        STATUS["settings"] = "done"
else:
    STATUS["infer"] = f"SKIPPED ({RUN_MODE}: by design)"; STATUS["settings"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): flow inference + demand settings")

### 04b — Gap thực của feed và β được dùng
Chỉ đếm bằng `flow_estimation.common.nearest_gap` của source.

In [ ]:
if DOWNSTREAM:
    # 04b. Gap thực của feed -> β nào được dùng. Dùng chính flow_estimation.common.nearest_gap (infer.py:70); không viết lại.
    import numpy as np
    sys.dont_write_bytecode = True
    if str(STAGE) not in sys.path: sys.path.insert(0, str(STAGE))
    from flow_estimation.common import nearest_gap
    P = json.load(open(require(FLOW_DIR / "perframe.json", "perframe.json"), encoding="utf-8"))
    gaps_beta = [int(g) for g in cal["beta"]]                       # gap có β trong calibration đang dùng
    use, invalid, pairs, near25 = {}, 0, 0, 0
    for key, d in P.items():
        dts, ok = np.array(d["dt"]), np.array(d["valid"])
        pairs += len(dts) - 1; invalid += int((~ok[1:]).sum())
        for g in nearest_gap(dts[ok], gaps_beta).tolist(): use[g] = use.get(g, 0) + 1
        near25 += int(((dts[ok] > 22.5) & (dts[ok] <= 27.5)).sum())
    print("cặp ảnh liên tiếp:", pairs, "| bị bỏ (dt > 35 s, dữ liệu thiếu):", invalid)
    for g in sorted(use): print(f"  dùng β({g:>2d} s): {use[g]:6d} cặp  ({use[g] / max(pairs - invalid, 1) * 100:5.1f}%)")
    print(f"  cặp có dt trong (22.5, 27.5] s: {near25} -> ánh xạ sang β gần nhất ở trên; KHÔNG có β riêng cho 25 s")
else:
    print(f"SKIPPED (RUN_MODE={RUN_MODE}): gap usage của feed")

## 04c — J02 Directional ROI flow (Cam2/Cam3/Cam4)
`python -m flow_estimation.directional infer` (cùng cache + calibration `ACTIVE`, full-frame detection; mask tĩnh full-frame; matching RIÊNG trong từng ROI; EXCLUDE bỏ; β class×gap) → `perframe_directional.json`; `... window` → `demand_settings_directional.json` (veh/h CHƯA scale; `fullframe` cho mọi camera = `window.py`). Hybrid: dùng đúng artifact hybrid, không thay bằng YOLO11x.


In [ ]:
if DOWNSTREAM and J02:
    with stage("directional"):
        ROI_CFG = require(STAGE / "flow_estimation/directional_roi.json", "J02 ROI config")
        require(STAGE / "sumo/config/camera_directions.json", "J02 camera_directions.json")
        j07_adaptive_on(ACTIVE["calibration"], quiet=True)   # J07: cùng quyết định như cell flow infer (chạy lại cell này riêng vẫn đúng)
        pymod("flow_estimation.directional", "infer", "--sessions", *SESSIONS, "--index", CITY_INDEX, "--exclude", *EXCLUDE_CAMS,
              "--dets", require(ACTIVE["dets_city"], "dets city ACTIVE"), "--calibration", require(ACTIVE["calibration"], "calibration ACTIVE"),
              "--roi", ROI_CFG, "--out", FLOW_DIR, *(["--adaptive-beta"] if ADAPTIVE_BETA_EFF else []))   # J07: --adaptive-beta chỉ khi ADAPTIVE_BETA_EFF
        args = ["window", "--perframe", require(FLOW_DIR / "perframe.json", "perframe.json"),
                "--perframe-directional", require(FLOW_DIR / "perframe_directional.json", "perframe_directional.json"),
                "--out", FLOW_DIR / "demand_settings_directional.json"]
        for name, s, a, b in SETTING_WINDOWS: args += ["--setting", name, s, a, b]
        pymod("flow_estimation.directional", *args)
        J02_SETTINGS = require(FLOW_DIR / "demand_settings_directional.json", "demand_settings_directional.json")
        _ds, _cur = json.load(open(J02_SETTINGS, encoding="utf-8")), json.load(open(FLOW_DIR / "demand_settings.json", encoding="utf-8"))
        for m in EVAL_MODES:      # J02E
            for cam, v in _cur[m]["cameras"].items():
                assert _ds[m]["cameras"][cam]["fullframe"] == v, f"J02 fullframe != demand_settings.json ({m}/{cam})"
        print("veh/h CHƯA scale (fullframe | FORWARD | REVERSE | (F+R)/fullframe):")
        for m in EVAL_MODES:      # J02E
            for cam, v in _ds[m]["cameras"].items():
                f, r = v.get("FORWARD", {}).get("total"), v.get("REVERSE", {}).get("total")
                ratio = f"{(f + r) / max(v['fullframe']['total'], 1):.2f}" if f is not None else "-"
                print(f"  {m:6s} {cam:30s} {v['fullframe']['total']:6d} | {f if f is not None else '-':>6} | {r if r is not None else '-':>6} | {ratio}")
        STATUS["directional"] = "done"
elif DOWNSTREAM:
    print("SKIPPED (DEMAND_MODE=baseline_5050): J02 directional")
else:
    STATUS["directional"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): J02 directional")


## 05 — IMC (M3 / J03) — PLACEHOLDER
Chưa implement. Không có code giả.

In [ ]:
# TODO(J03): Inverse Markov Chain + adapter routeSampler. Chưa implement.
print("M3 placeholder — chưa implement")

## 06 — Route Generation
Gọi `build_demand.py --split SPLIT --mode <mode> --scale SCALE` **trong STAGE**, không `--tag` → ghi `STAGE/sumo/scenarios/<split>/<mode>/` (`build_demand.py:160-163`) và cache `STAGE/sumo/demand_cache`. Giữ nguyên `osm.sumocfg`, `scenario.json` do script sinh.

**J02 (`DEMAND_MODE="j02_mixed"`):** thêm `--settings <demand_settings_directional.json> --directions sumo/config/camera_directions.json --tag j02` → `STAGE/sumo/scenarios_j02/<split>/<mode>/` + `counts_provenance.json` + `mismatch.<class>.xml`. Cam1 = fullframe/2×SCALE; Cam2/3/4 = rate×SCALE (không /2).

`SCEN_ROOT` = `STAGE/sumo/scenarios_j02` (j02_mixed) hoặc `STAGE/sumo/scenarios` (baseline_5050) — cùng giá trị được truyền cho controller ở 08.

In [ ]:
if DOWNSTREAM:
    SCEN_ROOT = STAGE / "sumo" / (f"scenarios_{J02_TAG}" if J02 else "scenarios")     # J02: thư mục riêng
    J02_ARGS = (["--settings", require(FLOW_DIR / "demand_settings_directional.json", "J02 settings"),
                 "--directions", require(STAGE / "sumo/config/camera_directions.json", "J02 directions"), "--tag", J02_TAG] if J02 else [])
    # scenarios cần dựng bằng build_demand.py (SPLIT_SEEDS cố định của source: train 1001 / val 2002 / test 3003)
    NEED_SCEN = [(SPLIT, m) for m in EVAL_MODES]
    if RUN_TRAIN:      # run_tsc.py train: env = split train, val_env = split val cùng --mode (run_tsc.py:150-152). KHÔNG dùng test thay val. J04: mọi mode trong MPLIGHT_TRAIN_MODES.
        for _tm in MPLIGHT_TRAIN_MODES: NEED_SCEN += [("train", _tm), ("val", _tm)]
    NEED_SCEN = list(dict.fromkeys(NEED_SCEN))
    with stage("routes"):
        for split, mode in NEED_SCEN:
            assert not (SCEN_ROOT / split / mode).exists(), f"{SCEN_ROOT/split/mode} đã tồn tại — dùng RUN_ID mới (không ghi đè)"
            pyscript("build_demand.py", "--split", split, "--mode", mode, "--scale", SCALE, *J02_ARGS, log=OUT_DIR / "logs" / f"build_demand_{split}_{mode}.log")
        if INCLUDE_STRESS:      # J04: scenario stress test-heavy do build_trips.py dựng sẵn (random-density, KHÔNG phải demand camera). STAGE không chứa sumo/scenarios => copy từ REPO (chỉ đọc).
            _ssrc = require(REPO_DIR / "sumo/scenarios/stress" / STRESS_MODE / "osm.sumocfg", f"scenario stress {STRESS_MODE} (REPO)").parent
            _sdst = SCEN_ROOT / "stress" / STRESS_MODE
            assert not _sdst.exists(), f"{_sdst} đã tồn tại — dùng RUN_ID mới (không ghi đè)"
            shutil.copytree(_ssrc, _sdst, ignore=shutil.ignore_patterns("__pycache__"))     # cfg dùng ../../../map/... => cùng độ sâu dưới STAGE/sumo/
            _sbad = [f.name for f in _ssrc.iterdir() if f.is_file() and _sha256(f) != _sha256(_sdst / f.name)]
            assert not _sbad, f"copy stress khác REPO: {_sbad}"
            print(f"stress {STRESS_MODE}: copy từ {_ssrc} -> {_sdst} (demand random-density, không phải demand camera)")
        STATUS["routes"] = "done (" + ", ".join(f"{s}/{m}" for s, m in NEED_SCEN) + (f", stress/{STRESS_MODE} (copy từ REPO)" if INCLUDE_STRESS else "") + ")"
else:
    STATUS["routes"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): route generation")

## 07 — SUMO Scenario Validation
Mọi `net-file`, `route-files`, `additional-files`, `gui-settings-file` trong `osm.sumocfg` phải tồn tại **và nằm trong STAGE**. `scenario.json` phải khớp `SCALE`/`SPLIT`/`mode`. `SCEN_SHA` = hash cố định của cấu hình+routes+vehicle types — dùng để chứng minh mọi controller chạy trên cùng demand.

**Binding:** ghi `OUT_DIR/scenario_expected_binding.json` (cfg + SHA cfg/net/routes theo `<split>/<mode>`). `scenario.json` phải khớp `DEMAND_MODE`; cfg phải nằm trong `SCEN_ROOT`.

In [ ]:
if DOWNSTREAM:
    def fingerprint_scenario(cfg):
        """Hash NỘI DUNG các input thực sự điều khiển SUMO. Theo tsc/env.py:_sumocfg_args, SUMO nhận net-file, route-files và các option ngoài
        {output, report, gui_only, input} của chính .sumocfg (additional-files / gui-settings KHÔNG được env dùng => không hash).
        Thêm: scenario.json, junctions.json + hcmc.net.xml.gz (run_tsc.load_junctions) và tsc/config.py (ENV_CONF/SUMO_CONF: cấu hình env + tín hiệu)."""
        cfg = Path(cfg); base = cfg.parent; net, routes = None, []
        for sec in ET.parse(cfg).getroot():
            if sec.tag.split("}")[-1] != "input": continue
            for opt in sec:
                k, v = opt.tag.split("}")[-1], opt.get("value")
                if k == "net-file": net = (base / v).resolve()
                elif k == "route-files": routes = [(base / x.strip()).resolve() for x in v.split(",")]
        if net is None or not routes: raise ValueError(f"{cfg}: thiếu net-file/route-files")
        comp = {"sumocfg": _sha256(cfg), "net": _sha256(net)}
        for r in routes: comp["route:" + r.name] = _sha256(r)
        comp["scenario.json"] = _sha256(base / "scenario.json") if ((base / "scenario.json").is_file() or base.resolve() != (SCEN_ROOT / "stress" / STRESS_MODE).resolve()) else "absent"   # J04: chỉ stress được thiếu scenario.json
        comp["controller_net (sumo/map/hcmc.net.xml.gz)"] = _sha256(STAGE / "sumo/map/hcmc.net.xml.gz")
        comp["junctions.json"] = _sha256(STAGE / "sumo/config/junctions.json")
        comp["tsc/config.py"] = _sha256(STAGE / "tsc/config.py")
        assert comp["net"] == comp["controller_net (sumo/map/hcmc.net.xml.gz)"], "net của scenario khác net dùng dựng junction/agent"
        return dict(overall=hashlib.sha256(json.dumps(comp, sort_keys=True).encode()).hexdigest(), components=comp)

    def check_scenario(mode, split=None):
        split = split or SPLIT
        cfg = require(SCEN_ROOT / split / mode / "osm.sumocfg", f"osm.sumocfg {split}/{mode}")
        refs, root = [], ET.parse(cfg).getroot()
        for sec in root:
            for opt in sec:
                k, v = opt.tag.split("}")[-1], opt.get("value")
                if k in ("net-file", "route-files", "additional-files", "gui-settings-file") and v:
                    refs += [(k, (cfg.parent / x.strip()).resolve()) for x in v.split(",")]
        missing = [str(p) for k, p in refs if not p.exists()]
        outside = [str(p) for k, p in refs if not str(p).startswith(str(STAGE.resolve()))]
        assert not missing, f"{split}/{mode}: file SUMO tham chiếu không tồn tại: {missing}"
        assert not outside, f"{split}/{mode}: tham chiếu ra ngoài STAGE: {outside}"
        meta = json.load(open(cfg.parent / "scenario.json", encoding="utf-8"))
        assert SCEN_ROOT.resolve() in cfg.resolve().parents, f"{cfg} nằm ngoài SCEN_ROOT {SCEN_ROOT}"          # INT-01
        assert (meta.get("demand") or {}).get("mode", "baseline_5050") == DEMAND_MODE, f"{split}/{mode}: scenario.json demand {meta.get('demand')} khác DEMAND_MODE={DEMAND_MODE}"
        assert meta["scale"] == SCALE and meta["split"] == split and meta["setting"] == mode, meta
        fp = fingerprint_scenario(cfg)
        return dict(mode=mode, split=split, cfg=cfg, n_refs=len(refs), meta=meta, fingerprint=fp, sha=fp["overall"][:12])

    def assert_frozen(entry, when=""):
        """Input SUMO của scenario phải y hệt lúc đóng băng (so NỘI DUNG, không so tên file)."""
        cur, ref = fingerprint_scenario(entry["cfg"]), entry["fingerprint"]
        if cur["overall"] != ref["overall"]:
            diff = sorted(k for k in set(cur["components"]) | set(ref["components"]) if cur["components"].get(k) != ref["components"].get(k))
            raise RuntimeError(f"input SUMO bị đổi {when}: {entry['split']}/{entry['mode']} -> {diff}")

    def expected_binding(entry):
        """INT-02: cfg + SHA cfg/net/routes mà controller PHẢI load (từ fingerprint đã đóng băng)."""
        c = entry["fingerprint"]["components"]
        return dict(cfg=str(Path(entry["cfg"]).resolve()), cfg_sha256=c["sumocfg"], net_sha256=c["net"],
                    routes={k.split(":", 1)[1]: v for k, v in c.items() if k.startswith("route:")})

    def binding_sha(b):
        """Hash input SUMO controller THỰC SỰ load (cfg/net/routes) — fairness dựa trên cái này, không dựa nhãn notebook."""
        routes = {n: (r["sha256"] if isinstance(r, dict) else r) for n, r in b["routes"].items()}
        return hashlib.sha256(json.dumps(dict(cfg=b["cfg_sha256"], net=b["net_sha256"], routes=routes), sort_keys=True).encode()).hexdigest()

    def verify_binding(b, entry, what):
        """So scenario do controller resolve (run_tsc binding) với scenario đã đóng băng. Sai => dừng, KHÔNG nhận metrics."""
        exp, diff = expected_binding(entry), []
        if os.path.normcase(str(Path(b["cfg"]).resolve())) != os.path.normcase(exp["cfg"]): diff.append(f"cfg {b['cfg']} != {exp['cfg']}")
        if SCEN_ROOT.resolve() not in Path(b["cfg"]).resolve().parents: diff.append(f"cfg ngoài SCEN_ROOT {SCEN_ROOT}")
        diff += [k for k in ("cfg_sha256", "net_sha256") if b.get(k) != exp[k]]
        if {n: r["sha256"] for n, r in b["routes"].items()} != exp["routes"]: diff.append("routes sha256")
        if (b.get("split"), b.get("mode")) != (entry["split"], entry["mode"]): diff.append(f"split/mode {b.get('split')}/{b.get('mode')}")
        _edm = entry.get("expected_demand_mode", DEMAND_MODE)      # J04: stress (không scenario.json) => baseline_5050
        if b.get("demand_mode") != _edm: diff.append(f"demand_mode {b.get('demand_mode')} != {_edm}")
        if diff: raise RuntimeError(f"SCENARIO BINDING MISMATCH ({what}): " + "; ".join(diff))
        return dict(b, binding_sha=binding_sha(b))

    def check_stress():
        """J04: stress test-heavy (static, KHÔNG có scenario.json, demand random-density). Cùng dạng entry như check_scenario."""
        split, mode = SPLIT, STRESS_MODE
        cfg = require(SCEN_ROOT / "stress" / mode / "osm.sumocfg", f"osm.sumocfg stress/{mode}")
        refs, root = [], ET.parse(cfg).getroot()
        for sec in root:
            for opt in sec:
                k, v = opt.tag.split("}")[-1], opt.get("value")
                if k in ("net-file", "route-files", "additional-files", "gui-settings-file") and v:
                    refs += [(k, (cfg.parent / x.strip()).resolve()) for x in v.split(",")]
        missing = [str(p) for k, p in refs if not p.exists()]
        outside = [str(p) for k, p in refs if not str(p).startswith(str(STAGE.resolve()))]
        assert not missing, f"stress/{mode}: file SUMO tham chiếu không tồn tại: {missing}"
        assert not outside, f"stress/{mode}: tham chiếu ra ngoài STAGE: {outside}"
        assert SCEN_ROOT.resolve() in cfg.resolve().parents, f"{cfg} nằm ngoài SCEN_ROOT {SCEN_ROOT}"
        assert not (cfg.parent / "scenario.json").exists(), "stress không được có scenario.json (demand camera)"
        fp = fingerprint_scenario(cfg)
        meta = {"demand": "stress_random", "note": "build_trips.py: random-density demand (DENSITY_SCALE test-heavy=12), KHÔNG phải demand ước lượng từ camera; giống hệt ở mọi split"}
        return dict(mode=mode, split=split, cfg=cfg, n_refs=len(refs), meta=meta, fingerprint=fp, sha=fp["overall"][:12],
                    expected_demand_mode="baseline_5050")      # run_tsc.scenario_binding báo "baseline_5050" khi không có scenario.json (run_tsc.py:124-135)

    with stage("scenario_check"):
        SCEN = {m: check_scenario(m, SPLIT) for m in EVAL_MODES}
        for m, s in SCEN.items():
            rcfg = REPO_DIR / "sumo/scenarios" / SPLIT / m / "osm.sumocfg"
            same = None
            if rcfg.is_file() and not IMAGES:   # chỉ thông tin (J02E: bỏ qua khi CITY_SOURCE=images): có tái tạo đúng scenario baseline của repo không
                norm = lambda p: p.read_bytes().splitlines()      # so sanh theo dong: bo qua khac biet CRLF/LF Windows-Linux
                same = all(norm(REPO_DIR / "sumo/scenarios" / SPLIT / m / f) == norm(s["cfg"].parent / f)
                           for f in ("osm.motorcycle.trips.xml", "osm.passenger.trips.xml", "osm.truck.trips.xml", "osm.bus.trips.xml", "osm.bicycle.trips.xml"))
            print(f"{m:7s} refs ok={s['n_refs']} | fingerprint {s['sha']} ({len(s['fingerprint']['components'])} thành phần) | seed {s['meta']['seed']} scale {s['meta']['scale']} | vehicles {s['meta']['vehicles']} | routes == repo baseline (bỏ qua CRLF): {same}" + (" [J02: khác baseline là ĐÚNG — demand theo hướng]" if J02 else ""))
        if RUN_TRAIN:      # điều kiện tiên quyết của run_tsc.py train: scenarios train + val (cùng mode, cùng net/scale/demand window, seed khác) — J04: mỗi mode trong MPLIGHT_TRAIN_MODES
            TRAIN_SCEN_BY_MODE = {}
            for _tm in MPLIGHT_TRAIN_MODES:
                TRAIN_SCEN_BY_MODE[_tm] = {sp: check_scenario(_tm, sp) for sp in ("train", "val")}
                t, v = TRAIN_SCEN_BY_MODE[_tm]["train"], TRAIN_SCEN_BY_MODE[_tm]["val"]
                assert t["fingerprint"]["components"]["net"] == v["fingerprint"]["components"]["net"] == next(iter(SCEN.values()))["fingerprint"]["components"]["net"], "train/val/eval khác network"
                assert t["meta"]["source_window"] == v["meta"]["source_window"] and t["meta"]["scale"] == v["meta"]["scale"], "train/val khác demand specification"
                assert t["meta"]["seed"] != v["meta"]["seed"], "train và val trùng seed"
                assert "test" not in (t["split"], v["split"]), "KHÔNG train/validate trên split test"
                for sp, s in TRAIN_SCEN_BY_MODE[_tm].items(): print(f"train-prereq {sp}/{_tm}: fingerprint {s['sha']} seed {s['meta']['seed']} vehicles {s['meta']['vehicles']}")
            TRAIN_SCEN = TRAIN_SCEN_BY_MODE[TRAIN_MODE]      # dạng phẳng {split: entry} của mode đầu tiên: cell 01b (metadata) vẫn đọc dạng này
        # J04: stress (nếu bật) + benchmark = camera modes + stress
        BENCH_SCEN = dict(SCEN, **({STRESS_MODE: check_stress()} if INCLUDE_STRESS else {}))
        if INCLUDE_STRESS:
            _st = BENCH_SCEN[STRESS_MODE]; print(f"stress {STRESS_MODE}: refs ok={_st['n_refs']} | fingerprint {_st['sha']} | demand: random-density (KHÔNG phải camera)")
        # INT-01/02: expected binding cho run_tsc --expect-binding (eval/train/custom dùng CÙNG file)
        EXPECT_BINDING_PATH = OUT_DIR / "scenario_expected_binding.json"
        _exp = {f"{s['split']}/{s['mode']}": expected_binding(s) for s in list(BENCH_SCEN.values()) + [e for _d in (TRAIN_SCEN_BY_MODE.values() if RUN_TRAIN else []) for e in _d.values()]}
        EXPECT_BINDING_PATH.write_text(json.dumps(_exp, indent=1), encoding="utf-8")
        print("SCEN_ROOT:", SCEN_ROOT, "| expected binding:", EXPECT_BINDING_PATH, "|", ", ".join(_exp))
        # controller: junction agents dựng được từ net (không mô phỏng)
        sh([sys.executable, "run_tsc.py", "junctions"], log=OUT_DIR / "logs" / "junctions.log")
        STATUS["scenario_check"] = "done"
else:
    STATUS["scenario_check"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): SUMO scenario validation")

### 07b — J02 target audit (counts_provenance.json + routeSampler mismatch)
Mỗi target: camera, mode (`fullframe_5050`|`directional_roi`), hướng nguồn, edge, rate đo, split_factor, SCALE, target. Kiểm: không edge trùng; edges = `camera_segments.json`; SCALE = cell 00a; deficit routeSampler.

Bắt buộc đủ 4 `mismatch.<class>.xml`, đúng 1 interval 0–3600, edge = target, `measuredCount` = `target_int`, có `deficit`/`GEH`. Thiếu/hỏng => lỗi (không coi là deficit 0). Deficit≠0 được báo (WARNING), không tự chặn. Deficit là khớp target ở mức route của routeSampler, KHÔNG phải số xe đi qua edge trong mô phỏng.


In [ ]:
if DOWNSTREAM and J02:
    with stage("directional_targets"):
        seg = json.load(open(STAGE / "sumo/config/camera_segments.json", encoding="utf-8"))
        seg_edges = {e for c, s in seg.items() if not c.startswith("_") for g in s["directions"] for e in g}
        J02_TARGETS = {}
        for m, s in SCEN.items():
            d = s["cfg"].parent; rows = json.load(open(require(d / "counts_provenance.json", f"counts_provenance {m}"), encoding="utf-8"))
            rows = rows["rows"] if isinstance(rows, dict) else rows
            keys = [(r["edge"], r["vclass"]) for r in rows]
            assert len(keys) == len(set(keys)), f"{m}: edge×class trùng trong counts_provenance"
            assert {r["edge"] for r in rows} == seg_edges, f"{m}: edges khác camera_segments.json"
            assert all(abs(float(r["scale"]) - SCALE) < 1e-12 for r in rows), f"{m}: scale khác SCALE"
            for r in rows:
                assert (r["mode"], float(r["split_factor"])) in (("fullframe_5050", 0.5), ("directional_roi", 1.0)), r
            # QA-01: thiếu/hỏng mismatch => LỖI (không bao giờ coi là deficit 0)
            deficit, geh = {}, {}
            for vc in ("motorcycle", "passenger", "truck", "bus"):
                f = require(d / f"mismatch.{vc}.xml", f"routeSampler mismatch {m}/{vc}")
                ints = list(ET.parse(f).getroot().iter("interval"))
                assert len(ints) == 1 and float(ints[0].get("begin")) == 0.0 and float(ints[0].get("end")) == 3600.0, f"{f}: cần đúng 1 interval 0-3600, thấy {[(i.get('begin'), i.get('end')) for i in ints]}"
                exp = {r["edge"]: int(r["target_int"]) for r in rows if r["vclass"] == vc}
                got = {}
                for e in ints[0].iter("edge"):
                    miss = [a for a in ("id", "measuredCount", "deficit", "GEH") if e.get(a) is None]
                    assert not miss, f"{f}: edge {e.get('id')} thiếu thuộc tính {miss}"
                    assert e.get("id") not in got, f"{f}: edge trùng {e.get('id')}"
                    got[e.get("id")] = (float(e.get("measuredCount")), float(e.get("deficit")), float(e.get("GEH")))
                assert set(got) == set(exp) and exp, f"{f}: edges {sorted(got)} != targets {sorted(exp)}"
                bad = {k: (v[0], exp[k]) for k, v in got.items() if v[0] != exp[k]}
                assert not bad, f"{f}: measuredCount khác target_int {bad}"
                for k, v in got.items(): deficit[f"{vc}:{k}"], geh[f"{vc}:{k}"] = v[1], v[2]
            assert len(deficit) == len(rows), f"{m}: {len(deficit)} edge×class trong mismatch != {len(rows)} targets"
            J02_TARGETS[m] = dict(n_rows=len(rows), total_target=sum(int(r["target_int"]) for r in rows), mismatch_entries=len(deficit),
                                  total_abs_deficit=sum(abs(v) for v in deficit.values()), max_GEH=max(geh.values()),
                                  nonzero_deficit={k: v for k, v in deficit.items() if v})
            t = J02_TARGETS[m]
            print(f"{m:6s} targets={len(rows)} Σtarget={t['total_target']} | mismatch {t['mismatch_entries']}/{len(rows)} edge×class | Σ|deficit|={t['total_abs_deficit']:g} max GEH={t['max_GEH']:.2f}"
                  + (f" | WARNING deficit≠0: {t['nonzero_deficit']}" if t["nonzero_deficit"] else ""))
        print("(deficit = khớp target ở mức route của routeSampler; KHÔNG phải số xe đi qua edge trong mô phỏng)")
        _nz = sum(len(t["nonzero_deficit"]) for t in J02_TARGETS.values())
        STATUS["directional_targets"] = "done" + (f" (WARNING: {_nz} edge×class deficit≠0)" if _nz else " (deficit=0 mọi edge×class)")
elif DOWNSTREAM:
    print("SKIPPED (DEMAND_MODE=baseline_5050): J02 target audit")
else:
    STATUS["directional_targets"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): J02 target audit")


## 08 — Controller selection & SUMO simulation
- Có sẵn: `fixedtime`, `maxpressure` (+ `mplight` nếu có checkpoint) chạy bằng `python run_tsc.py eval --algo … --mode … --split … --seed … --run-counts … --out … --scenario-root SCEN_ROOT --expect-binding <expected>` (headless, `libsumo` hoặc fallback `traci`). Mỗi run ghi `binding_*.json` (cfg/net/routes SHA thực sự load); notebook so lại với scenario đã đóng băng rồi mới nhận metrics.
- Controller của bạn: source **không có registry** (`run_tsc.py:85,139-148` hard-code). Notebook chỉ nối module của bạn vào **vòng lặp có sẵn**: `run_tsc.load_junctions()` → `run_tsc.make_env()` (cùng `scenario_root`/`expect_binding`) → `run_tsc.run_episode()` (gọi `agent.act(obs, explore=False)`, trả `{junction_id: phase_index}`). Không có simulation loop mới.
- **Fair benchmark:** cùng `SPLIT`, `mode`, `SEED`, `RUN_COUNTS`; hash scenario được kiểm trước/sau mỗi run. Chỉ policy đèn thay đổi.
- Giới hạn của source (không sửa): `fixedtime` chạy chương trình đèn của network (30 s xanh + 3 s vàng, `build_network.py:40`), còn agent quyết định mỗi 10 s với 5 s vàng khi đổi pha (`tsc/config.py:31-32`); không có min/max green hay all-red hiệu lực (`ALL_RED_TIME=0`).

In [ ]:
if DOWNSTREAM:
    import numbers, numpy as np
    from collections.abc import Mapping
    BUILTIN = ("fixedtime", "maxpressure", "mplight")
    PLAN_RUNS, SKIPPED = [], []
    with stage("controllers"):
        # 08a. Chọn controller (từ config). Thiếu điều kiện -> ghi rõ SKIPPED, không giả lập.
        for c in CONTROLLERS:
            if c not in BUILTIN: raise ValueError(f"controller không có trong source: {c} (user agent cấu hình riêng bằng USER_AGENT_MODULE)")
            if c != "mplight": PLAN_RUNS.append(c)      # mplight chỉ chạy khi có MPLIGHT_CKPT
        if "mplight" in CONTROLLERS and (MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE or RUN_TRAIN):      # J04: checkpoint có thể sinh ra ở cell 08b (training) => KHÔNG yêu cầu file ở đây
            PLAN_RUNS.append("mplight")
            for _p in list(MPLIGHT_CKPT_BY_MODE.values()) + ([MPLIGHT_CKPT_ALL] if MPLIGHT_CKPT_ALL else []): require(_p, "MPLIGHT_CKPT")
        elif "mplight" in CONTROLLERS: SKIPPED.append(("mplight", "không có checkpoint (MPLIGHT_CKPT=None) và RUN_TRAIN=False"))
        else: SKIPPED.append(("mplight", "không yêu cầu (không có trong CONTROLLERS)"))
        if USER_AGENT_MODULE:
            if USER_AGENT_NAME in BUILTIN or USER_AGENT_NAME in CONTROLLERS: raise ValueError(f"USER_AGENT_NAME '{USER_AGENT_NAME}' trùng tên controller có sẵn {BUILTIN}")
            if not re.fullmatch(r"[A-Za-z0-9_-]+", USER_AGENT_NAME): raise ValueError(f"USER_AGENT_NAME không hợp lệ (dùng chữ/số/_/-): {USER_AGENT_NAME!r}")
            PLAN_RUNS.append(USER_AGENT_NAME)
        else: SKIPPED.append((USER_AGENT_NAME, "chưa cấu hình USER_AGENT_MODULE"))
        PLAN_RUNS = list(dict.fromkeys(PLAN_RUNS))
        if not PLAN_RUNS: raise ValueError("không có controller nào để chạy")
    print("sẽ chạy:", PLAN_RUNS); print("bỏ qua :", SKIPPED)

    if "MPLIGHT_SELECTED" not in globals(): MPLIGHT_SELECTED, MPLIGHT_SELECTION = {}, {}     # mode -> checkpoint tự chọn theo val (cell 08b); J05: = trial ĐẦU TIÊN (tương thích)
    if "MPLIGHT_SELECTED_TRIALS" not in globals(): MPLIGHT_SELECTED_TRIALS, MPLIGHT_SELECTION_TRIALS = {}, {}     # J05: mode -> {train_seed: checkpoint / selection dict}
    J05_TI_COLS = ["delay", "delay_with_depart", "delay_all", "waiting_time", "waiting_time_all", "trip_duration", "n_trips_unfinished", "n_trips_undeparted"]      # J05: cột từ tsc/tripinfo.py::parse_tripinfo (thêm khi BENCH_TRIPINFO)

    def _mplight_is_manual(mode):
        src = STRESS_MPLIGHT_FROM if mode == STRESS_MODE else mode
        return bool(MPLIGHT_CKPT_BY_MODE.get(mode) or MPLIGHT_CKPT_ALL or MPLIGHT_CKPT_BY_MODE.get(src))

    def _mplight_trials(mode):
        """J05: danh sách train_seed của các lượt eval MPLight cho `mode`: [None] nếu checkpoint thủ công (1 lượt, không có train seed), ngược lại MPLIGHT_TRAIN_SEEDS_EFF."""
        return [None] if _mplight_is_manual(mode) else list(MPLIGHT_TRAIN_SEEDS_EFF)

    def _n_runs(ctrl, mode): return len(_mplight_trials(mode)) if ctrl == "mplight" else 1

    def _mplight_resolve(mode, train_seed=None):
        """Thứ tự: MPLIGHT_CKPT[mode] (dict) > MPLIGHT_CKPT (1 đường dẫn) > checkpoint tự chọn theo val của trial `train_seed` (None => trial đầu). Stress: mode stress -> dict[stress]/str, rồi checkpoint của STRESS_MPLIGHT_FROM."""
        src = STRESS_MPLIGHT_FROM if mode == STRESS_MODE else mode
        for origin, smode, p in (("manual_dict", mode, MPLIGHT_CKPT_BY_MODE.get(mode)), ("manual_all", mode, MPLIGHT_CKPT_ALL),
                                 ("manual_dict", src, MPLIGHT_CKPT_BY_MODE.get(src))):
            if not p: continue
            p = require(p, f"checkpoint MPLight cho mode {mode}")
            mr = re.search(r"round_(\d+)", Path(p).name); rnd = int(mr.group(1)) if mr else None
            return dict(path=Path(p), origin=origin, source_mode=smode, round=rnd, train_seed=None)
        tr = MPLIGHT_SELECTED_TRIALS.get(src) or {}
        ts = train_seed if train_seed is not None else next(iter(tr), None)
        if ts in tr and tr[ts]:
            return dict(path=Path(require(tr[ts], f"checkpoint MPLight cho mode {mode} (train seed {ts})")), origin="auto_selected", source_mode=src, round=MPLIGHT_SELECTION_TRIALS[src][ts]["round"], train_seed=ts)
        raise RuntimeError(f"MPLight: không có checkpoint cho mode '{mode}' train seed {train_seed} (cần MPLIGHT_CKPT hoặc training cell 08b với mode '{src}'; selected={sorted(MPLIGHT_SELECTED)})")

    def mplight_ckpt(mode, train_seed=None): return _mplight_resolve(mode, train_seed)["path"]

    def write_meta(rd, ctrl, mode, results_json, extra=None):
        s = BENCH_SCEN[mode]; b = (extra or {}).get("actual_binding")
        if not b: raise RuntimeError(f"{mode}/{ctrl}: thiếu actual scenario binding — không ghi kết quả benchmark")   # INT-02
        _edm = s.get("expected_demand_mode", DEMAND_MODE)      # J04: stress => baseline_5050 (run_tsc không có scenario.json)
        extra = dict(extra, demand_mode=_edm, demand_source=("stress_random" if mode == STRESS_MODE else "camera:" + DEMAND_MODE), episodes=EVAL_EPISODES,
                     scenario_root=str(SCEN_ROOT), actual_scenario_cfg=b["cfg"], actual_binding_sha=b["binding_sha"],
                     expected_binding_sha=binding_sha(expected_binding(s)), binding_verified=b["binding_sha"] == binding_sha(expected_binding(s)))
        if ctrl == "mplight":
            ri = _mplight_resolve(mode, extra.get("train_seed"))
            extra.update(mplight_ckpt=str(ri["path"]), mplight_ckpt_sha256=_sha256(ri["path"]), mplight_round=ri["round"], mplight_ckpt_origin=ri["origin"], mplight_ckpt_source_mode=ri["source_mode"])
        else: extra.update(mplight_ckpt=None, mplight_ckpt_sha256=None, mplight_round=None, mplight_ckpt_origin=None, mplight_ckpt_source_mode=None)
        if ctrl != "mplight" or ri["origin"] != "auto_selected": extra["train_seed"] = None      # J05: baseline cố định / checkpoint thủ công => không có train seed
        else: extra["train_seed"] = ri["train_seed"]
        extra["bench_tripinfo"] = BENCH_TRIPINFO
        meta = dict(scenario_id=f"{SPLIT}/{mode}", scenario_sha=s["sha"], scenario_fingerprint=s["fingerprint"]["overall"],
                    scenario_fingerprint_components=s["fingerprint"]["components"], controller=ctrl, seed=SEED, mode=mode, split=SPLIT,
                    scale=SCALE, run_counts=RUN_COUNTS, source_scenario_cfg=str(s["cfg"]), results_path=str(results_json),
                    active=ACTIVE["label"], run_id=RUN_ID, **(extra or {}))
        (rd / "meta.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False), encoding="utf-8")

    def _check_episodes(res, what):
        _seeds = list(range(SEED, SEED + EVAL_EPISODES))
        assert isinstance(res, list) and len(res) == EVAL_EPISODES, f"{what}: mong {EVAL_EPISODES} episode, thấy {len(res) if isinstance(res, list) else res!r}"
        assert [r.get("seed") for r in res] == _seeds and [r.get("episode") for r in res] == list(range(EVAL_EPISODES)), f"{what}: seed/episode {[(r.get('episode'), r.get('seed')) for r in res]} != {_seeds}"

    def run_builtin(ctrl, mode, train_seed=None):
        _tag = f"{ctrl}_t{train_seed}_seed{SEED}" if (ctrl == "mplight" and train_seed is not None) else f"{ctrl}_seed{SEED}"      # J05: mỗi trial MPLight 1 thư mục
        rd = OUT_DIR / "bench" / mode / _tag; assert not rd.exists(), f"{rd} đã có — không ghi đè"; rd.mkdir(parents=True)
        a = ["run_tsc.py", "eval", "--algo", ctrl, "--mode", mode, "--split", SPLIT, "--seed", SEED, "--run-counts", RUN_COUNTS, "--episodes", EVAL_EPISODES, "--out", rd,
             "--scenario-root", SCEN_ROOT, "--expect-binding", EXPECT_BINDING_PATH]      # INT-01/02: root + binding tường minh
        if ctrl == "mplight": ckpt = mplight_ckpt(mode, train_seed); a += ["--model", ckpt]
        if BENCH_TRIPINFO: a += ["--tripinfo-dir", rd / "tripinfo"]      # J05: tripinfo mỗi episode (REPO tsc/tripinfo.py); metric gộp vào eval json
        txt = pyscript(*a, log=rd / "run.log")
        out = sorted(rd.glob("eval_*.json")); assert len(out) == 1, f"mong 1 file eval_*.json trong {rd}, thấy {out}"
        _res = json.load(open(out[0], encoding="utf-8")); _check_episodes(_res, f"{mode}/{ctrl}")
        if BENCH_TRIPINFO:
            _miss = [c for c in J05_TI_COLS if any(c not in r for r in _res)]; assert not _miss, f"{mode}/{ctrl}: eval json thiếu metric tripinfo {_miss} (run_tsc trong STAGE cũ, không có --tripinfo-dir?)"
            _tix = sorted((rd / "tripinfo").glob("tripinfo_*.xml")); assert len(_tix) == EVAL_EPISODES, f"{mode}/{ctrl}: mong {EVAL_EPISODES} file tripinfo, thấy {len(_tix)}"
            _bad = [r["episode"] for r in _res if r["n_trips_arrived"] != r["arrived"]]
            if _bad: print(f"  LƯU Ý {mode}/{ctrl}: tripinfo arrived != env arrived ở episode {_bad} (xem run.log)")
        else: assert not any("delay" in r for r in _res), f"{mode}/{ctrl}: BENCH_TRIPINFO=False nhưng eval json có metric tripinfo"
        mb = re.search(r"backend=(\S+)", txt); backend = mb.group(1) if mb else NA
        BACKENDS[f"{mode}/{ctrl}"] = backend
        bnd = sorted(rd.glob("binding_*.json")); assert len(bnd) == 1, f"run_tsc không ghi binding_*.json trong {rd} (source STAGE cũ?) — không nhận metrics"
        b = verify_binding(json.load(open(bnd[0], encoding="utf-8")), BENCH_SCEN[mode], f"{mode}/{ctrl}")
        assert (b["algo"], b["seed"], b["run_counts"], b.get("episodes")) == (ctrl, SEED, RUN_COUNTS, EVAL_EPISODES), b
        assert bool(b.get("tripinfo_dir")) == BENCH_TRIPINFO, f"{mode}/{ctrl}: binding tripinfo_dir={b.get('tripinfo_dir')!r} không khớp BENCH_TRIPINFO={BENCH_TRIPINFO}"
        if ctrl == "mplight": assert b.get("model_sha256") == _sha256(ckpt), f"{mode}/mplight: checkpoint run_tsc load khác checkpoint đã chọn"
        write_meta(rd, ctrl, mode, out[0], dict(backend=backend, actual_binding=b, train_seed=train_seed)); return rd
    def validate_actions(a, junctions):
        """Hợp đồng của SumoTSCEnv.step (tsc/env.py:154): actions[j.id] -> chỉ số phase NGUYÊN trong [0, j.num_phases).
        Chỉ kiểm tra; KHÔNG ép kiểu, KHÔNG thay phase sai bằng mặc định, KHÔNG thêm safety policy."""
        if a is None: raise ValueError("agent trả None (None chỉ hợp lệ với FixedTimeAgent/control=False)")
        if not isinstance(a, Mapping): raise TypeError(f"agent.act phải trả dict {{junction_id: phase_index}}, nhận {type(a).__name__}")
        for j in junctions:
            if j.id not in a: raise ValueError(f"thiếu action cho junction {j.id}")
            v = a[j.id]
            if isinstance(v, (bool, np.bool_)) or not isinstance(v, numbers.Integral):
                raise TypeError(f"action của junction {j.id} phải là số nguyên (phase index), nhận {v!r} ({type(v).__name__}); không chấp nhận bool/float/chuỗi")
            if not 0 <= int(v) < j.num_phases:
                raise ValueError(f"action của junction {j.id} = {int(v)} ngoài tập pha hợp lệ 0..{j.num_phases - 1}")
        return a

    def run_user(mode):
        sys.path.insert(0, str(STAGE))
        for p in USER_AGENT_PATHS: sys.path.insert(0, str(p))
        import run_tsc                                   # từ STAGE (source gốc, không sửa)
        from tsc.tripinfo import parse_tripinfo, tripinfo_sumo_args      # J05: cùng helper REPO mà run_tsc eval dùng
        assert Path(run_tsc.__file__).resolve().parent == STAGE.resolve(), run_tsc.__file__
        mod = importlib.import_module(USER_AGENT_MODULE); factory = getattr(mod, USER_AGENT_FACTORY)
        junctions = run_tsc.load_junctions(run_tsc.SUMO_CONF["PHASE_MODE"])
        agent = factory(junctions)
        if not callable(getattr(agent, "act", None)): raise TypeError(f"{USER_AGENT_MODULE}.{USER_AGENT_FACTORY}(junctions) phải trả object có act(obs, explore=False)")
        class Checked:                                    # chỉ kiểm hợp đồng action; không thay đổi hành vi
            def act(self, obs, explore=False): return validate_actions(agent.act(obs, explore=False), junctions)
        rd = OUT_DIR / "bench" / mode / f"{USER_AGENT_NAME}_seed{SEED}"; assert not rd.exists(), f"{rd} đã có — không ghi đè"; rd.mkdir(parents=True)
        ns = argparse.Namespace(mode=mode, split=SPLIT, seed=SEED, gui=False, traci=False, run_counts=RUN_COUNTS,
                                scenario_root=SCEN_ROOT, expect_binding=EXPECT_BINDING_PATH)      # INT-01/02: cùng interface với CLI
        results, b0 = [], None
        for k in range(EVAL_EPISODES):      # J04: run_tsc.run_episode đóng env => env mới mỗi episode (như run_tsc eval: seed SEED+k)
            env = run_tsc.make_env(ns, junctions, control=True)
            if BENCH_TRIPINFO:      # J05: như run_tsc.cmd_eval — env mới mỗi episode => extra_sumo_args = base + tripinfo args
                (rd / "tripinfo").mkdir(parents=True, exist_ok=True)
                _ti = rd / "tripinfo" / f"tripinfo_{USER_AGENT_NAME}_{SPLIT}_{mode}_ep{k}.xml"; env.extra_sumo_args = list(env.extra_sumo_args) + tripinfo_sumo_args(_ti)
            if not hasattr(env, "binding"): raise RuntimeError("run_tsc trong STAGE không ghi scenario binding — không nhận metrics")
            b = dict(verify_binding(env.binding, BENCH_SCEN[mode], f"{mode}/{USER_AGENT_NAME} ep{k}"), algo=USER_AGENT_NAME, seed=SEED, episodes=EVAL_EPISODES, run_counts=RUN_COUNTS)
            if b0 is None:
                b0 = b; (rd / "binding.json").write_text(json.dumps(b0, indent=1), encoding="utf-8")
            else: assert b["binding_sha"] == b0["binding_sha"], f"{mode}/{USER_AGENT_NAME}: binding đổi giữa các episode"
            t0 = time.time(); m = run_tsc.run_episode(env, Checked(), explore=False, seed=SEED + k)
            if BENCH_TRIPINFO:
                m.update(parse_tripinfo(_ti), tripinfo_file=str(_ti))
                if m["n_trips_arrived"] != m["arrived"]: print(f"  LƯU Ý: tripinfo arrived {m['n_trips_arrived']} != env arrived {m['arrived']} (episode {k})")
            m.update(episode=k, seed=SEED + k, wall_s=round(time.time() - t0, 1)); results.append(m)
            print(f"[{USER_AGENT_NAME}] episode {k}: {run_tsc.fmt(m)}")
        out = rd / f"eval_{USER_AGENT_NAME}_{SPLIT}_{mode}_{time.strftime('%Y%m%d_%H%M%S')}.json"
        out.write_text(json.dumps(results, indent=2), encoding="utf-8")
        _check_episodes(results, f"{mode}/{USER_AGENT_NAME}")
        BACKENDS[f"{mode}/{USER_AGENT_NAME}"] = env.backend
        write_meta(rd, USER_AGENT_NAME, mode, out, dict(backend=env.backend, actual_binding=b0, train_seed=None))
        return rd
else:
    print(f"SKIPPED (RUN_MODE={RUN_MODE}): chọn controller")

In [ ]:
if DOWNSTREAM:
    # 08b. (J04: mặc định RUN_TRAIN=True; J05: mỗi mode × MPLIGHT_TRAIN_SEEDS) huấn luyện MPLight bằng run_tsc.py train rồi TỰ CHỌN checkpoint theo val (mplight_selection.json); chạy TRƯỚC cell 08c nên eval dùng checkpoint đã chọn.
    # run_tsc.py train cần: scenarios split=train (episode học) + split=val CÙNG --mode (greedy test mỗi --test-every vòng) — do cell 06 dựng bằng build_demand.py.
    def _select_mplight_ckpt(tm, rdir, out_path):
        """J04b/J05: logic ở REPO tsc/benchmark.py::select_mplight_checkpoint (progress.csv -> hàng val tốt nhất; KHÔNG dùng split test); ghi train/<mode>/seed<s>/mplight_selection.json."""
        if str(STAGE) not in sys.path: sys.path.insert(0, str(STAGE))
        from tsc.benchmark import select_mplight_checkpoint
        try: return select_mplight_checkpoint(rdir, sumo_seed=SEED, mode=tm, out_path=out_path)
        except (RuntimeError, FileNotFoundError) as _e: raise RuntimeError(f"{tm}: chọn checkpoint thất bại (TRAIN_TEST_EVERY={TRAIN_TEST_EVERY}): {_e}") from _e
    if RUN_TRAIN:
        with stage("training"):
            assert "TRAIN_SCEN_BY_MODE" in globals(), "chưa có scenarios train/val: chạy cell 06/07 với RUN_TRAIN=True"
            TRAIN_OUT = OUT_DIR / "train"
            assert not _files_under(TRAIN_OUT), f"{TRAIN_OUT} đã có file — không ghi đè"
            TRAIN_RUN_DIRS, TRAIN_BINDING, TRAIN_SUMMARY = {}, {}, {}
            TRAIN_RUN_DIRS_TRIALS, TRAIN_BINDING_TRIALS, LEARNING_CURVE_FILES = {}, {}, {}      # J05: [mode][train_seed]
            MPLIGHT_SELECTED_TRIALS, MPLIGHT_SELECTION_TRIALS = {}, {}
            for tm in MPLIGHT_TRAIN_MODES:      # J04: train lần lượt từng mode; J05: mỗi (mode, train seed) = 1 run_tsc.py train riêng (train split + val split cùng mode)
                ts = TRAIN_SCEN_BY_MODE[tm]
                for sp in ("train", "val"):
                    s = ts[sp]; assert s["split"] == sp and s["mode"] == tm, s
                    require(s["cfg"], f"osm.sumocfg {sp}/{tm}"); assert_frozen(s, f"trước training {tm}")
                TRAIN_RUN_DIRS_TRIALS[tm], TRAIN_BINDING_TRIALS[tm], MPLIGHT_SELECTED_TRIALS[tm], MPLIGHT_SELECTION_TRIALS[tm], _trial_sum = {}, {}, {}, {}, {}
                for tseed in MPLIGHT_TRAIN_SEEDS_EFF:
                    tout = TRAIN_OUT / tm / f"seed{tseed}"; assert not _files_under(tout), f"{tout} đã có file — không ghi đè"
                    t0 = time.time()
                    pyscript("run_tsc.py", "train", "--algo", "mplight", "--mode", tm, "--split", "train", "--seed", tseed, "--run-counts", RUN_COUNTS,
                             "--rounds", TRAIN_ROUNDS, "--generators", TRAIN_GENERATORS, "--test-every", TRAIN_TEST_EVERY, "--out", tout,
                             "--scenario-root", SCEN_ROOT, "--expect-binding", EXPECT_BINDING_PATH, "--device", "cuda" if (ON_KAGGLE and globals().get("gpu")) else "cpu",
                             log=OUT_DIR / "logs" / f"train_{tm}_seed{tseed}.log")
                    for sp in ("train", "val"): assert_frozen(ts[sp], f"sau training {tm} seed {tseed}")
                    runs = sorted(tout.glob("mplight_*")); assert len(runs) == 1, f"mong 1 thư mục run trong {tout}, thấy {runs}"
                    rdir = runs[0]; ckpts = sorted((rdir / "models").glob("round_*.pt"))
                    assert len(ckpts) == TRAIN_ROUNDS and (rdir / "progress.csv").is_file() and (rdir / "config.json").is_file(), (rdir, len(ckpts))
                    tb = json.load(open(require(rdir / "scenario_binding.json", f"scenario_binding.json (training {tm} seed {tseed})"), encoding="utf-8"))
                    TRAIN_BINDING_TRIALS[tm][tseed] = {sp: verify_binding(tb[sp], ts[sp], f"training {sp}/{tm} seed {tseed}") for sp in ("train", "val")}   # INT-02
                    sel = _select_mplight_ckpt(tm, rdir, tout / "mplight_selection.json")
                    MPLIGHT_SELECTION_TRIALS[tm][tseed] = sel; MPLIGHT_SELECTED_TRIALS[tm][tseed] = Path(sel["checkpoint"]); TRAIN_RUN_DIRS_TRIALS[tm][tseed] = rdir
                    _trial_sum[tseed] = dict(run_dir=str(rdir), n_checkpoints=len(ckpts), selected_round=sel["round"], val_travel_time=sel["val_metrics"].get("test_travel_time"), wall_s=round(time.time() - t0, 1))
                    print(f"[train {tm} seed {tseed}] {len(ckpts)} checkpoint | chọn round {sel['round']} (val ATT {sel['val_metrics'].get('test_travel_time')}, {sel['n_val_points']} điểm val) | {sel['checkpoint']}")
                _f = MPLIGHT_TRAIN_SEEDS_EFF[0]      # tương thích các cell cũ / cell 01b: trial ĐẦU TIÊN
                MPLIGHT_SELECTION[tm] = MPLIGHT_SELECTION_TRIALS[tm][_f]; MPLIGHT_SELECTED[tm] = MPLIGHT_SELECTED_TRIALS[tm][_f]
                TRAIN_RUN_DIRS[tm] = TRAIN_RUN_DIRS_TRIALS[tm][_f]; TRAIN_BINDING[tm] = TRAIN_BINDING_TRIALS[tm][_f]
                TRAIN_SUMMARY[tm] = dict(_trial_sum[_f], trials=_trial_sum)
                if LEARNING_CURVES:      # J05: vẽ ở REPO (tsc/benchmark.py::plot_learning_curves); thiếu matplotlib => chỉ cảnh báo
                    try:
                        from tsc.benchmark import plot_learning_curves
                        _png = plot_learning_curves({f"seed{k}": TRAIN_RUN_DIRS_TRIALS[tm][k] / "progress.csv" for k in MPLIGHT_TRAIN_SEEDS_EFF}, TRAIN_OUT / tm / "learning_curve.png", title=f"MPLight {tm}")
                        LEARNING_CURVE_FILES[tm] = str(_png); print(f"[train {tm}] learning curve: {_png}")
                    except ImportError as _e: print(f"CẢNH BÁO: không vẽ learning curve ({_e}); bỏ qua (run vẫn hợp lệ)")
            TRAIN_RUN_DIR = TRAIN_RUN_DIRS[TRAIN_MODE]      # tương thích cell 01b (mode đầu tiên, trial đầu tiên)
            STATUS["training"] = "done (" + "; ".join(f"{m}: {len(MPLIGHT_TRAIN_SEEDS_EFF)} trial × {TRAIN_ROUNDS} ckpt, chọn round " + "/".join(str(MPLIGHT_SELECTION_TRIALS[m][k]["round"]) for k in MPLIGHT_TRAIN_SEEDS_EFF) for m in MPLIGHT_TRAIN_MODES) + ", binding verified)"
        print("xong training; checkpoint đã chọn theo val được dùng tự động ở cell 08c:", {m: {k: str(p) for k, p in d.items()} for m, d in MPLIGHT_SELECTED_TRIALS.items()})
    else:
        print("skip training (RUN_TRAIN=False)")
else:
    print(f"SKIPPED (RUN_MODE={RUN_MODE}): training MPLight")

In [ ]:
if DOWNSTREAM:
    # 08c. Chạy SUMO headless: mọi controller trên CÙNG scenario/seed. Fingerprint NỘI DUNG input SUMO được kiểm TRƯỚC và SAU mỗi run.
    RUN_DIRS = []
    EXPECTED_RUNS = sum(_n_runs(_c, _m) for _m in BENCH_MODES for _c in PLAN_RUNS)      # J04: camera modes + stress (nếu bật); J05: MPLight = 1 run / trial
    with stage("controllers"):
        for mode in BENCH_MODES:
            for ctrl in PLAN_RUNS:
                for tseed in (_mplight_trials(mode) if ctrl == "mplight" else [None]):      # J05: mỗi trial MPLight 1 lượt eval
                    assert_frozen(BENCH_SCEN[mode], f"trước run {ctrl}")
                    print(f"\n=== {mode} | {ctrl}{'' if tseed is None else f' (train seed {tseed})'} | seed {SEED}..{SEED + EVAL_EPISODES - 1} ({EVAL_EPISODES} episode) | split {SPLIT} | {RUN_COUNTS}s | fingerprint {BENCH_SCEN[mode]['sha']} ===")
                    rd = run_user(mode) if ctrl == USER_AGENT_NAME and USER_AGENT_MODULE else run_builtin(ctrl, mode, tseed)
                    assert_frozen(BENCH_SCEN[mode], f"sau run {ctrl}")
                    RUN_DIRS.append(rd)
        if len(RUN_DIRS) != EXPECTED_RUNS: raise RuntimeError(f"mong {EXPECTED_RUNS} run, có {len(RUN_DIRS)}")
        STATUS["controllers"] = f"done ({len(RUN_DIRS)}/{EXPECTED_RUNS} run)"
else:
    STATUS["controllers"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): SUMO controller runs")

## 09 — Benchmark results
Chỉ dùng metrics mà `SumoTSCEnv.metrics()` / `run_tsc.run_episode` thực sự trả về (`tsc/env.py:253-270`, `run_tsc.py:116-117`): `travel_time` (ATT kiểu CityFlow, xe chưa tới tính tới hết episode), `travel_time_arrived`, `arrived`, `departed`, `in_network_at_end`, `pending_insertion`, `teleports`, `avg_queue`, `mean_reward`, `wall_s`. Không có P95/fairness/safety trong source nên không báo cáo.

In [ ]:
if DOWNSTREAM:
    COLS_BASE = ["travel_time", "travel_time_arrived", "arrived", "departed", "in_network_at_end", "pending_insertion", "teleports", "avg_queue", "mean_reward", "wall_s"]
    COLS = COLS_BASE + (J05_TI_COLS if BENCH_TRIPINFO else [])      # J05: + delay/waiting/trip_duration/n_trips_* (tripinfo)
    with stage("benchmark"):
        _META_KEYS = ("scenario_id", "scenario_sha", "scenario_fingerprint", "controller", "mode", "split", "scale", "source_scenario_cfg", "results_path",
                      "demand_mode", "scenario_root", "actual_scenario_cfg", "actual_binding_sha", "expected_binding_sha", "binding_verified")
        _META_OPT = ("demand_source", "mplight_round", "mplight_ckpt", "mplight_ckpt_sha256")
        _exp_seeds = list(range(SEED, SEED + EVAL_EPISODES))
        runs, ep_rows = [], []
        for meta_p in sorted((OUT_DIR / "bench").rglob("meta.json")):
            meta = json.load(open(meta_p, encoding="utf-8")); res = json.load(open(require(meta["results_path"], "results_path"), encoding="utf-8"))
            assert isinstance(res, list) and len(res) == EVAL_EPISODES, f"{meta_p}: mong {EVAL_EPISODES} episode, thấy {len(res) if isinstance(res, list) else res!r}"
            assert [r["seed"] for r in res] == _exp_seeds and [r["episode"] for r in res] == list(range(EVAL_EPISODES)), f"{meta_p}: seed/episode không khớp {_exp_seeds}"
            assert meta.get("episodes") == EVAL_EPISODES, f"{meta_p}: episodes {meta.get('episodes')} != {EVAL_EPISODES}"
            assert meta.get("bench_tripinfo") == BENCH_TRIPINFO, f"{meta_p}: bench_tripinfo {meta.get('bench_tripinfo')} != BENCH_TRIPINFO={BENCH_TRIPINFO}"
            base = {**{k: meta[k] for k in _META_KEYS}, **{k: meta.get(k) for k in _META_OPT}, "base_seed": meta["seed"], "train_seed": meta.get("train_seed")}      # J05: train_seed (None = baseline cố định / ckpt thủ công)
            runs.append(base)
            for r in res:
                _miss = [c for c in COLS if c not in r]; assert not _miss, f"{meta_p} episode {r.get('episode')}: thiếu metric {_miss}"      # J05: tripinfo metric bắt buộc khi BENCH_TRIPINFO
                ep_rows.append({**base, "episode": r["episode"], "seed": r["seed"], **{c: r[c] for c in COLS}})      # J04: ĐỌC MỌI episode (không chỉ [0])
        if not runs: raise RuntimeError("Chưa có kết quả benchmark — cell 08c chưa chạy")
        if len(runs) != EXPECTED_RUNS: raise RuntimeError(f"mong {EXPECTED_RUNS} run benchmark, thấy {len(runs)}")
        # công bằng: trong mỗi mode, mọi controller cùng scenario_fingerprint (nội dung input) / seed / scale / split / cùng DANH SÁCH seed episode
        for mode in BENCH_MODES:
            g = [r for r in runs if r["mode"] == mode]
            assert len(g) == sum(_n_runs(c, mode) for c in PLAN_RUNS), f"{mode}: thiếu/thừa run ({[(r['controller'], r['train_seed']) for r in g]})"      # J05: MPLight có thể nhiều trial
            for c in PLAN_RUNS:      # mỗi controller đúng tập train seed mong đợi ([None] cho baseline / ckpt thủ công)
                _want = _mplight_trials(mode) if c == "mplight" else [None]
                assert sorted(map(str, (r["train_seed"] for r in g if r["controller"] == c))) == sorted(map(str, _want)), f"{mode}/{c}: train seed {[r['train_seed'] for r in g if r['controller'] == c]} != {_want}"
            _edm = BENCH_SCEN[mode].get("expected_demand_mode", DEMAND_MODE)      # stress: baseline_5050 (không scenario.json)
            bad = [r["controller"] for r in g if not (r["binding_verified"] is True and r["actual_binding_sha"] == r["expected_binding_sha"] == binding_sha(expected_binding(BENCH_SCEN[mode])) and r["demand_mode"] == _edm)]
            if bad: raise RuntimeError(f"{mode}: controller {bad} KHÔNG chạy scenario đã đóng băng (binding) — không gộp kết quả")
            assert len({(r["actual_binding_sha"], r["scenario_fingerprint"], r["base_seed"], r["scale"], r["split"], r["demand_mode"]) for r in g}) == 1, f"{mode}: controller khác scenario/seed!"
            _sd = {(r["controller"], r["train_seed"]): [e["seed"] for e in ep_rows if e["mode"] == mode and e["controller"] == r["controller"] and e["train_seed"] == r["train_seed"]] for r in g}
            assert all(v == _exp_seeds for v in _sd.values()), f"{mode}: run khác danh sách seed episode {_sd}"
        print(f"FAIR CHECK ok: mọi controller trong mỗi mode ({BENCH_MODES}) load CÙNG cfg/net/routes (binding thực tế), cùng seed {_exp_seeds}/scale/split/demand\n")
        # tổng hợp theo (mode, controller): GỘP mọi trial (n = n_trials × episode), mean, std MẪU (ddof=1); n=1 => std = NaN (không đủ mẫu)
        import math, statistics
        _order = lambda r: (BENCH_MODES.index(r["mode"]), PLAN_RUNS.index(r["controller"]) if r["controller"] in PLAN_RUNS else len(PLAN_RUNS), r["controller"])
        if str(STAGE) not in sys.path: sys.path.insert(0, str(STAGE))
        from tsc.benchmark import aggregate_episodes, write_rows, format_mean_std_table, improvement_table      # J04b/J05: logic ở REPO tsc/benchmark.py
        _runs_sorted = sorted(runs, key=lambda r: (_order(r), -1 if r["train_seed"] is None else r["train_seed"]))
        _pooled, _seen = [], []
        for r in _runs_sorted:      # 1 dòng gốc / (mode, controller); trường khác nhau giữa trial (results_path, mplight_round/ckpt/sha) nối bằng ";"
            if (r["mode"], r["controller"]) in _seen: continue
            _seen.append((r["mode"], r["controller"]))
            rs = [x for x in _runs_sorted if x["mode"] == r["mode"] and x["controller"] == r["controller"]]
            _row = {k: (rs[0][k] if len({str(x[k]) for x in rs}) == 1 else ";".join(str(x[k]) for x in rs)) for k in rs[0] if k != "train_seed"}
            _row["n_trials"] = len(rs); _row["train_seeds"] = ";".join("-" if x["train_seed"] is None else str(x["train_seed"]) for x in rs)
            _pooled.append(_row)
        agg = aggregate_episodes(_pooled, ep_rows, COLS)
        _nl = f"gộp trial: n = n_trials × {EVAL_EPISODES} episode (MPLight n_trials = {len(MPLIGHT_TRAIN_SEEDS_EFF)}; baseline cố định n_trials = 1)"
        print(format_mean_std_table(agg, COLS_BASE, n_label=_nl))
        if BENCH_TRIPINFO: print("\n" + format_mean_std_table(agg, J05_TI_COLS, n_label="tripinfo (delay = timeLoss; *_all gồm xe unfinished/undeparted)"))
        write_rows(ep_rows, OUT_DIR / "bench" / "results_episodes.csv", OUT_DIR / "bench" / "results_episodes.json", nan_to_none=False)
        write_rows(agg, OUT_DIR / "bench" / "results.csv", OUT_DIR / "bench" / "results.json")      # bảng TỔNG HỢP (cell 09 kiểm file này)
        TRIALS_TABLE_WRITTEN = any(r["n_trials"] > 1 for r in agg)
        if TRIALS_TABLE_WRITTEN:      # J05: bảng theo (mode, controller, train_seed) — chỉ khi có > 1 trial
            agg_t = aggregate_episodes(_runs_sorted, ep_rows, COLS, key=("mode", "controller", "train_seed"))
            write_rows(agg_t, OUT_DIR / "bench" / "results_trials.csv", OUT_DIR / "bench" / "results_trials.json")
            print("\nTheo trial (mean theo train seed):"); print(f"{'mode':11s}{'controller':12s}{'seed':>5s}{'n':>3s}" + "".join(f"{m[:11]:>13s}" for m in ("travel_time", "arrived", "avg_queue") + (("delay", "delay_all") if BENCH_TRIPINFO else ())))
            for r in agg_t:
                if any(x["controller"] == r["controller"] and x["mode"] == r["mode"] and x["train_seed"] != r["train_seed"] for x in agg_t):
                    print(f"{r['mode']:11s}{r['controller']:12s}{str(r['train_seed']):>5s}{r['n']:3d}" + "".join(f"{r[m]:13.2f}" for m in ("travel_time", "arrived", "avg_queue") + (("delay", "delay_all") if BENCH_TRIPINFO else ())))
        IMPROVEMENT_WRITTEN = REPORT_BASELINE in PLAN_RUNS
        if IMPROVEMENT_WRITTEN:      # J05: % so với REPORT_BASELINE (mean đã gộp trial); dương = tốt hơn mốc (lower-is-better: (base-x)/base; arrived: (x-base)/base)
            impr = improvement_table(agg, REPORT_BASELINE)
            write_rows(impr, OUT_DIR / "bench" / "improvement.csv", OUT_DIR / "bench" / "improvement.json")
            _im = [m for m in ("travel_time", "delay", "delay_all", "waiting_time", "arrived", "avg_queue", "teleports") if m in COLS]
            print(f"\nCải thiện so với {REPORT_BASELINE} (giá trị trung bình, % trong ngoặc: dương = tốt hơn):")
            print(f"{'mode':11s}{'controller':12s}" + "".join(f"{m[:19]:>20s}" for m in _im))
            for r in impr: print(f"{r['mode']:11s}{r['controller']:12s}" + "".join(f"{r[m]:11.2f} ({r[m + '_impr_pct']:+6.1f}%)" for m in _im))
        else: print(f"\nJ05: REPORT_BASELINE={REPORT_BASELINE!r} không có trong controller đã chạy {PLAN_RUNS} => bỏ qua improvement.csv")
        print("\nlưu:", OUT_DIR / "bench" / "results.csv", "(tổng hợp) |", OUT_DIR / "bench" / "results_episodes.csv", f"({len(ep_rows)} dòng)", "| improvement.csv" if IMPROVEMENT_WRITTEN else "", "| results_trials.csv" if TRIALS_TABLE_WRITTEN else "")
        # thông tin: đối chiếu với baseline đã commit trong repo (chỉ khi cùng split=test, seed 0, 3600 s; so với episode seed SEED; chỉ in khi file baseline tồn tại)
        if SPLIT == "test" and SEED == 0 and RUN_COUNTS == 3600 and not IMAGES:   # J02E: so với baseline repo chỉ khi repo_cache
            print("\n-- so với results/baselines của repo (ATT, episode đầu seed 0):")
            _first_ts = MPLIGHT_TRAIN_SEEDS_EFF[0]
            for e in ep_rows:
                if e["episode"] != 0 or e["train_seed"] not in (None, _first_ts): continue      # J05: trial đầu tiên
                p = REPO_DIR / "results/baselines" / f"test_{e['controller']}_{e['mode']}.json"
                if p.is_file(): print(f"  {e['mode']:11s}{e['controller']:12s} mới {e['travel_time']:8.2f} | repo {json.load(open(p))[0]['travel_time']:8.2f}")
        STATUS["benchmark"] = f"done ({len(agg)} dòng tổng hợp, {len(runs)} run, {len(ep_rows)} episode)"
else:
    STATUS["benchmark"] = f"SKIPPED ({RUN_MODE}: by design)"; print(f"SKIPPED (RUN_MODE={RUN_MODE}): benchmark results")

## Benchmark protocol (J05)
Cách đọc `bench/results.csv`. Số liệu/`file:line` = REPO hiện tại; Chỉ lấy ý tưởng từ paper, không copy code.
- **Horizon / demand:** `RUN_COUNTS` = 3600 s/episode (`tsc/config.py:35`); `SCALE` = 0.25 (`build_demand.py:50`); seed demand theo split train 1001 / val 2002 / test 3003 (`build_demand.py:48`), cố định ⇒ mọi episode/controller chạy CÙNG scenario; seed SUMO eval = `SEED` … `SEED+EVAL_EPISODES-1` (chỉ đổi tính ngẫu nhiên SUMO, không đổi demand).
- **Điều khiển:** `MIN_ACTION_TIME` 10 s (gồm yellow), `YELLOW_TIME` 5 s, `ALL_RED_TIME` 0 s (`tsc/config.py:31-33`). Teleport: `time-to-teleport` 300 s trong `sumo/scenarios/*/*/osm.sumocfg:17`, giữ nguyên (đọc qua `tsc/env.py:33-57`).
- **FixedTime** = chương trình đèn có sẵn của net, không can thiệp (`FixedTimeAgent` trả None + `control=False`: `tsc/agents.py:15-21`, `tsc/env.py:128,155`). **MaxPressure** = mỗi 10 s chọn phase có tổng pressure lớn nhất (`tsc/agents.py:24-42`). **MPLight** = train (split train) → chọn checkpoint theo travel time trên split val (`tsc/benchmark.py:26-58`, không dùng test) → eval trên test; `MPLIGHT_TRAIN_SEEDS` = số trial train độc lập (mặc định 1; báo cáo nên 5).
- **Metric của env** (`tsc/env.py:253-270`): `travel_time` (kiểu CityFlow) = trung bình (arrive hoặc cuối episode − depart) trên mọi xe đã depart ⇒ XE CHƯA TỚI ĐÍCH vẫn tính tới hết 3600 s; `travel_time_arrived` chỉ xe đã tới; `arrived` = throughput trong horizon; `departed`, `in_network_at_end`, `pending_insertion` (xe SUMO chưa chèn được), `teleports` (`env.py:199`); `avg_queue` = tổng xe halting trên các cạnh vào 6 nút, trung bình theo giây (`env.py:185`).
- **Metric tripinfo** (`BENCH_TRIPINFO`, `tsc/tripinfo.py`): `delay` = timeLoss trung bình của xe đã arrived; `delay_with_depart` = timeLoss + departDelay (chờ được chèn vào mạng); `waiting_time`, `trip_duration` (duration) cũng chỉ xe arrived; `delay_all`, `delay_with_depart_all`, `waiting_time_all` gồm cả xe unfinished (giá trị MỘT PHẦN tới cuối episode) và undeparted; `n_trips_unfinished/undeparted` = số xe đó.
- **Gộp:** `results.csv` = mean và std mẫu (ddof=1) gộp mọi trial × episode (`n = n_trials × EVAL_EPISODES`); std trộn dao động seed SUMO và trial. Từng trial: `results_trials.csv` (chỉ khi > 1 trial). `improvement.csv`: % so `REPORT_BASELINE` trên mean đã gộp: lower-is-better (travel_time, delay, waiting, queue, teleports…) = (base − x)/base × 100, `arrived` = (x − base)/base × 100; DƯƠNG = tốt hơn mốc.
- **Survivor bias:** metric chỉ-arrived (`travel_time_arrived`, `delay`, `waiting_time`, `trip_duration`) có lợi cho controller để kẹt xe lại trong mạng (xe kẹt không được tính). Luôn đọc cùng `arrived`, `travel_time`, `*_all`, `n_trips_unfinished/undeparted`, `teleports`.
- **Khác RESCO** (theo paper RESCO trong `paper code/`, đã đọc ở audit J05; không copy code GPL): metric từ tripinfo (timeLoss, duration, waitingTime) · yellow 3 s + 2 s red · teleport tắt (-1) · 5 trial · libsumo. VietTSC: yellow 5 s, all-red 0, teleport 300 s, 1 trial mặc định, libsumo nếu cài (`tsc/env.py:75-84`).
- **Khác LibSignal** (theo paper LibSignal trong `paper code/`, đã đọc ở audit J05): travel time chỉ xe đã arrived, queue, delay, throughput · FixedTime `t_fixed` 30 s · MaxPressure `t_min` 10 s · std qua các run. VietTSC: FixedTime = signal plan của net (không phải chu kỳ đều 30 s), `travel_time` tính cả xe chưa tới.

## 10 — Summary & E2E status

In [ ]:
# J02E fix5
# 09. Nghiệm thu: chỉ PASS khi ĐÚNG các stage/artifact BẮT BUỘC của RUN_MODE (của chính RUN_ID này) đã thành công.
print("RUN_MODE:", RUN_MODE, "| BACKEND:", DETECTOR_BACKEND, "| RUN_ID:", RUN_ID, "|", ACTIVE["label"])
print("\n-- stage status")
for k, v in STATUS.items(): print(f"  {k:19s} {v}")
OK = lambda v: str(v).startswith(("done", "loaded")) and "BLOCKED" not in str(v)      # J02E fix2: BLOCKED không bao giờ là done
T0 = _datetime.fromisoformat(T_START).timestamp() - 2
def _fresh(p):          # file có thật, không rỗng, nằm trong OUT_DIR và được tạo SAU khi run này bắt đầu
    p = Path(p)
    return p.is_file() and p.stat().st_size > 0 and OUT_DIR.resolve() in p.resolve().parents and p.stat().st_mtime >= T0

REQUIRED = {"baseline_cache":   ["preflight", "stage", "calib_set", "detection", "calibration", "infer", "settings", "routes", "scenario_check", "controllers", "benchmark"],
            "full_recompute":   ["preflight", "stage", "calib_set", "detection", "calibration", "infer", "settings", "routes", "scenario_check", "controllers", "benchmark"],
            "calibration_only": ["preflight", "stage", "calib_set", "detection", "calibration", "calibration_report"],
            "image_review":     ["preflight", "stage", "input_discovery", "detection", "roi_frames"]}[RUN_MODE]      # J02E
if RUN_MODE == "baseline_cache" and str(STATUS.get("calib_set")).startswith("SKIPPED (baseline_cache"): REQUIRED = [k for k in REQUIRED if k != "calib_set"]      # J02E fix7: calibration set không dùng khi calibration từ artifact
if RUN_TRAIN: REQUIRED = REQUIRED + ["training"]            # training được yêu cầu => phải thành công, kiểm riêng
if J02 and DOWNSTREAM: REQUIRED = REQUIRED + ["directional", "directional_targets"]      # J02
if IMAGES and RUN_MODE not in ("calibration_only", "image_review"): REQUIRED = REQUIRED + ["input_discovery"]      # J02E
if HAS_CITY_DETS and not IMAGE_REVIEW: REQUIRED = REQUIRED + ["roi_frames"]      # J02E
if HYBRID: REQUIRED = REQUIRED + ["hybrid_backend"] + (["hybrid_diagnostics"] if EXPORT_HYBRID_DIAGNOSTICS else [])      # J01
problems = [f"stage '{k}' lỗi: {v}" for k, v in FAILED.items()]
problems += [f"stage bắt buộc '{k}' chưa hoàn tất: {STATUS.get(k)}" for k in REQUIRED if not OK(STATUS.get(k)) and k not in FAILED]
unchanged = tree_sha(REPO_DIR) == SRC_SNAPSHOT
print("\nSource REPO unchanged:", unchanged)
if not unchanged: problems.append("REPO bị thay đổi")

if RUN_MODE == "baseline_cache":      # artifact baseline có sẵn trong repo phải còn đó
    problems += [f"thiếu artifact baseline: {p}" for p in (ACTIVE["dets_calib"], ACTIVE["dets_city"], ACTIVE["calibration"]) if p is not None and not Path(p).is_file()]
elif IMAGE_REVIEW:      # J02E: detection city (mới, trừ khi tái dùng cache) + roi_frames (kiểm sau); không calibration
    problems += [f"artifact không mới/không có: {ACTIVE['dets_city']}"] if (not CITY_REUSE and not _fresh(ACTIVE["dets_city"])) else []
    problems += [f"thiếu dets city tái dùng: {ACTIVE['dets_city']}"] if (CITY_REUSE and not Path(ACTIVE["dets_city"]).is_file()) else []
else:                                 # detection + calibration phải MỚI (của run này), không phải cache cũ
    arts = [ACTIVE["dets_calib"], ACTIVE["calibration"]] + ([ACTIVE["dets_city"]] if (RUN_MODE == "full_recompute" and not CITY_REUSE) else [])      # J02E
    problems += [f"artifact không mới/không có: {p}" for p in arts if not _fresh(p)]
    if "recomputed" not in str(STATUS.get("detection")) or "recomputed" not in str(STATUS.get("calibration")): problems.append("detection/calibration không phải recompute")
if HYBRID and RUN_MODE != "baseline_cache":     # J01: sidecar provenance của chính run này
    problems += [f"thiếu provenance hybrid: {p}" for p in ([] if IMAGE_REVIEW else [str(ACTIVE["dets_calib"]) + ".provenance.json", ACTIVE["calibration_provenance"]]) + ([str(ACTIVE["dets_city"]) + ".provenance.json"] if (RUN_MODE == "full_recompute" or IMAGE_REVIEW) else []) if not Path(p).is_file()]   # J02E: image_review chỉ có city
if DOWNSTREAM:
    flow_arts = [FLOW_DIR / "perframe.json", FLOW_DIR / "demand_settings.json", OUT_DIR / "bench" / "results.csv", OUT_DIR / "bench" / "results.json", OUT_DIR / "bench" / "results_episodes.csv", OUT_DIR / "bench" / "results_episodes.json"]   # J04
    if globals().get("IMPROVEMENT_WRITTEN"): flow_arts += [OUT_DIR / "bench" / "improvement.csv", OUT_DIR / "bench" / "improvement.json"]      # J05
    if globals().get("TRIALS_TABLE_WRITTEN"): flow_arts += [OUT_DIR / "bench" / "results_trials.csv", OUT_DIR / "bench" / "results_trials.json"]      # J05
    if "mplight" in PLAN_RUNS and len(MPLIGHT_TRAIN_SEEDS_EFF) > 1 and any(not _mplight_is_manual(_m) for _m in BENCH_MODES) and not globals().get("TRIALS_TABLE_WRITTEN"): problems.append("J05: nhiều trial MPLight nhưng không có results_trials.csv")
    if "mplight" in PLAN_RUNS and REPORT_BASELINE in PLAN_RUNS and not globals().get("IMPROVEMENT_WRITTEN"): problems.append("J05: thiếu improvement.csv")
    if J02: flow_arts += [FLOW_DIR / "perframe_directional.json", FLOW_DIR / "demand_settings_directional.json"]   # J02
    if ADAPTIVE_BETA_EFF: flow_arts += [FLOW_DIR / "adaptive_beta_used.json"]   # J07: infer + directional cùng ghi vào FLOW_DIR; thiếu/không mới => problem
    problems += [f"artifact không mới/không có: {p}" for p in flow_arts if not _fresh(p)]
    problems += [f"thiếu scenario {SPLIT}/{m}" for m in BENCH_MODES if "BENCH_SCEN" not in globals() or m not in BENCH_SCEN or not Path(BENCH_SCEN[m]["cfg"]).is_file()]   # J04: camera modes + stress
    _bm = [json.load(open(p, encoding="utf-8")) for p in sorted((OUT_DIR / "bench").rglob("meta.json"))]      # INT-02
    problems += [f"benchmark {x.get('scenario_id')}/{x.get('controller')}: scenario binding chưa xác minh" for x in _bm
                 if not (x.get("binding_verified") is True and x.get("actual_binding_sha") == x.get("expected_binding_sha"))]
    if RUN_TRAIN:      # J04: kiểm TỪNG mode train; J05: và từng train seed (trial)
        _tb, _td, _ts = (globals().get("TRAIN_BINDING_TRIALS") or {}), (globals().get("TRAIN_RUN_DIRS_TRIALS") or {}), (globals().get("MPLIGHT_SELECTED_TRIALS") or {})
        for _m in MPLIGHT_TRAIN_MODES:
            for _s in MPLIGHT_TRAIN_SEEDS_EFF:
                _w = f"{_m} seed {_s}"
                if set((_tb.get(_m) or {}).get(_s) or {}) != {"train", "val"}: problems.append(f"training {_w}: scenario binding train/val chưa xác minh")
                _d = (_td.get(_m) or {}).get(_s)
                if not _d or len(list((Path(_d) / "models").glob("round_*.pt"))) != TRAIN_ROUNDS: problems.append(f"RUN_TRAIN=True nhưng {_w}: không đủ {TRAIN_ROUNDS} checkpoint")
                _sp = OUT_DIR / "train" / _m / f"seed{_s}" / "mplight_selection.json"
                _sel = (_ts.get(_m) or {}).get(_s)
                if not _fresh(_sp): problems.append(f"RUN_TRAIN=True nhưng thiếu mplight_selection.json cho {_w}: {_sp}")
                elif not (_sel and Path(_sel).is_file() and json.load(open(_sp, encoding="utf-8")).get("checkpoint") == str(_sel)): problems.append(f"training {_w}: checkpoint đã chọn không khớp mplight_selection.json")
            if LEARNING_CURVES and __import__("importlib.util", fromlist=["find_spec"]).find_spec("matplotlib") is not None and not _fresh(OUT_DIR / "train" / _m / "learning_curve.png"): problems.append(f"J05: matplotlib có nhưng thiếu learning_curve.png cho {_m}")      # thiếu matplotlib => không bắt buộc
    _bm_n = len(_bm); _er = globals().get("EXPECTED_RUNS")
    if _er is not None and _bm_n != _er: problems.append(f"benchmark: mong {_er} meta, thấy {_bm_n}")
    problems += [f"benchmark {x.get('scenario_id')}/{x.get('controller')}: episodes {x.get('episodes')} != EVAL_EPISODES={EVAL_EPISODES}" for x in _bm if x.get("episodes") != EVAL_EPISODES]
    try: _epj = json.load(open(OUT_DIR / "bench" / "results_episodes.json", encoding="utf-8")); _nep = len(_epj)
    except Exception: _nep = -1
    if _er is not None and _nep != _er * EVAL_EPISODES: problems.append(f"results_episodes: mong {(_er or 0) * EVAL_EPISODES} dòng, thấy {_nep}")
elif RUN_MODE == "calibration_only":      # J02E: image_review không có output calibration
    REQ6 = ["detections.jsonl.gz", "calibration.json", "calibration_run.log", "calibration_summary.csv", "calibration_per_fold.csv", "experiment_metadata.json"]
    if HYBRID: REQ6 += ["detections.jsonl.gz.provenance.json", "calibration.json.provenance.json"]      # J01
    problems += [f"thiếu output calibration: {n}" for n in REQ6 if not (CAL_RUN_DIR / n).is_file() or (CAL_RUN_DIR / n).stat().st_size == 0]
    try: _ms = json.load(open(META_PATH, encoding="utf-8"))["execution_status"]
    except Exception as e: _ms = f"không đọc được metadata ({e})"
    if _ms != "COMPLETE": problems.append(f"metadata execution_status = {_ms!r} (cần COMPLETE)")
if HAS_CITY_DETS:      # J02E: artifact roi_frames (số đếm tức thời) phải mới
    problems += [f"artifact roi_frames không mới/không có: {OUT_DIR / 'roi_frames' / n}" for n in ("counts_per_frame.csv", "summary_camera_roi.csv", "roi_frames_provenance.json") if not _fresh(OUT_DIR / "roi_frames" / n)]
problems = [p for p in problems if p]
_GT = globals().get("GROUND_TRUTH_KIND")      # J02E fix5: baseline_cache + CALIBRATION_ARTIFACT => lấy ground_truth_kind từ kết quả calibration_artifact (01c)
if _GT in (None, NA):
    try: _GT = json.load(open(OUT_DIR / "calibration" / "calibration_artifact.json", encoding="utf-8")).get("ground_truth_kind") or NA
    except Exception: _GT = NA
GROUND_TRUTH_KIND = _GT
_BLK = [str(v) for v in list(STATUS.values()) + list(FAILED.values()) if "BLOCKED: INSUFFICIENT_DATA" in str(v)]      # J02E fix2
_BLK = list(dict.fromkeys(m[m.index("BLOCKED: INSUFFICIENT_DATA"):] for m in _BLK))
if _BLK: problems = [p for p in problems if "BLOCKED: INSUFFICIENT_DATA" not in p] + _BLK

final = "COMPLETE" if not problems else (("BLOCKED: INSUFFICIENT_DATA — " + "; ".join(p.replace("BLOCKED: INSUFFICIENT_DATA — ", "") for p in _BLK)) if _BLK else "FAILED: " + "; ".join(problems))      # J02E fix2
try: write_run_metadata(final)           # status cuối + provenance đầy đủ (hash, version, scenario fingerprint, output paths)
except Exception as e: problems.append(f"không ghi được metadata cuối: {e}")
try:      # J04: bổ sung vào experiment_metadata.json (cell 01b không đổi) — cấu hình Full-mode + checkpoint đã chọn
    _g4 = globals(); _mj = json.load(open(META_PATH, encoding="utf-8"))
    _mj["j04_full_mode"] = dict(eval_episodes=EVAL_EPISODES, bench_modes=BENCH_MODES, include_stress=INCLUDE_STRESS, stress_mplight_from=STRESS_MPLIGHT_FROM if INCLUDE_STRESS else None,
                                run_train=RUN_TRAIN, train_rounds=TRAIN_ROUNDS, train_generators=TRAIN_GENERATORS, train_test_every=TRAIN_TEST_EVERY, mplight_train_modes=MPLIGHT_TRAIN_MODES,
                                mplight_ckpt_user=MPLIGHT_CKPT_USER if isinstance(MPLIGHT_CKPT_USER, (dict, str, type(None))) else str(MPLIGHT_CKPT_USER),
                                mplight_selected={m: dict(checkpoint=str(p), round=(_g4.get("MPLIGHT_SELECTION") or {}).get(m, {}).get("round"), sha256=(_g4.get("MPLIGHT_SELECTION") or {}).get(m, {}).get("checkpoint_sha256")) for m, p in (_g4.get("MPLIGHT_SELECTED") or {}).items()},
                                train_summary=_g4.get("TRAIN_SUMMARY") or None, installed_libsumo=_g4.get("LIBSUMO_AVAILABLE"),
                                tsc_benchmark_sha256=(_sha256(STAGE / "tsc/benchmark.py") if (STAGE / "tsc/benchmark.py").is_file() else None),      # J04b
                                train_scenario_fingerprints={f"{sp}/{m}": s["fingerprint"]["overall"] for m, d in (_g4.get("TRAIN_SCEN_BY_MODE") or {}).items() for sp, s in d.items()} or None)
    _mj["j05_protocol"] = dict(bench_tripinfo=BENCH_TRIPINFO, mplight_train_seeds=MPLIGHT_TRAIN_SEEDS, mplight_train_seeds_effective=MPLIGHT_TRAIN_SEEDS_EFF, report_baseline=REPORT_BASELINE, learning_curves=LEARNING_CURVES,
                               tripinfo_py_sha256=(_sha256(STAGE / "tsc/tripinfo.py") if (STAGE / "tsc/tripinfo.py").is_file() else None),
                               benchmark_py_sha256=(_sha256(STAGE / "tsc/benchmark.py") if (STAGE / "tsc/benchmark.py").is_file() else None),
                               run_tsc_py_sha256=(_sha256(STAGE / "run_tsc.py") if (STAGE / "run_tsc.py").is_file() else None),
                               improvement_csv=str(OUT_DIR / "bench" / "improvement.csv") if (OUT_DIR / "bench" / "improvement.csv").is_file() else None,
                               results_trials_csv=str(OUT_DIR / "bench" / "results_trials.csv") if (OUT_DIR / "bench" / "results_trials.csv").is_file() else None,
                               learning_curves_files=_g4.get("LEARNING_CURVE_FILES") or None,
                               mplight_selected_trials={m: {str(k): dict(checkpoint=str(p), round=((_g4.get("MPLIGHT_SELECTION_TRIALS") or {}).get(m) or {}).get(k, {}).get("round"), sha256=((_g4.get("MPLIGHT_SELECTION_TRIALS") or {}).get(m) or {}).get(k, {}).get("checkpoint_sha256")) for k, p in d.items()} for m, d in (_g4.get("MPLIGHT_SELECTED_TRIALS") or {}).items()} or None)
    _j07f = FLOW_DIR / "adaptive_beta_used.json"   # J07
    _j07u = json.load(open(_j07f, encoding="utf-8")) if _j07f.is_file() else {}   # J07
    _mj["j07_adaptive_beta"] = dict(adaptive_beta_knob=ADAPTIVE_BETA, adaptive_beta_effective=bool(ADAPTIVE_BETA_EFF), status=STATUS.get("adaptive_beta"),   # J07
                                    gamma=_j07u.get("gamma"), gamma_source=_j07u.get("gamma_source"), clip=_j07u.get("clip"),
                                    adaptive_beta_used_json=str(_j07f) if _j07f.is_file() else None, calibration=str(ACTIVE.get("calibration")))
    _tmp = META_PATH.with_name(META_PATH.name + ".tmp"); _tmp.write_text(json.dumps(_mj, indent=1, ensure_ascii=False, default=str), encoding="utf-8"); os.replace(_tmp, META_PATH)
except Exception as _e4: print("(J04: không bổ sung được metadata:", type(_e4).__name__, _e4, ")")
print("metadata:", META_PATH)
try:      # J02E fix5: khối tóm tắt không bao giờ được làm mất verdict (stage chưa chạy => thiếu biến)
    _g = globals()
    print("J02E:", CITY_SOURCE, "| index:", _g.get("CITY_INDEX"), "| roi_frames:", OUT_DIR / "roi_frames", "| ground truth:", GROUND_TRUTH_KIND, "|", STATUS.get("roi_frames"))
    if DOWNSTREAM:
        print("detections city :", ACTIVE.get("dets_city")); print("calibration     :", ACTIVE.get("calibration"))
        print("settings        :", FLOW_DIR / "demand_settings.json"); print("scenarios       :", (_g["SCEN_ROOT"] / SPLIT) if _g.get("SCEN_ROOT") else "(chưa tạo — stage routes/scenario chưa chạy)", "| binding:", _g.get("EXPECT_BINDING_PATH"))
        print("benchmark       :", OUT_DIR / "bench" / "results.csv"); print("scale/seed/split:", SCALE, SEED, SPLIT)
        print("J04 Full-mode   : EVAL_EPISODES", EVAL_EPISODES, "| BENCH_MODES", BENCH_MODES, "| CONTROLLERS chạy", _g.get("PLAN_RUNS"), "| INCLUDE_STRESS", INCLUDE_STRESS)
        print("benchmark (mỗi episode):", OUT_DIR / "bench" / "results_episodes.csv")
        for _m, _p in (_g.get("MPLIGHT_SELECTED") or {}).items(): print(f"  checkpoint MPLight đã chọn (val) [{_m}]: round {(_g.get('MPLIGHT_SELECTION') or {}).get(_m, {}).get('round')} -> {_p}")
        for _m, _d in (_g.get("MPLIGHT_SELECTED_TRIALS") or {}).items():
            if len(_d) > 1: print(f"  trial MPLight [{_m}]:", {k: (_g.get("MPLIGHT_SELECTION_TRIALS") or {}).get(_m, {}).get(k, {}).get("round") for k in _d}, "(round chọn theo train seed)")
        if MPLIGHT_CKPT_USER: print("  MPLIGHT_CKPT thủ công:", MPLIGHT_CKPT_USER)
        print("J05 protocol    : BENCH_TRIPINFO", BENCH_TRIPINFO, "| MPLIGHT_TRAIN_SEEDS", MPLIGHT_TRAIN_SEEDS_EFF, "| REPORT_BASELINE", REPORT_BASELINE, "| improvement:", OUT_DIR / "bench" / "improvement.csv" if _g.get("IMPROVEMENT_WRITTEN") else "(không)", "| results_trials:", OUT_DIR / "bench" / "results_trials.csv" if _g.get("TRIALS_TABLE_WRITTEN") else "(không)")
        print("J07 adaptive beta: ADAPTIVE_BETA", repr(ADAPTIVE_BETA), "| effective", _g.get("ADAPTIVE_BETA_EFF"), "|", STATUS.get("adaptive_beta"), "| used:", FLOW_DIR / "adaptive_beta_used.json" if _g.get("ADAPTIVE_BETA_EFF") else "(không)")   # J07
        for _m, _p in (_g.get("LEARNING_CURVE_FILES") or {}).items(): print(f"  learning curve [{_m}]: {_p}")
        if _g.get("SKIPPED"): print("controller bị bỏ qua (theo cấu hình):", _g.get("SKIPPED"))
        print("J02 demand:", DEMAND_MODE, "| directional settings:", FLOW_DIR / "demand_settings_directional.json" if J02 else "-")
        print("Placeholder chưa implement: 05 IMC (J03)")
    elif IMAGE_REVIEW:
        print("image_review: detections", ACTIVE.get("dets_city"), "| roi_frames", OUT_DIR / "roi_frames")
        print("Đã bỏ qua có chủ đích (image_review): calibration, flow inference, demand settings, routeSampler, SUMO, controllers, benchmark.")
    else:
        print("calibration run dir:", _g.get("CAL_RUN_DIR"))
        print("Đã bỏ qua có chủ đích: flow inference, demand settings, routeSampler, SUMO, controllers, benchmark.")
except Exception as _e5: print("(tóm tắt không đầy đủ:", type(_e5).__name__, _e5, ")")
if problems:
    if _BLK: print("\nCALIBRATION BLOCKED/INSUFFICIENT_DATA — " + "; ".join(p.replace("BLOCKED: INSUFFICIENT_DATA — ", "") for p in _BLK))      # J02E fix2
    print("\n" + ("CALIBRATION KHÔNG HOÀN CHỈNH" if RUN_MODE == "calibration_only" else ("IMAGE_REVIEW FAIL" if IMAGE_REVIEW else ("E2E: BLOCKED/INSUFFICIENT_DATA" if _BLK else "E2E: FAIL"))) + " — " + "; ".join(problems))
    raise RuntimeError(("CALIBRATION BLOCKED/INSUFFICIENT_DATA — " + "; ".join(p.replace("BLOCKED: INSUFFICIENT_DATA — ", "") for p in _BLK) + " | " if _BLK else "") + "Run KHÔNG đạt các stage/artifact bắt buộc của RUN_MODE='%s': %s" % (RUN_MODE, "; ".join(problems)))
_GTK = str(GROUND_TRUTH_KIND)      # J02E fix6: mọi kind bắt đầu bằng SYNTHETIC là synthetic; in kind thật
_SYN = " — CODE TEST — SYNTHETIC GT (not scientific)" if _GTK == "SYNTHETIC_GT_CODE_TEST" else (f" — {_GTK} (synthetic calibration; NOT real-world, NOT scientific)" if _GTK.startswith("SYNTHETIC") else "")      # J02E: calibration trên ground truth tổng hợp = CODE TEST
if IMAGE_REVIEW: print("\nIMAGE_REVIEW COMPLETE — detection city + đếm xe theo ROI từng frame (instantaneous count, KHÔNG phải flow); calibration/flow/SUMO/benchmark BỎ QUA có chủ đích")
elif RUN_MODE == "calibration_only": print("\nCALIBRATION COMPLETE" + _SYN)
else: print(f"\nE2E: PASS{_SYN} — mọi stage bắt buộc của RUN_MODE='{RUN_MODE}' đã chạy thật trong RUN_ID này" + (" (gồm training)" if RUN_TRAIN else ""))